## Nama : Hazel Novero Winata

## NIM : 2802519114

## Link Vidio: <https://drive.google.com/drive/folders/1irfbmjds6OJg06BBivk-rkVUnxBles2Z?usp=sharing>

## A. Exploratory Data Analysis

In [ ]:
# STANDARD LIBRARY
import os
import warnings

# DATA MANIPULATION
import numpy as np
import pandas as pd

# VISUALIZATION
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# PREPROCESSING
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# EVALUATION METRICS
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error
)

# DEEP LEARNING
import tensorflow as tf
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Input,
    LSTM
)
from tensorflow.keras.losses import Huber
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

# WARNING SETTINGS
warnings.filterwarnings("ignore")

print("All libraries loaded successfully.")


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.6f}".format)
sns.set_theme(style="whitegrid")


Cell ini digunakan untuk mengatur tampilan output agar proses eksplorasi data lebih mudah dibaca. Seluruh kolom pada DataFrame akan ditampilkan tanpa disingkat, sedangkan angka desimal ditampilkan hingga enam digit di belakang koma. Selain itu, tema grafik diatur menggunakan `whitegrid` agar pola data dan garis bantu pada visualisasi terlihat lebih jelas.

In [ ]:
PRICE_COLS = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
NUM_COLS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]


Cell ini mendefinisikan daftar kolom yang terdapat pada dataset saham. Variabel `PRICE_COLS` berisi seluruh kolom, yaitu `Date`, `Open`, `High`, `Low`, `Close`, `Adj Close`, dan `Volume`. Sementara itu, `NUM_COLS` hanya berisi kolom numerik dan tidak memasukkan `Date` karena kolom tersebut akan diproses sebagai data waktu.

Daftar kolom ini digunakan untuk mempermudah pemeriksaan struktur data dan proses konversi tipe data pada tahap berikutnya. Walaupun seluruh kolom diperiksa pada tahap EDA, model LSTM nantinya hanya menggunakan kolom `Date` dan `Close` sesuai ketentuan soal.

### 1. Load Dataset

In [ ]:
amd_raw = pd.read_csv("data/AMD.csv")
amd_raw.shape


In [ ]:
aapl_raw = pd.read_csv("data/AAPL.csv")
aapl_raw.shape


In [ ]:
meta_path = "data/symbols_valid_meta.csv"

if os.path.exists(meta_path):
    meta = pd.read_csv(meta_path)
    print("Metadata berhasil dibaca:", meta.shape)
else:
    meta = None
    print("symbols_valid_meta.csv belum ada di folder data.")


In [ ]:
print(f"AMD  : {amd_raw.shape[0]:,} baris, {amd_raw.shape[1]} kolom")
print(f"AAPL : {aapl_raw.shape[0]:,} baris, {aapl_raw.shape[1]} kolom")


Hasil pemeriksaan ukuran data menunjukkan bahwa dataset AMD memiliki **10.098 baris dan 7 kolom**, sedangkan dataset AAPL memiliki **9.909 baris dan 7 kolom**. Setiap baris merepresentasikan satu hari perdagangan saham, bukan satu hari kalender.

AMD memiliki 189 observasi lebih banyak daripada AAPL karena periode historis AMD dimulai lebih awal. Walaupun jumlah observasinya berbeda, kedua dataset tetap akan diproses secara terpisah agar pola harga, skala nilai, pembagian data, scaler, dan model yang digunakan tidak saling tercampur.

### 2. Tampilan Awal Data

In [ ]:
amd_raw.head()


In [ ]:
aapl_raw.head()


In [ ]:
amd_raw.tail()


In [ ]:
aapl_raw.tail()


In [ ]:
amd_raw.dtypes.to_frame('AMD Data Type')


In [ ]:
aapl_raw.dtypes.to_frame('AAPL Data Type')


Tipe data pada dataset AMD dan AAPL sudah sesuai. Kolom `Open`, `High`, `Low`, `Close`, dan `Adj Close` bertipe `float64`, sedangkan `Volume` bertipe `int64`. Kolom `Date` masih bertipe `object`, sehingga perlu dikonversi menjadi `datetime64[ns]` agar dapat digunakan untuk analisis time series dan pembagian data secara kronologis.

### 2. Konversi kolom Date dan Pengurutan Data

In [ ]:
amd = amd_raw.copy()
amd["Date"] = pd.to_datetime(amd["Date"], errors="coerce")


In [ ]:
amd[NUM_COLS] = amd[NUM_COLS].apply(pd.to_numeric, errors="coerce")


In [ ]:
aapl = aapl_raw.copy()
aapl["Date"] = pd.to_datetime(aapl["Date"], errors="coerce")


In [ ]:
aapl[NUM_COLS] = aapl[NUM_COLS].apply(pd.to_numeric, errors="coerce")


In [ ]:
amd = amd.sort_values("Date").reset_index(drop=True)
amd.head()


In [ ]:
aapl = aapl.sort_values("Date").reset_index(drop=True)
aapl.head()


Kolom `Date` pada dataset AMD dan AAPL diubah dari tipe `object` menjadi `datetime`. Setelah itu, data diurutkan berdasarkan tanggal dari yang paling lama ke yang paling baru agar urutan kronologis time series tetap terjaga.

### 3. Pemeriksaan Kualitas Data

In [ ]:
amd_quality = pd.Series({
    "Rows": len(amd),
    "Start Date": amd["Date"].min(),
    "End Date": amd["Date"].max(),
    "Missing Date": amd["Date"].isna().sum(),
    "Missing Close": amd["Close"].isna().sum(),
    "Duplicate Rows": amd.duplicated().sum(),
    "Duplicate Dates": amd["Date"].duplicated().sum(),
    "Ascending": amd["Date"].is_monotonic_increasing,
    "Weekend Rows": amd["Date"].dt.dayofweek.isin([5, 6]).sum(),
    "Close = 0": amd["Close"].eq(0).sum(),
    "Close < 0": amd["Close"].lt(0).sum(),
    "Infinite Close": np.isinf(amd["Close"]).sum()
}, name="AMD")

amd_quality.to_frame()


Dataset AMD memiliki 10.098 baris dengan periode 17 Maret 1980 hingga 1 April 2020. Tidak ditemukan missing value pada kolom `Date` maupun `Close`, tidak ada baris atau tanggal duplikat, dan data sudah tersusun secara ascending. Seluruh observasi juga hanya berada pada hari perdagangan karena tidak terdapat data akhir pekan. Nilai `Close` tidak memiliki nilai nol, negatif, maupun infinite, sehingga seluruh baris aman digunakan untuk tahap analisis dan modeling.

In [ ]:
aapl_quality = pd.Series({
    "Rows": len(aapl),
    "Start Date": aapl["Date"].min(),
    "End Date": aapl["Date"].max(),
    "Missing Date": aapl["Date"].isna().sum(),
    "Missing Close": aapl["Close"].isna().sum(),
    "Duplicate Rows": aapl.duplicated().sum(),
    "Duplicate Dates": aapl["Date"].duplicated().sum(),
    "Ascending": aapl["Date"].is_monotonic_increasing,
    "Weekend Rows": aapl["Date"].dt.dayofweek.isin([5, 6]).sum(),
    "Close = 0": aapl["Close"].eq(0).sum(),
    "Close < 0": aapl["Close"].lt(0).sum(),
    "Infinite Close": np.isinf(aapl["Close"]).sum()
}, name="AAPL")

aapl_quality.to_frame()


Dataset AAPL memiliki 9.909 baris dengan periode 12 Desember 1980 hingga 1 April 2020. Tidak ditemukan missing value pada kolom `Date` maupun `Close`, tidak ada baris atau tanggal duplikat, dan data sudah tersusun secara ascending. Tidak terdapat data pada akhir pekan serta tidak ditemukan nilai `Close` yang nol, negatif, atau infinite. Dengan demikian, seluruh baris dapat digunakan untuk analisis dan modeling.

In [ ]:
amd_profile = pd.DataFrame({
    "Type": amd.dtypes.astype(str),
    "Missing": amd.isna().sum(),
    "Unique": amd.nunique()
})

amd_profile["Zero"] = np.nan
amd_profile["Negative"] = np.nan
amd_profile["Minimum"] = np.nan
amd_profile["Maximum"] = np.nan


In [ ]:
amd_profile.loc[NUM_COLS, "Zero"] = amd[NUM_COLS].eq(0).sum()
amd_profile.loc[NUM_COLS, "Negative"] = amd[NUM_COLS].lt(0).sum()
amd_profile.loc[NUM_COLS, "Minimum"] = amd[NUM_COLS].min()
amd_profile.loc[NUM_COLS, "Maximum"] = amd[NUM_COLS].max()

amd_profile


Kode ini digunakan untuk membuat profil setiap kolom pada dataset AMD. Informasi yang diperiksa meliputi tipe data, jumlah missing value, jumlah nilai unik, jumlah nilai nol, jumlah nilai negatif, serta nilai minimum dan maksimum pada setiap kolom.

Berdasarkan output, seluruh kolom tidak memiliki missing value. Kolom `Open` memiliki 760 nilai nol, sedangkan kolom `Volume` memiliki 2 nilai nol. Kolom harga lainnya tidak memiliki nilai nol maupun negatif. Nilai minimum pada kolom `Close` adalah 1,62 dan tidak terdapat nilai negatif, sehingga kolom `Close` tetap valid digunakan sebagai input dan target model LSTM.

Anomali pada `Open` dan `Volume` hanya dicatat sebagai bagian dari pemeriksaan kualitas data. Baris tersebut tidak perlu dihapus karena proses modeling hanya menggunakan kolom `Date` dan `Close`.

In [ ]:
aapl_profile = pd.DataFrame({
    "Type": aapl.dtypes.astype(str),
    "Missing": aapl.isna().sum(),
    "Unique": aapl.nunique()
})

aapl_profile["Zero"] = np.nan
aapl_profile["Negative"] = np.nan
aapl_profile["Minimum"] = np.nan
aapl_profile["Maximum"] = np.nan


In [ ]:
aapl_profile.loc[NUM_COLS, "Zero"] = aapl[NUM_COLS].eq(0).sum()
aapl_profile.loc[NUM_COLS, "Negative"] = aapl[NUM_COLS].lt(0).sum()
aapl_profile.loc[NUM_COLS, "Minimum"] = aapl[NUM_COLS].min()
aapl_profile.loc[NUM_COLS, "Maximum"] = aapl[NUM_COLS].max()

aapl_profile


Berdasarkan output, dataset AAPL memiliki 9.909 tanggal unik dan tidak terdapat missing value pada seluruh kolom. Semua kolom harga dan volume juga tidak memiliki nilai nol maupun negatif.

Nilai `Close` berada pada rentang sekitar 0,196429 hingga 327,200012, sedangkan `Adj Close` berada pada rentang sekitar 0,155638 hingga 327,200012. Perbedaan jumlah nilai unik antara `Close` dan `Adj Close` menunjukkan bahwa kedua kolom tidak selalu memiliki nilai yang sama.

Kolom `Volume` memiliki nilai minimum 347.200 dan maksimum 1.855.410.200. Secara keseluruhan, tidak ditemukan masalah kualitas data yang mengharuskan penghapusan baris pada dataset AAPL, terutama pada kolom `Date` dan `Close` yang akan digunakan untuk modeling.

### 4. Metadata Saham

File `symbols_valid_meta.csv` digunakan sebagai metadata pendukung untuk menjelaskan identitas setiap simbol saham. Dari file ini diketahui bahwa AMD adalah **Advanced Micro Devices, Inc.**, sedangkan AAPL adalah **Apple Inc.**. Keduanya merupakan **common stock** yang tercatat di Nasdaq, bukan ETF, dan memiliki round lot size sebesar 100 saham.

### 5. Pola Kalender Perdagangan

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]


In [ ]:
amd_weekdays = (
    amd["Date"].dt.day_name()
    .value_counts()
    .reindex(day_order, fill_value=0)
)

amd_weekdays.to_frame("AMD Count")


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(amd_weekdays.index, amd_weekdays.values)
plt.title("Distribution of Trading Days - AMD")
plt.xlabel("Day")
plt.ylabel("Number of Observations")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Output menunjukkan bahwa data AMD hanya tercatat pada hari perdagangan, yaitu Senin sampai Jumat. Tidak terdapat observasi pada hari Sabtu dan Minggu. Jumlah data paling sedikit berada pada hari Senin, yaitu 1.913 observasi, sedangkan Selasa dan Rabu memiliki jumlah terbanyak, masing-masing 2.069 observasi.

Perbedaan jumlah antarhari kemungkinan disebabkan oleh hari libur bursa. Oleh karena itu, tanggal yang tidak muncul pada akhir pekan atau hari libur tidak dianggap sebagai missing value dan tidak perlu diisi dengan interpolasi.

In [ ]:
aapl_weekdays = (
    aapl["Date"].dt.day_name()
    .value_counts()
    .reindex(day_order, fill_value=0)
)

aapl_weekdays.to_frame("AAPL Count")


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(aapl_weekdays.index, aapl_weekdays.values)
plt.title("Distribution of Trading Days - AAPL")
plt.xlabel("Day")
plt.ylabel("Number of Observations")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Output menunjukkan bahwa data AAPL hanya tercatat pada hari perdagangan, yaitu Senin sampai Jumat. Tidak terdapat observasi pada hari Sabtu dan Minggu. Jumlah observasi paling sedikit berada pada hari Senin, yaitu 1.875 data, sedangkan jumlah terbanyak berada pada hari Selasa, yaitu 2.031 data.

Perbedaan jumlah antarhari kemungkinan disebabkan oleh hari libur bursa. Oleh karena itu, tanggal yang tidak tersedia pada akhir pekan atau hari libur tidak dianggap sebagai missing value dan tidak perlu diisi.

In [ ]:
amd_gap = pd.DataFrame({
    "Previous Date": amd["Date"].shift(),
    "Current Date": amd["Date"],
    "Gap Days": amd["Date"].diff().dt.days
}).dropna()

amd_gap["Gap Days"].value_counts().sort_index().to_frame("AMD Count")


In [ ]:
amd_long_gap = amd_gap[amd_gap["Gap Days"] > 4].sort_values(
    "Gap Days", ascending=False
)

amd_long_gap


Output menunjukkan bahwa dari 10.098 observasi AMD terdapat 10.097 selisih antartanggal, karena baris pertama tidak memiliki tanggal sebelumnya untuk dibandingkan.

Sebagian besar data memiliki jeda **1 hari**, yaitu sebanyak 7.912 kali, yang menunjukkan adanya observasi pada hari perdagangan yang berurutan. Jeda **3 hari** muncul sebanyak 1.840 kali dan umumnya berkaitan dengan perpindahan dari hari Jumat ke Senin. Jeda **2 hari** terjadi 96 kali, sedangkan jeda **4 hari** terjadi 246 kali, yang kemungkinan berkaitan dengan hari libur bursa atau akhir pekan yang diperpanjang.

Terdapat tiga jeda yang lebih panjang dari empat hari, yaitu satu jeda selama 7 hari dan dua jeda selama 5 hari. Hal ini menunjukkan bahwa tidak terdapat observasi yang tercatat pada tanggal-tanggal di antara dua periode tersebut. Namun, dari output ini saja belum dapat dipastikan apakah jeda tersebut disebabkan oleh penutupan bursa, hari libur, atau keterbatasan pencatatan data.

Oleh karena itu, tanggal yang tidak muncul tidak langsung dianggap sebagai missing value. Dataset ini merepresentasikan urutan hari perdagangan, bukan kalender harian, sehingga tanggal yang tidak memiliki observasi tidak perlu langsung ditambahkan atau diisi melalui interpolasi.

In [ ]:
aapl_gap = pd.DataFrame({
    "Previous Date": aapl["Date"].shift(),
    "Current Date": aapl["Date"],
    "Gap Days": aapl["Date"].diff().dt.days
}).dropna()

aapl_gap["Gap Days"].value_counts().sort_index().to_frame("AAPL Count")


In [ ]:
aapl_long_gap = aapl_gap[aapl_gap["Gap Days"] > 4].sort_values(
    "Gap Days", ascending=False
)

aapl_long_gap


Output menunjukkan bahwa dari 9.909 observasi AAPL terdapat 9.908 selisih antartanggal. Sebagian besar data memiliki jeda **1 hari**, yaitu sebanyak 7.763 kali. Jeda **3 hari** muncul sebanyak 1.805 kali dan umumnya berkaitan dengan perpindahan dari hari Jumat ke Senin. Jeda **2 hari** terjadi 94 kali, sedangkan jeda **4 hari** terjadi 243 kali.

Terdapat tiga jeda yang lebih panjang dari empat hari, yaitu satu jeda selama 7 hari dan dua jeda selama 5 hari. Tanggal jeda panjang tersebut sama dengan yang ditemukan pada dataset AMD, sehingga kemungkinan berkaitan dengan kalender perdagangan yang berlaku secara umum, meskipun penyebab pastinya tetap memerlukan verifikasi tambahan.

Oleh karena itu, tanggal yang tidak muncul tidak langsung dianggap sebagai missing value. Dataset AAPL merepresentasikan urutan hari perdagangan, sehingga tanggal di luar sesi perdagangan tidak perlu ditambahkan atau diisi melalui interpolasi.

In [ ]:
common_start = max(amd["Date"].min(), aapl["Date"].min())

amd_dates = set(amd.loc[amd["Date"] >= common_start, "Date"])
aapl_dates = set(aapl.loc[aapl["Date"] >= common_start, "Date"])


In [ ]:
only_amd = sorted(amd_dates - aapl_dates)
only_aapl = sorted(aapl_dates - amd_dates)

print("Common start date :", common_start)
print("Dates only in AMD :", len(only_amd))
print("Dates only in AAPL:", len(only_aapl))


In [ ]:
pd.DataFrame({"Only in AMD": only_amd})


In [ ]:
pd.DataFrame({"Only in AAPL": only_aapl})


Kode ini membandingkan tanggal perdagangan AMD dan AAPL mulai dari tanggal awal yang sama, yaitu 12 Desember 1980.

Hasilnya menunjukkan bahwa hanya terdapat satu tanggal yang ada pada AMD tetapi tidak ada pada AAPL, yaitu 10 Agustus 1981. Tidak terdapat tanggal yang hanya ada pada AAPL. Secara keseluruhan, kalender perdagangan kedua dataset hampir sama, tetapi keduanya tetap diproses secara terpisah karena jumlah observasi dan pola harga berbeda.

### 6. Audit OHLCV

In [ ]:
amd_ohlcv = pd.Series({
    "Open = 0": amd["Open"].eq(0).sum(),
    "Volume = 0": amd["Volume"].eq(0).sum(),
    "High < Low": amd["High"].lt(amd["Low"]).sum(),
    "High < max(Open, Close)": amd["High"].lt(
        amd[["Open", "Close"]].max(axis=1)
    ).sum(),
    "Low > min(Open, Close)": amd["Low"].gt(
        amd[["Open", "Close"]].min(axis=1)
    ).sum(),
    "Low Violation when Open > 0": (
        (amd["Open"] > 0) &
        amd["Low"].gt(amd[["Open", "Close"]].min(axis=1))
    ).sum()
}, name="AMD")

amd_ohlcv.to_frame()


Kode ini digunakan untuk memeriksa konsistensi nilai **Open, High, Low, Close, dan Volume** pada dataset AMD.

Hasil audit menunjukkan terdapat **760 nilai `Open = 0`** dan **2 nilai `Volume = 0`**. Tidak ditemukan kasus `High < Low` maupun `High` yang lebih kecil dari nilai maksimum antara `Open` dan `Close`, sehingga struktur harga tertinggi dan terendah tetap konsisten.

Nilai `Low > min(Open, Close)` muncul sebanyak 760 kali karena pada baris yang sama nilai `Open` tercatat 0. Setelah pemeriksaan dibatasi hanya pada baris dengan `Open > 0`, jumlah pelanggaran menjadi 0. Artinya, masalah tersebut berasal dari anomali nilai `Open`, bukan kesalahan pada kolom `Low`.

In [ ]:
aapl_ohlcv = pd.Series({
    "Open = 0": aapl["Open"].eq(0).sum(),
    "Volume = 0": aapl["Volume"].eq(0).sum(),
    "High < Low": aapl["High"].lt(aapl["Low"]).sum(),
    "High < max(Open, Close)": aapl["High"].lt(
        aapl[["Open", "Close"]].max(axis=1)
    ).sum(),
    "Low > min(Open, Close)": aapl["Low"].gt(
        aapl[["Open", "Close"]].min(axis=1)
    ).sum(),
    "Low Violation when Open > 0": (
        (aapl["Open"] > 0) &
        aapl["Low"].gt(aapl[["Open", "Close"]].min(axis=1))
    ).sum()
}, name="AAPL")

aapl_ohlcv.to_frame()


In [ ]:
amd_open_zero = amd[amd["Open"] == 0]

print("Jumlah Open = 0:", len(amd_open_zero))
print("Periode:", amd_open_zero["Date"].min(), "sampai", amd_open_zero["Date"].max())


### 7. Cek Hasil Audit Lebih Lanjut

In [ ]:
amd_open_zero.head()


In [ ]:
amd_open_zero.tail()


Pada dataset AMD ditemukan 760 baris dengan nilai `Open = 0`, yang terjadi secara berurutan dari 17 Maret 1980 sampai 18 Maret 1983. Nilai tersebut tidak masuk akal apabila dianggap sebagai harga pembukaan sebenarnya karena pada tanggal yang sama nilai `High`, `Low`, dan `Close` tetap positif.

Sebagai contoh, pada 17 Maret 1980 diperoleh:

- `Open = 0`
- `High = 3,302083`
- `Low = 3,125000`
- `Close = 3,145833`

Dalam perdagangan normal, harga pembukaan seharusnya berada di antara harga terendah dan harga tertinggi pada hari tersebut. Oleh karena itu, nilai `Open = 0` kemungkinan menunjukkan bahwa data harga pembukaan pada periode tersebut tidak tersedia atau tidak tercatat dengan benar.

Hasil audit juga menunjukkan `Low > min(Open, Close)` sebanyak 760 kali. Hal ini bukan berarti kolom `Low` bermasalah. Karena `Open` bernilai 0, nilai minimum antara `Open` dan `Close` otomatis menjadi 0, sehingga nilai `Low` yang positif akan selalu lebih besar dari nilai tersebut. Ketika pemeriksaan hanya dilakukan pada baris dengan `Open > 0`, tidak ditemukan pelanggaran.

Pada baris-baris tersebut, nilai `Date` tetap tidak kosong, tidak duplikat, dan tersusun secara kronologis. Nilai `Close` juga tidak kosong, tidak nol, tidak negatif, serta masih berada di antara `Low` dan `High`.

Sebagai contoh, pada 17 Maret 1980:

$$
3{,}125000 \leq 3{,}145833 \leq 3{,}302083
$$

Hubungan tersebut menunjukkan bahwa nilai `Close` masih berada di antara harga terendah dan harga tertinggi pada hari yang sama.

Dengan demikian, kolom `Date` dan `Close` tetap konsisten secara struktur untuk kebutuhan modeling. Namun, hasil ini tidak membuktikan kebenaran historis absolutnya, melainkan hanya menunjukkan bahwa tidak ditemukan masalah struktural pada kedua kolom tersebut.

Baris dengan `Open = 0` tetap dipertahankan karena model LSTM hanya menggunakan kolom `Date` dan `Close`. Menghapus 760 baris tersebut akan ikut menghilangkan sekitar tiga tahun histori harga penutupan AMD yang masih dapat digunakan. Oleh karena itu, anomali dicatat sebagai keterbatasan pada kolom `Open`, tetapi tidak digunakan sebagai alasan untuk menghapus seluruh baris.

Apabila kolom `Open` akan digunakan sebagai fitur pada penelitian lain, nilai-nilai tersebut harus diperiksa dan ditangani lebih lanjut terlebih dahulu, misalnya melalui verifikasi terhadap sumber data asli atau metode penanganan missing value yang dapat dipertanggungjawabkan.

In [ ]:
amd_volume_zero = amd[amd["Volume"] == 0]
amd_volume_zero


In [ ]:
close_adj_table = pd.DataFrame({
    "Ticker": ["AMD", "AAPL"],
    "Same": [
        np.isclose(amd["Close"], amd["Adj Close"]).sum(),
        np.isclose(aapl["Close"], aapl["Adj Close"]).sum()
    ],
    "Different": [
        (~np.isclose(amd["Close"], amd["Adj Close"])).sum(),
        (~np.isclose(aapl["Close"], aapl["Adj Close"])).sum()
    ]
})

close_adj_table


Pada dataset AMD ditemukan dua baris dengan `Volume = 0`, yaitu pada 5 Oktober 1982 dan 2 Januari 2015. Pada kedua tanggal tersebut, nilai `Open`, `High`, `Low`, `Close`, dan `Adj Close` memiliki nilai yang sama dan tetap positif. Kondisi ini menunjukkan bahwa struktur harga pada baris tersebut tidak saling bertentangan, tetapi dari dataset saja belum dapat dipastikan apakah benar tidak terjadi transaksi atau nilai volume tidak tercatat dengan baik.

Kedua baris tetap dipertahankan karena model LSTM tidak menggunakan `Volume`, sedangkan nilai `Date` dan `Close` masih tersedia dan konsisten secara struktur. Menghapus baris hanya berdasarkan `Volume = 0` dapat menghilangkan observasi harga penutupan tanpa bukti bahwa nilai `Close` juga salah. Namun, kondisi ini tetap dicatat sebagai keterbatasan data. Apabila `Volume` akan digunakan sebagai fitur, kedua baris tersebut perlu diverifikasi atau ditangani lebih lanjut.

Perbandingan `Close` dan `Adj Close` menunjukkan bahwa seluruh 10.098 baris AMD memiliki nilai yang sama pada kedua kolom tersebut. Pada AAPL, hanya 38 baris yang memiliki nilai sama, sedangkan 9.871 baris memiliki nilai yang berbeda. Perbedaan ini terjadi karena `Adj Close` merupakan harga penutupan yang telah disesuaikan terhadap aksi korporasi, seperti dividen atau pemecahan saham.

Walaupun `Adj Close` dapat mencerminkan nilai historis yang telah disesuaikan, model tetap menggunakan kolom `Close` karena hal tersebut telah ditetapkan dalam soal. Penggunaan target harus konsisten dan tidak mencampurkan `Close` dengan `Adj Close`.

### 8. Statistik dan Pola Close

In [ ]:
amd_close_summary = pd.Series({
    "Ticker": "AMD",
    "Count": amd["Close"].count(),
    "Minimum": amd["Close"].min(),
    "Q1": amd["Close"].quantile(0.25),
    "Median": amd["Close"].median(),
    "Mean": amd["Close"].mean(),
    "Q3": amd["Close"].quantile(0.75),
    "Maximum": amd["Close"].max(),
    "Std": amd["Close"].std(),
    "Skewness": amd["Close"].skew(),
    "Minimum Date": amd.loc[amd["Close"].idxmin(), "Date"],
    "Maximum Date": amd.loc[amd["Close"].idxmax(), "Date"]
}, name="AMD")

amd_close_summary.to_frame("Value")


Kolom `Close` AMD memiliki 10.098 observasi dengan nilai minimum sebesar 1,62 pada 27 Juli 2015 dan nilai maksimum sebesar 58,90 pada 19 Februari 2020. Sebanyak 50% nilai harga penutupan berada di antara kuartil pertama sebesar 4,94 dan kuartil ketiga sebesar 14,71, dengan median sebesar 9,06.

Nilai rata-rata sebesar 11,21 lebih tinggi daripada median, sedangkan nilai skewness sebesar 1,75 menunjukkan bahwa distribusi `Close` miring ke kanan. Artinya, sebagian besar harga berada pada rentang yang relatif rendah, tetapi terdapat sejumlah harga tinggi pada bagian akhir periode yang membentuk ekor distribusi ke arah kanan.

Standar deviasi sebesar 8,28 menunjukkan bahwa nilai harga penutupan memiliki penyebaran yang cukup besar terhadap rata-ratanya. Namun, nilai ini belum dapat langsung digunakan untuk menyimpulkan volatilitas harian karena volatilitas akan dianalisis lebih tepat menggunakan perubahan atau return harian.

Nilai minimum dan maksimum tidak langsung dihapus sebagai outlier karena keduanya dapat mencerminkan perubahan harga saham yang benar-benar terjadi pada periode berbeda. Hasil ini menunjukkan bahwa harga AMD mengalami perubahan skala dan pola sepanjang waktu, sehingga urutan kronologis harus dipertahankan dalam proses modeling.

In [ ]:
aapl_close_summary = pd.Series({
    "Ticker": "AAPL",
    "Count": aapl["Close"].count(),
    "Minimum": aapl["Close"].min(),
    "Q1": aapl["Close"].quantile(0.25),
    "Median": aapl["Close"].median(),
    "Mean": aapl["Close"].mean(),
    "Q3": aapl["Close"].quantile(0.75),
    "Maximum": aapl["Close"].max(),
    "Std": aapl["Close"].std(),
    "Skewness": aapl["Close"].skew(),
    "Minimum Date": aapl.loc[aapl["Close"].idxmin(), "Date"],
    "Maximum Date": aapl.loc[aapl["Close"].idxmax(), "Date"]
}, name="AAPL")

aapl_close_summary.to_frame("Value")


Kolom `Close` AAPL memiliki 9.909 observasi dengan nilai minimum sebesar 0,196429 pada 8 Juli 1982 dan nilai maksimum sebesar 327,200012 pada 12 Februari 2020. Sebanyak 50% nilai harga penutupan berada di antara kuartil pertama sebesar 1,071429 dan kuartil ketiga sebesar 35,761429, dengan median sebesar 1,732143.

Nilai rata-rata sebesar 32,62 jauh lebih tinggi daripada median. Selain itu, nilai skewness sebesar 2,18 menunjukkan bahwa distribusi `Close` sangat miring ke kanan. Artinya, sebagian besar harga historis AAPL berada pada rentang yang relatif rendah, sedangkan harga-harga yang sangat tinggi pada periode akhir membentuk ekor panjang di sisi kanan distribusi.

Standar deviasi sebesar 58,47 menunjukkan penyebaran nilai harga penutupan yang sangat besar. Namun, nilai ini belum dapat langsung digunakan untuk menyimpulkan volatilitas harian karena dipengaruhi oleh perubahan skala harga dalam jangka panjang. Volatilitas akan dianalisis lebih tepat menggunakan return harian.

Nilai minimum dan maksimum tidak langsung dihapus sebagai outlier karena keduanya dapat mencerminkan kondisi harga pada periode historis yang berbeda. Hasil ini menunjukkan bahwa AAPL mengalami tren kenaikan jangka panjang dan perubahan distribusi yang kuat, sehingga data harus diproses secara kronologis dan scaling perlu dilakukan secara hati-hati.

In [ ]:
## perbandingan keduanya
close_summary_table = pd.DataFrame([
    amd_close_summary,
    aapl_close_summary
])

close_summary_table


Perbandingan menunjukkan bahwa AMD memiliki 10.098 observasi, sedangkan AAPL memiliki 9.909 observasi. AAPL memiliki rentang harga dan rata-rata yang jauh lebih besar, serta distribusi yang lebih miring ke kanan dibandingkan AMD. Oleh karena itu, kedua saham perlu dianalisis, dinormalisasi, dan dimodelkan secara terpisah.

#### 8A. Historical Closing Price

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(amd["Date"], amd["Close"])
plt.title("Historical Closing Price - AMD")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(aapl["Date"], aapl["Close"])
plt.title("Historical Closing Price - AAPL")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.tight_layout()
plt.show()


Grafik menunjukkan bahwa pola harga penutupan AMD dan AAPL berbeda cukup jelas. Harga AMD cenderung lebih fluktuatif dan siklis, dengan beberapa periode kenaikan tajam yang kemudian diikuti penurunan besar. Pola ini menunjukkan bahwa AMD memiliki perubahan regime pasar dan volatilitas historis yang cukup tinggi.

Sebaliknya, AAPL menunjukkan tren kenaikan jangka panjang yang jauh lebih kuat, terutama setelah pertengahan tahun 2000-an. Harga AAPL meningkat hingga mencapai level tertinggi pada awal 2020, meskipun terdapat beberapa koreksi tajam selama perjalanannya.

Kedua grafik menunjukkan bahwa data harga tidak bersifat stasioner karena rata-rata dan rentang nilainya berubah sepanjang waktu. Selain itu, skala harga AMD dan AAPL sangat berbeda, sehingga kedua dataset harus diproses, dinormalisasi, dan dimodelkan secara terpisah menggunakan scaler masing-masing. Nilai tinggi atau penurunan tajam pada grafik juga tidak langsung dihapus sebagai outlier karena dapat merepresentasikan kondisi pasar yang benar-benar terjadi.

#### 8B. Distributing of Closing Price

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(amd["Close"], bins=60, kde=True, color="C0")
plt.title("Distribution of Closing Price - AMD")
plt.xlabel("Closing Price")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(aapl["Close"], bins=60, kde=True, color="C0")
plt.title("Distribution of Closing Price - AAPL")
plt.xlabel("Closing Price")
plt.tight_layout()
plt.show()


Histogram menunjukkan bahwa distribusi harga penutupan AMD dan AAPL sama-sama miring ke kanan. Pada AMD, sebagian besar nilai `Close` berada pada rentang rendah hingga menengah, sedangkan hanya sedikit observasi yang mencapai harga tinggi hingga mendekati 60.

Distribusi AAPL jauh lebih tidak simetris. Sebagian besar harga historis berada pada rentang yang sangat rendah, sementara harga pada periode akhir meningkat hingga lebih dari 300 sehingga membentuk ekor panjang di sisi kanan. Hal ini sesuai dengan nilai skewness AAPL yang lebih tinggi daripada AMD.

Pola tersebut menunjukkan bahwa distribusi harga berubah sepanjang waktu dan tidak mengikuti distribusi normal. Nilai harga tinggi tidak langsung dihapus sebagai outlier karena dapat mencerminkan perubahan harga historis yang sebenarnya. Perbedaan rentang harga juga memperkuat keputusan bahwa AMD dan AAPL harus menggunakan scaler serta model yang terpisah.

#### 8C. Coba Pakai Log Tfranform

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(np.log1p(amd["Close"]), bins=60, kde=True, color="C0")
plt.title("Log-Transformed Close Distribution - AMD")
plt.xlabel("log(1 + Close)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(np.log1p(aapl["Close"]), bins=60, kde=True, color="C0")
plt.title("Log-Transformed Close Distribution - AAPL")
plt.xlabel("log(1 + Close)")
plt.tight_layout()
plt.show()


Setelah transformasi `log(1 + Close)`, rentang harga yang sangat besar menjadi lebih terkompresi sehingga bentuk distribusi lebih mudah diamati.

Pada AMD, distribusi terlihat lebih seimbang dibandingkan distribusi asli, tetapi masih memiliki beberapa puncak. Hal ini menunjukkan bahwa harga AMD berasal dari beberapa periode atau kondisi pasar yang berbeda.

Pada AAPL, transformasi log mengurangi kemiringan distribusi, tetapi masih terlihat dua kelompok utama, yaitu kelompok harga historis yang rendah dan kelompok harga yang jauh lebih tinggi pada periode berikutnya. Pola ini memperkuat adanya perubahan skala dan regime harga sepanjang waktu.

Transformasi log pada tahap EDA ini hanya digunakan untuk membantu visualisasi distribusi. Baseline AMD dan AAPL tetap menggunakan nilai `Close` asli yang kemudian dinormalisasi menggunakan `MinMaxScaler`, dengan scaler hanya di-fit pada data train untuk mencegah data leakage.

Pada modified model, transformasi log dapat diuji sebagai salah satu eksperimen, terutama pada AAPL karena distribusinya lebih right-skewed dan memiliki perubahan skala harga yang lebih besar. Transformasi log hanya akan digunakan apabila memberikan performa validation yang lebih baik.

Jika transformasi log digunakan, hasil prediksi harus dikembalikan terlebih dahulu ke skala harga asli menggunakan inverse scaling dan `expm1` sebelum menghitung RMSE, MAE, dan MAPE. Dengan demikian, seluruh metrik tetap dapat diinterpretasikan dalam satuan harga saham asli.

#### 8D.Per Dekade

In [ ]:
amd_decade = amd[["Date", "Close"]].copy()
amd_decade["Decade"] = amd_decade["Date"].dt.year // 10 * 10


In [ ]:
amd_decade_summary = amd_decade.groupby("Decade")["Close"].agg(
    ["count", "min", "mean", "median", "max", "first", "last"]
)

amd_decade_summary["change_pct"] = (
    amd_decade_summary["last"] / amd_decade_summary["first"] - 1
) * 100

amd_decade_summary


Ringkasan per dekade digunakan untuk melihat perubahan pola harga AMD dalam jangka panjang. Hasilnya menunjukkan bahwa nilai minimum, rata-rata, median, dan maksimum berbeda cukup besar pada setiap dekade. Hal ini memperlihatkan bahwa AMD mengalami beberapa kondisi atau regime pasar yang berbeda, bukan satu pola harga yang stabil sepanjang periode.

Pengelompokan 10 tahun dipilih karena dataset memiliki rentang waktu hampir 40 tahun, sehingga pola jangka panjang lebih mudah diamati. Analisis ini hanya digunakan pada tahap EDA dan tidak menjadi pembagian data untuk modeling.

Perlu diperhatikan bahwa dekade 1980 tidak dimulai dari Januari 1980, sedangkan kelompok 2020 hanya berisi data sampai 1 April 2020. Oleh karena itu, persentase perubahan pada kedua kelompok tersebut tidak dapat dibandingkan secara langsung dengan dekade yang memiliki data lengkap selama 10 tahun.

Ringkasan per dekade menunjukkan bahwa harga AMD memiliki pola yang lebih fluktuatif dan siklis dibandingkan AAPL.

Pada periode 1980–1989, harga meningkat dari sekitar 3,15 menjadi 3,94 atau naik sebesar 25,17%. Meskipun nilai maksimum mencapai 20,50, harga akhir kembali berada pada level yang relatif rendah.

Pada periode 1990–1999, harga AMD meningkat lebih kuat dari 4,13 menjadi 14,47 atau sekitar 250,76%. Nilai rata-rata pada periode ini sebesar 10,27 dengan harga maksimum 23,69.

Pada periode 2000–2009, harga awal sebesar 15,50 turun menjadi 9,68 atau berkurang sekitar 37,55%. Walaupun harga sempat mencapai maksimum 47,50, penurunan pada akhir periode menunjukkan bahwa kenaikan harga tidak bertahan secara konsisten.

Pada periode 2010–2019, harga meningkat dari 9,70 menjadi 45,86 atau sekitar 372,78%. Namun, harga minimum sebesar 1,62 menunjukkan bahwa periode ini juga mencakup penurunan yang sangat dalam sebelum mengalami pemulihan dan kenaikan tajam.

Kelompok 2020 hanya mencakup 63 hari perdagangan hingga 1 April 2020. Harga turun dari sekitar 49,10 menjadi 43,66 atau sebesar 11,08%, dengan nilai maksimum 58,90. Karena hanya mencakup sebagian kecil tahun 2020, hasil ini tidak dapat dibandingkan langsung dengan periode 10 tahun lainnya.

Secara keseluruhan, AMD tidak menunjukkan tren kenaikan yang konsisten pada setiap dekade. Harga mengalami beberapa fase kenaikan tajam, penurunan besar, dan pemulihan, sehingga pola AMD lebih bersifat siklis dan memiliki perubahan regime pasar yang kuat.

In [ ]:
aapl_decade = aapl[["Date", "Close"]].copy()
aapl_decade["Decade"] = aapl_decade["Date"].dt.year // 10 * 10


In [ ]:
aapl_decade_summary = aapl_decade.groupby("Decade")["Close"].agg(
    ["count", "min", "mean", "median", "max", "first", "last"]
)

aapl_decade_summary["change_pct"] = (
    aapl_decade_summary["last"] / aapl_decade_summary["first"] - 1
) * 100

aapl_decade_summary


Ringkasan per dekade menunjukkan adanya kenaikan skala harga AAPL yang sangat besar dari waktu ke waktu.

Pada periode 1980–1989, harga penutupan rata-rata masih sebesar 0,80 dengan median 0,56. Harga meningkat dari 0,51 pada awal periode menjadi 1,26 pada akhir periode, atau naik sekitar 145,22%.

Pada periode 1990–1999, rata-rata harga meningkat menjadi 1,40 dan harga akhir mencapai 3,67. Perubahan dari harga awal ke harga akhir sebesar 176,01%.

Kenaikan menjadi jauh lebih kuat pada periode 2000–2009. Harga meningkat dari sekitar 4,00 menjadi 30,10 atau naik sebesar 653,03%. Nilai rata-ratanya sebesar 8,64, sedangkan nilai maksimum mencapai 30,23.

Periode 2010–2019 menunjukkan pertumbuhan paling besar. Harga meningkat dari 30,57 menjadi 293,65 atau sekitar 860,49%. Rata-rata harga pada periode ini sebesar 110,35 dan nilai maksimum mencapai 293,65. Hasil ini menunjukkan adanya perubahan skala harga dan tren naik jangka panjang yang sangat kuat.

Kelompok 2020 hanya mencakup 63 hari perdagangan hingga 1 April 2020, sehingga bukan satu dekade penuh. Pada periode singkat tersebut, harga turun dari 300,35 menjadi 240,91 atau sekitar 19,79%, dengan harga minimum 224,37 dan maksimum 327,20. Penurunan ini menunjukkan koreksi tajam pada awal 2020, tetapi tidak dapat dibandingkan langsung dengan periode 10 tahun lainnya.

Secara keseluruhan, hasil ini memperlihatkan bahwa distribusi dan skala harga AAPL berubah sangat besar antarperiode. Kondisi tersebut menunjukkan adanya tren dan perubahan regime harga, sehingga data tidak dapat dianggap memiliki pola yang stabil sepanjang waktu.

#### 8E. Analisis Tahunan

In [ ]:
amd_yearly = (
    amd.set_index("Date")["Close"]
    .resample("YE")
    .agg(["min", "mean", "median", "max"])
    .dropna()
)

amd_yearly["Year"] = amd_yearly.index.year
amd_yearly.tail(15)


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(amd_yearly["Year"], amd_yearly["mean"])
plt.title("Annual Average Closing Price - AMD")
plt.xlabel("Year")
plt.ylabel("Annual Average Close")
plt.tight_layout()
plt.show()


Analisis tahunan merangkum nilai minimum, rata-rata, median, dan maksimum harga penutupan AMD pada setiap tahun. Hasilnya memperlihatkan bahwa perubahan harga AMD tidak berlangsung secara stabil, melainkan mengalami beberapa fase kenaikan dan penurunan yang kuat.

Pada 2006, rata-rata harga AMD mencapai sekitar 28,11 dengan nilai maksimum 42,10. Setelah itu, harga mengalami penurunan tajam hingga rata-rata hanya sekitar 5,35 pada 2008 dan 4,43 pada 2009. Harga sempat meningkat kembali pada 2010, tetapi kemudian kembali melemah hingga mencapai rata-rata terendah sekitar 2,33 pada 2015 dengan harga minimum 1,62.

Sejak 2016, terlihat fase pemulihan yang kuat. Rata-rata harga meningkat dari 5,25 pada 2016 menjadi 12,41 pada 2017, 17,21 pada 2018, dan 29,94 pada 2019. Pada data 2020, rata-rata harga mencapai sekitar 48,10 dengan nilai maksimum 58,90.

Grafik rata-rata harga tahunan memperjelas bahwa AMD memiliki pola yang siklis, yaitu kenaikan tajam dapat diikuti oleh penurunan besar sebelum kembali mengalami pemulihan. Pola tersebut menunjukkan adanya perubahan kondisi atau regime pasar sepanjang periode pengamatan.

Perlu diperhatikan bahwa data tahun 2020 hanya tersedia sampai 1 April 2020. Oleh karena itu, rata-rata tahun 2020 belum mewakili satu tahun penuh dan tidak dapat dibandingkan secara langsung dengan tahun-tahun lengkap lainnya.

In [ ]:
aapl_yearly = (
    aapl.set_index("Date")["Close"]
    .resample("YE")
    .agg(["min", "mean", "median", "max"])
    .dropna()
)

aapl_yearly["Year"] = aapl_yearly.index.year
aapl_yearly.tail(15)


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(aapl_yearly["Year"], aapl_yearly["mean"])
plt.title("Annual Average Closing Price - AAPL")
plt.xlabel("Year")
plt.ylabel("Annual Average Close")
plt.tight_layout()
plt.show()


Hasil berikut merupakan analisis tahunan harga penutupan AAPL. Analisis menunjukkan bahwa harga penutupan AAPL memiliki tren kenaikan jangka panjang yang kuat, meskipun tetap mengalami beberapa periode koreksi.

Pada 2006, rata-rata harga masih sekitar 10,12. Nilai tersebut meningkat menjadi 20,28 pada 2008 dan 37,12 pada 2010. Kenaikan berlanjut hingga rata-rata mencapai 82,29 pada 2012, sebelum mengalami penurunan menjadi 67,52 pada 2013.

Setelah kembali meningkat menjadi 120,04 pada 2015, rata-rata harga turun menjadi 104,60 pada 2016. Namun, AAPL kembali mengalami pertumbuhan kuat pada tahun-tahun berikutnya, yaitu sekitar 150,55 pada 2017, 189,05 pada 2018, dan 208,26 pada 2019.

Pada data 2020, rata-rata harga mencapai 293,32 dengan nilai maksimum 327,20. Namun, data tahun 2020 hanya tersedia sampai 1 April 2020 sehingga belum mewakili satu tahun penuh dan tidak dapat dibandingkan secara langsung dengan tahun-tahun lengkap lainnya.

Grafik rata-rata harga tahunan memperlihatkan bahwa pertumbuhan AAPL semakin cepat setelah pertengahan tahun 2000-an. Walaupun terdapat beberapa penurunan sementara, arah pergerakan jangka panjang tetap meningkat. Pola ini menunjukkan adanya tren kuat dan perubahan skala harga antarperiode, sehingga data AAPL bersifat non-stationary.

### 9. Return dan Volatilitas

Return merupakan perubahan nilai investasi dalam suatu periode. Pada analisis ini, return harian dihitung berdasarkan perubahan harga penutupan saham dari satu hari perdagangan ke hari perdagangan berikutnya menggunakan rumus:

$$
r\_t =
\frac{Close\_t-Close\_{t-1}}
{Close\_{t-1}}
\times 100%
$$

Nilai return positif menunjukkan bahwa harga penutupan mengalami kenaikan dibandingkan hari sebelumnya, sedangkan return negatif menunjukkan penurunan harga. Sebagai contoh, perubahan harga dari 100 menjadi 105 menghasilkan return sebesar 5%.

Pada kode, return dihitung menggunakan fungsi `pct_change()`. Fungsi tersebut menghasilkan perubahan dalam bentuk desimal atau fractional change, sehingga hasilnya dikalikan dengan 100 agar ditampilkan dalam persentase. Observasi pertama menghasilkan nilai kosong karena belum memiliki harga sebelumnya sebagai pembanding.

Return yang dihitung pada penelitian ini merupakan **price return** karena hanya menggunakan kolom `Close`. Perhitungan tersebut belum memasukkan dividen atau bentuk pendapatan lain, sehingga tidak sama dengan total return investasi.

Volatilitas merupakan ukuran besarnya variasi return dalam suatu periode. Pada penelitian ini, volatilitas dihitung menggunakan standar deviasi return. Nilai volatilitas yang tinggi menunjukkan bahwa return lebih sering bergerak jauh dari rata-ratanya sehingga pergerakan harga lebih tidak stabil. Sebaliknya, volatilitas yang rendah menunjukkan bahwa return cenderung bergerak dalam rentang yang lebih sempit.

Volatilitas tidak menunjukkan arah pergerakan harga. Nilai volatilitas yang tinggi dapat terjadi karena kenaikan besar, penurunan besar, atau kombinasi keduanya. Selain itu, volatilitas merupakan salah satu ukuran risiko pasar, tetapi tidak mencakup seluruh jenis risiko investasi.

Kolom `Rolling Volatility 21D (%)` menghitung standar deviasi return pada 21 hari perdagangan terakhir:

$$
\sigma\_{t,21}
=============
Std(r\_{t-20},\ldots,r\_t)\times 100%
$$

Periode 21 hari perdagangan digunakan sebagai pendekatan terhadap satu bulan perdagangan. Perhitungan dilakukan secara bergerak, sehingga setiap baris menggambarkan tingkat volatilitas berdasarkan 21 return terbaru pada saat tersebut. Bagian awal data akan menghasilkan nilai kosong sampai tersedia 21 return valid yang cukup untuk melakukan perhitungan.

Nilai rolling volatility yang meningkat menunjukkan bahwa pergerakan harga pada periode tersebut menjadi lebih tidak stabil. Sebaliknya, penurunan rolling volatility menunjukkan kondisi harga yang relatif lebih tenang. Nilai pada kode masih berupa volatilitas untuk jendela return harian dan belum diannualisasi.

Analisis return dan volatilitas digunakan untuk memahami perubahan harga, tingkat ketidakstabilan, serta kejadian ekstrem pada AMD dan AAPL. Variabel tersebut digunakan sebagai bagian dari EDA, sedangkan model LSTM utama tetap menggunakan urutan nilai `Close` sesuai ketentuan soal.

**Sumber**

CFA Institute. (2024, May 3). *Why you don’t need to learn Greek to understand investment risk*. <https://www.cfainstitute.org/insights/articles/understanding-investment-risk>

CFA Institute Research and Policy Center. (2024, June 12). *Still misperceived? A fresh look at Bitcoin volatility*. <https://rpc.cfainstitute.org/blogs/enterprising-investor/2024/still-misperceived-a-fresh-look-at-bitcoin-volatility>

U.S. Securities and Exchange Commission. (n.d.). *Introduction to investing*. Investor.gov. <https://www.investor.gov/introduction-investing>

U.S. Securities and Exchange Commission. (n.d.). *What is risk?* Investor.gov. <https://www.investor.gov/introduction-investing/investing-basics/what-risk>

In [ ]:
amd_analysis = amd.copy()
amd_analysis["Daily Return"] = amd_analysis["Close"].pct_change()
amd_analysis["Daily Return (%)"] = amd_analysis["Daily Return"] * 100
amd_analysis["Rolling Volatility 21D (%)"] = (
    amd_analysis["Daily Return"].rolling(21).std() * 100
)


In [ ]:
aapl_analysis = aapl.copy()
aapl_analysis["Daily Return"] = aapl_analysis["Close"].pct_change()
aapl_analysis["Daily Return (%)"] = aapl_analysis["Daily Return"] * 100
aapl_analysis["Rolling Volatility 21D (%)"] = (
    aapl_analysis["Daily Return"].rolling(21).std() * 100
)


In [ ]:
amd_return = amd_analysis["Daily Return"].dropna()

amd_return_summary = pd.Series({
    "Ticker": "AMD",
    "Mean Daily Return (%)": amd_return.mean() * 100,
    "Daily Volatility (%)": amd_return.std() * 100,
    "1% Quantile (%)": amd_return.quantile(0.01) * 100,
    "99% Quantile (%)": amd_return.quantile(0.99) * 100,
    "Largest Increase (%)": amd_return.max() * 100,
    "Largest Increase Date": amd_analysis.loc[amd_return.idxmax(), "Date"],
    "Largest Decrease (%)": amd_return.min() * 100,
    "Largest Decrease Date": amd_analysis.loc[amd_return.idxmin(), "Date"]
}, name="AMD")

amd_return_summary.to_frame("Value")


Hasil analisis menunjukkan bahwa AMD memiliki rata-rata return harian sebesar 0,0976%. Nilai positif ini mengindikasikan bahwa secara historis harga AMD cenderung meningkat secara rata-rata per hari perdagangan. Namun, nilai tersebut tidak berarti harga selalu naik sebesar 0,0976% setiap hari karena return aktual dapat berubah secara signifikan.

Volatilitas harian AMD sebesar 3,7787%, yang dihitung dari standar deviasi return harian. Nilai ini menunjukkan bahwa return AMD memiliki penyebaran yang cukup besar di sekitar rata-ratanya, sehingga pergerakan harga hariannya relatif berfluktuasi.

Nilai kuantil 1% sebesar -9,7905% berarti sekitar 1% return harian terburuk berada pada nilai tersebut atau lebih rendah. Sementara itu, kuantil 99% sebesar 10,9178% menunjukkan bahwa sekitar 1% return harian tertinggi berada di atas nilai tersebut. Dengan demikian, sebagian besar return harian berada di antara sekitar -9,79% dan 10,92%.

Kenaikan harian terbesar AMD tercatat sebesar 52,2901% pada 22 April 2016. Sebaliknya, penurunan harian terbesar mencapai -37,9310% pada 18 Juni 1992. Kedua nilai tersebut menunjukkan bahwa AMD pernah mengalami perubahan harga harian yang sangat ekstrem.

Secara keseluruhan, AMD memiliki return rata-rata harian yang positif, tetapi disertai volatilitas dan perubahan ekstrem yang cukup tinggi. Nilai ekstrem tersebut tidak langsung dihapus sebagai outlier karena dapat merepresentasikan kejadian pasar yang benar-benar terjadi dan merupakan bagian penting dari karakteristik historis saham AMD.

#### 9A. Daily Return

In [ ]:
aapl_return = aapl_analysis["Daily Return"].dropna()

aapl_return_summary = pd.Series({
    "Ticker": "AAPL",
    "Mean Daily Return (%)": aapl_return.mean() * 100,
    "Daily Volatility (%)": aapl_return.std() * 100,
    "1% Quantile (%)": aapl_return.quantile(0.01) * 100,
    "99% Quantile (%)": aapl_return.quantile(0.99) * 100,
    "Largest Increase (%)": aapl_return.max() * 100,
    "Largest Increase Date": aapl_analysis.loc[aapl_return.idxmax(), "Date"],
    "Largest Decrease (%)": aapl_return.min() * 100,
    "Largest Decrease Date": aapl_analysis.loc[aapl_return.idxmin(), "Date"]
}, name="AAPL")

aapl_return_summary.to_frame("Value")


In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(amd_analysis["Date"], amd_analysis["Daily Return (%)"])
plt.axhline(0)
plt.title("Daily Return - AMD")
plt.xlabel("Date")
plt.ylabel("Daily Return (%)")
plt.tight_layout()
plt.show()


Grafik menunjukkan pergerakan return harian AMD sepanjang periode pengamatan. Sebagian besar return berfluktuasi di sekitar 0%, yang berarti perubahan harga dari satu hari perdagangan ke hari berikutnya umumnya relatif kecil dibandingkan beberapa kejadian ekstrem.

Meskipun demikian, terdapat sejumlah lonjakan positif dan negatif yang besar. Lonjakan tertinggi terlihat pada 2016 dengan return sekitar 52,29%, sedangkan penurunan terbesar terjadi pada 1992 dengan return sekitar -37,93%. Nilai tersebut sesuai dengan hasil ringkasan statistik sebelumnya.

Grafik juga memperlihatkan adanya **volatility clustering**, yaitu periode dengan perubahan harga yang besar cenderung berdekatan dengan perubahan besar lainnya, sedangkan periode yang tenang cenderung diikuti oleh pergerakan yang relatif kecil. Pola ini terlihat terutama pada sekitar akhir 1990-an hingga awal 2000-an serta beberapa periode setelah 2008.

Dibandingkan grafik harga `Close`, return harian terlihat lebih banyak berpusat di sekitar nilai yang relatif tetap. Namun, besar penyebarannya berubah sepanjang waktu. Hal ini menunjukkan bahwa rata-rata return relatif lebih stabil, tetapi varians atau volatilitasnya tidak konstan.

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(aapl_analysis["Date"], aapl_analysis["Daily Return (%)"])
plt.axhline(0)
plt.title("Daily Return - AAPL")
plt.xlabel("Date")
plt.ylabel("Daily Return (%)")
plt.tight_layout()
plt.show()


Hasil analisis menunjukkan bahwa AAPL memiliki rata-rata return harian sebesar 0,1041%. Nilai positif ini mengindikasikan bahwa secara historis harga AAPL cenderung meningkat secara rata-rata per hari perdagangan, meskipun return aktual tetap dapat bernilai positif maupun negatif.

Volatilitas harian AAPL sebesar 2,8738%, yang menunjukkan tingkat penyebaran return harian di sekitar nilai rata-ratanya. Nilai ini memperlihatkan bahwa perubahan harga AAPL dari satu hari perdagangan ke hari berikutnya cukup berfluktuasi.

Kuantil 1% sebesar -7,1427% berarti sekitar 1% return harian berada pada nilai -7,14% atau lebih rendah. Sementara itu, kuantil 99% sebesar 8,0231% menunjukkan bahwa sekitar 1% return harian berada pada nilai 8,02% atau lebih tinggi. Dengan demikian, sekitar 98% return harian berada di antara kedua batas tersebut.

Kenaikan harian terbesar AAPL tercatat sebesar 33,2278% pada 6 Agustus 1997. Sebaliknya, penurunan harian terbesar mencapai -51,8692% pada 29 September 2000. Hasil tersebut menunjukkan bahwa AAPL pernah mengalami pergerakan harga harian yang sangat ekstrem.

Secara keseluruhan, AAPL memiliki rata-rata return harian yang positif, tetapi tetap disertai fluktuasi dan risiko penurunan yang besar pada periode tertentu. Nilai ekstrem tersebut tidak langsung dihapus karena dapat merepresentasikan kejadian pasar yang benar-benar terjadi dalam data historis.

#### 21 Day Trading Day Rolling Votality

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(
    amd_analysis["Date"],
    amd_analysis["Rolling Volatility 21D (%)"]
)
plt.title("21-Trading-Day Rolling Volatility - AMD")
plt.xlabel("Date")
plt.ylabel("Rolling Volatility (%)")
plt.tight_layout()
plt.show()


Grafik menunjukkan pergerakan return harian AAPL sepanjang periode pengamatan. Sebagian besar nilai return berfluktuasi di sekitar 0%, yang berarti perubahan harga dari satu hari perdagangan ke hari berikutnya umumnya relatif kecil dibandingkan beberapa kejadian ekstrem.

Meskipun demikian, terlihat sejumlah lonjakan positif dan negatif yang besar. Kenaikan harian terbesar mencapai sekitar 33,23% pada 6 Agustus 1997, sedangkan penurunan terbesar mencapai sekitar -51,87% pada 29 September 2000. Penurunan tersebut tampak sebagai lonjakan negatif paling ekstrem pada grafik.

Grafik juga menunjukkan adanya **volatility clustering**, yaitu periode dengan perubahan besar cenderung berdekatan dengan perubahan besar lainnya. Pola ini terlihat terutama pada akhir 1990-an hingga awal 2000-an, sekitar 2008, dan awal 2020. Sebaliknya, terdapat pula periode ketika return bergerak dalam rentang yang lebih sempit dan relatif stabil.

Dibandingkan data harga `Close` yang memiliki tren naik kuat, return harian lebih banyak berpusat di sekitar nilai yang relatif tetap. Namun, besar penyebarannya berubah sepanjang waktu. Hal ini menunjukkan bahwa rata-rata return relatif lebih stabil, tetapi varians atau volatilitasnya tidak konstan.

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(
    aapl_analysis["Date"],
    aapl_analysis["Rolling Volatility 21D (%)"]
)
plt.title("21-Trading-Day Rolling Volatility - AAPL")
plt.xlabel("Date")
plt.ylabel("Rolling Volatility (%)")
plt.tight_layout()
plt.show()


Grafik menunjukkan perubahan volatilitas AAPL yang dihitung menggunakan standar deviasi return selama 21 hari perdagangan terakhir. Periode 21 hari digunakan sebagai pendekatan terhadap satu bulan perdagangan.

Nilai rolling volatility tidak konstan sepanjang waktu. Pada periode yang relatif tenang, volatilitas berada pada kisaran sekitar 1%–3%. Namun, pada beberapa periode terjadi lonjakan yang menunjukkan bahwa return harian AAPL bergerak jauh lebih tidak stabil.

Lonjakan terbesar terlihat sekitar tahun 2000 dengan volatilitas lebih dari 12%. Lonjakan lain juga muncul sekitar 1983–1984, 1987, 1997, 2008, dan awal 2020. Hal ini menunjukkan adanya **volatility clustering**, yaitu periode dengan volatilitas tinggi cenderung muncul secara berkelompok dan kemudian diikuti oleh periode yang lebih stabil.

Secara keseluruhan, grafik memperlihatkan bahwa risiko perubahan harga AAPL berubah dari waktu ke waktu. Oleh karena itu, meskipun rata-rata return dapat terlihat relatif stabil, tingkat ketidakpastian atau varians return tidak dapat dianggap konstan. Nilai volatilitas pada grafik ini belum diannualisasi karena masih dihitung langsung dari return harian dalam rolling window 21 hari.

### 10. Maximum Drawdown

Maximum drawdown merupakan penurunan terbesar harga suatu aset dari nilai tertinggi sebelumnya atau **peak** menuju nilai terendah setelahnya atau **trough** selama periode pengamatan. Metrik ini menunjukkan kerugian historis terburuk yang mungkin dialami apabila aset dibeli pada titik puncak dan nilainya diukur kembali pada titik terendah sebelum pulih.

Drawdown pada waktu ke-(t) dihitung menggunakan rumus:

$$
Drawdown\_t =
\left(
\frac{Close\_t}{Running\ Peak\_t}-1
\right)
\times 100%
$$

`Running Peak` merupakan harga penutupan tertinggi yang pernah tercapai sampai waktu tersebut. Ketika harga berada pada puncak terbaru, nilai drawdown adalah 0%. Apabila harga berada di bawah puncaknya, drawdown bernilai negatif.

Sebagai contoh, apabila harga mencapai puncak sebesar 100 lalu turun menjadi 60, drawdown-nya adalah:

$$
\left(\frac{60}{100}-1\right)\times100%=-40%
$$

Artinya, harga mengalami penurunan sebesar 40% dari puncak sebelumnya. Maximum drawdown adalah nilai drawdown yang paling negatif selama seluruh periode pengamatan.

Maximum drawdown membantu mengukur **downside risk**, yaitu seberapa dalam penurunan historis yang pernah terjadi. Berbeda dari volatilitas yang mengukur keseluruhan penyebaran return positif dan negatif, maximum drawdown berfokus pada penurunan dari puncak menuju titik terendah. Namun, maximum drawdown hanya menggambarkan kejadian historis dan tidak menjamin besarnya penurunan yang akan terjadi pada masa depan.

Pada kode AMD, `cummax()` digunakan untuk menghitung harga tertinggi berjalan pada setiap tanggal. Drawdown kemudian dihitung dengan membandingkan nilai `Close` terhadap harga tertinggi berjalan tersebut.

Fungsi `idxmin()` digunakan untuk mencari baris dengan drawdown paling negatif sebagai titik `trough`. Setelah itu, kode mencari harga tertinggi yang terjadi sebelum trough sebagai titik `peak`. Data setelah trough diperiksa untuk menemukan tanggal pertama ketika harga kembali mencapai atau melewati harga peak sebelumnya. Tanggal tersebut ditetapkan sebagai `Recovery Date`.

Apabila harga belum kembali mencapai peak sampai akhir dataset, nilai `Recovery Date` akan ditampilkan sebagai `NaT`. Ringkasan akhir menampilkan persentase maximum drawdown, tanggal dan harga peak, tanggal dan harga trough, serta tanggal pemulihan.

Perhitungan ini menggunakan kolom `Close` sesuai ketentuan soal. Analisis maximum drawdown digunakan sebagai bagian dari EDA untuk memahami penurunan historis terburuk dan perubahan regime harga, bukan sebagai fitur langsung pada model LSTM.

**Sumber**

CFA Institute Research and Policy Center. (2013, February 12). *Sculpting investment portfolios: Maximum drawdown and optimal portfolio strategy*. <https://rpc.cfainstitute.org/blogs/enterprising-investor/2013/sculpting-investment-portfolios-maximum-drawdown-and-optimal-portfolio-strategy>

CFA Institute Research and Policy Center. (2018). *Performance drawdowns in asset management*. <https://rpc.cfainstitute.org/research/cfa-digest/2018/01/dig-v48-n1-3>

Morningstar. (n.d.). *Maximum drawdown*. <https://www.morningstar.com/investing-terms/maximum-drawdown>

In [ ]:
amd_drawdown = amd[["Date", "Close"]].copy()
amd_drawdown["Running Peak"] = amd_drawdown["Close"].cummax()
amd_drawdown["Drawdown"] = (
    amd_drawdown["Close"] / amd_drawdown["Running Peak"] - 1
)


In [ ]:
amd_trough_idx = amd_drawdown["Drawdown"].idxmin()
amd_peak_idx = amd_drawdown.loc[:amd_trough_idx, "Close"].idxmax()
amd_peak_price = amd_drawdown.loc[amd_peak_idx, "Close"]

amd_recovery = amd_drawdown.loc[amd_trough_idx + 1:]
amd_recovery = amd_recovery[amd_recovery["Close"] >= amd_peak_price]


In [ ]:
amd_drawdown_summary = pd.Series({
    "Ticker": "AMD",
    "Maximum Drawdown (%)": amd_drawdown.loc[amd_trough_idx, "Drawdown"] * 100,
    "Peak Date": amd_drawdown.loc[amd_peak_idx, "Date"],
    "Peak Price": amd_peak_price,
    "Trough Date": amd_drawdown.loc[amd_trough_idx, "Date"],
    "Trough Price": amd_drawdown.loc[amd_trough_idx, "Close"],
    "Recovery Date": (
        pd.NaT if amd_recovery.empty
        else amd_recovery.iloc[0]["Date"]
    )
}, name="AMD")

amd_drawdown_summary.to_frame("Value")


Hasil menunjukkan bahwa AMD mengalami maximum drawdown sebesar **-96,59%**. Penurunan tersebut terjadi dari harga puncak sebesar **47,50** pada **21 Juni 2000** hingga mencapai harga terendah sebesar **1,62** pada **27 Juli 2015**.

Nilai -96,59% berarti bahwa harga AMD kehilangan hampir seluruh nilainya dibandingkan dengan puncak sebelumnya. Sebagai contoh, investasi senilai 100 pada titik puncak secara teoritis hanya tersisa sekitar 3,41 pada titik terendah. Nilai ini merupakan penurunan kumulatif dari peak ke trough, bukan penurunan yang terjadi dalam satu hari.

Perjalanan dari titik puncak menuju titik terendah berlangsung sekitar 15 tahun. Setelah mencapai trough pada 2015, harga AMD pertama kali kembali mencapai atau melewati harga puncak 47,50 pada **2 Januari 2020**. Dengan demikian, proses pemulihan dari trough membutuhkan sekitar 4 tahun 5 bulan, sedangkan keseluruhan periode dari peak hingga recovery berlangsung hampir 20 tahun.

Hasil ini menunjukkan bahwa AMD pernah mengalami penurunan historis yang sangat dalam serta membutuhkan waktu pemulihan yang panjang. Kondisi tersebut memperkuat temuan bahwa pola harga AMD bersifat siklis dan mengalami perubahan regime pasar yang besar sepanjang periode pengamatan.

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(amd_drawdown["Date"], amd_drawdown["Drawdown"] * 100)
plt.axhline(0)
plt.title("Historical Drawdown - AMD")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.tight_layout()
plt.show()


Grafik menunjukkan drawdown historis AMD, yaitu persentase penurunan harga `Close` dari harga tertinggi yang pernah tercapai sebelumnya atau `running peak`.

Garis pada 0% menunjukkan bahwa harga sedang berada pada puncak baru. Semakin jauh nilai drawdown berada di bawah 0%, semakin besar penurunan harga dibandingkan puncak sebelumnya.

Grafik memperlihatkan bahwa AMD mengalami beberapa periode penurunan yang dalam dan berlangsung lama. Penurunan besar terlihat pada akhir 1980-an hingga awal 1990-an, awal 2000-an, serta periode setelah 2007. Drawdown terdalam terjadi sekitar 2015 dan mencapai kurang lebih -96,59%, sesuai dengan hasil perhitungan maximum drawdown sebelumnya.

Setelah puncak harga pada 21 Juni 2000, AMD berada di bawah harga puncak tersebut selama hampir dua dekade. Harga baru kembali mendekati garis 0% pada awal 2020, yang menunjukkan bahwa harga telah kembali mencapai atau melewati puncak sebelumnya.

Secara keseluruhan, grafik menunjukkan bahwa AMD memiliki risiko penurunan yang sangat besar dan periode pemulihan yang panjang. Hal ini memperkuat kesimpulan bahwa pergerakan harga AMD bersifat siklis dan mengalami perubahan kondisi pasar yang kuat sepanjang waktu.

In [ ]:
aapl_drawdown = aapl[["Date", "Close"]].copy()
aapl_drawdown["Running Peak"] = aapl_drawdown["Close"].cummax()
aapl_drawdown["Drawdown"] = (
    aapl_drawdown["Close"] / aapl_drawdown["Running Peak"] - 1
)


In [ ]:
aapl_trough_idx = aapl_drawdown["Drawdown"].idxmin()
aapl_peak_idx = aapl_drawdown.loc[:aapl_trough_idx, "Close"].idxmax()
aapl_peak_price = aapl_drawdown.loc[aapl_peak_idx, "Close"]

aapl_recovery = aapl_drawdown.loc[aapl_trough_idx + 1:]
aapl_recovery = aapl_recovery[aapl_recovery["Close"] >= aapl_peak_price]


In [ ]:
aapl_drawdown_summary = pd.Series({
    "Ticker": "AAPL",
    "Maximum Drawdown (%)": aapl_drawdown.loc[aapl_trough_idx, "Drawdown"] * 100,
    "Peak Date": aapl_drawdown.loc[aapl_peak_idx, "Date"],
    "Peak Price": aapl_peak_price,
    "Trough Date": aapl_drawdown.loc[aapl_trough_idx, "Date"],
    "Trough Price": aapl_drawdown.loc[aapl_trough_idx, "Close"],
    "Recovery Date": (
        pd.NaT if aapl_recovery.empty
        else aapl_recovery.iloc[0]["Date"]
    )
}, name="AAPL")

aapl_drawdown_summary.to_frame("Value")


In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(aapl_drawdown["Date"], aapl_drawdown["Drawdown"] * 100)
plt.axhline(0)
plt.title("Historical Drawdown - AAPL")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.tight_layout()
plt.show()


Grafik menunjukkan drawdown historis AAPL, yaitu persentase penurunan harga `Close` dari harga tertinggi yang telah tercapai sebelumnya atau `running peak`.

Garis 0% menunjukkan bahwa harga sedang berada pada puncak baru. Ketika garis bergerak semakin jauh ke bawah, berarti harga mengalami penurunan yang semakin besar dari puncak sebelumnya. Harga baru dianggap pulih ketika drawdown kembali mencapai 0%.

Drawdown terdalam terlihat pada akhir 1990-an dan mencapai sekitar **-82,22%**. Nilai tersebut sesuai dengan hasil maximum drawdown sebelumnya, yaitu penurunan dari harga puncak pada 2 April 1991 menuju titik terendah pada 23 Desember 1997. Harga kemudian kembali mencapai puncak sebelumnya pada 3 September 1999.

Selain penurunan maksimum tersebut, grafik juga menunjukkan beberapa periode drawdown besar lainnya, terutama pada awal 1980-an, awal 2000-an, sekitar 2008, serta beberapa koreksi setelah 2010. Namun, sebagian drawdown pada periode selanjutnya memiliki durasi pemulihan yang lebih pendek dibandingkan penurunan pada 1990-an.

Secara keseluruhan, grafik memperlihatkan bahwa meskipun harga AAPL memiliki tren kenaikan jangka panjang, saham ini tetap mengalami beberapa periode penurunan yang dalam. Hal ini menunjukkan bahwa tren pertumbuhan jangka panjang tidak menghilangkan risiko kerugian sementara dan waktu pemulihan yang dapat berlangsung cukup lama.

### 11. Autokorelasi

Autokorelasi adalah hubungan antara suatu nilai time series dengan nilai dirinya sendiri pada periode sebelumnya.

Contohnya:

Lag 1: membandingkan harga penutupan hari ini dengan harga penutupan satu hari perdagangan sebelumnya.
Lag 5: membandingkan harga hari ini dengan harga lima hari perdagangan sebelumnya.

In [ ]:
autocorr_summary = pd.DataFrame({
    "Ticker": ["AMD", "AAPL"],
    "Close Lag-1": [
        amd["Close"].autocorr(1),
        aapl["Close"].autocorr(1)
    ],
    "Return Lag-1": [
        amd_return.autocorr(1),
        aapl_return.autocorr(1)
    ]
})

autocorr_summary


In [ ]:
amd_close_acf = pd.DataFrame({
    "Lag": range(1, 31),
    "Autocorrelation": [
        amd["Close"].autocorr(lag)
        for lag in range(1, 31)
    ]
})


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(amd_close_acf["Lag"], amd_close_acf["Autocorrelation"])
plt.axhline(0)
plt.title("Close Autocorrelation - AMD")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.tight_layout()
plt.show()


Grafik menunjukkan bahwa harga penutupan AMD memiliki autokorelasi yang sangat tinggi pada seluruh lag 1–30. Pada lag 1, nilainya mendekati 1 dan kemudian menurun secara perlahan hingga tetap berada di atas sekitar 0,9 pada lag 30.

Artinya, harga `Close` AMD saat ini masih memiliki hubungan linear yang kuat dengan harga beberapa hari perdagangan sebelumnya. Lag 5 kira-kira mewakili satu minggu perdagangan, sedangkan lag 20–30 mewakili sekitar satu hingga satu setengah bulan perdagangan.

Pola penurunan yang sangat lambat menunjukkan adanya persistensi, tren, dan sifat non-stationary pada level harga AMD. Hasil ini mendukung penggunaan data berurutan dalam LSTM, termasuk window lima hari. Namun, autokorelasi tinggi tidak otomatis menunjukkan bahwa model dapat memprediksi perubahan harga dengan mudah, karena model dapat menghasilkan prediksi yang terlihat baik hanya dengan mengikuti harga terakhir.

In [ ]:
amd_return_acf = pd.DataFrame({
    "Lag": range(1, 31),
    "Autocorrelation": [
        amd_return.autocorr(lag)
        for lag in range(1, 31)
    ]
})


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(amd_return_acf["Lag"], amd_return_acf["Autocorrelation"])
plt.axhline(0)
plt.title("Daily Return Autocorrelation - AMD")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.tight_layout()
plt.show()


Grafik menunjukkan bahwa autokorelasi return harian AMD pada lag 1–30 berada sangat dekat dengan nol, dengan nilai hanya sekitar -0,03 sampai 0,03. Artinya, return pada suatu hari memiliki hubungan linear yang sangat lemah dengan return pada beberapa hari perdagangan sebelumnya.

Beberapa lag menunjukkan nilai positif atau negatif kecil, tetapi tidak membentuk pola yang konsisten. Hal ini mengindikasikan bahwa arah dan besar perubahan harga AMD berikutnya sulit diprediksi hanya menggunakan return masa lalu.

Namun, grafik ini belum menampilkan batas confidence interval, sehingga signifikansi statistik setiap lag belum dapat ditentukan. Secara umum, hasil ini menunjukkan bahwa meskipun harga `Close` memiliki autokorelasi tinggi, return hariannya hampir tidak memiliki ketergantungan linear.

In [ ]:
aapl_close_acf = pd.DataFrame({
    "Lag": range(1, 31),
    "Autocorrelation": [
        aapl["Close"].autocorr(lag)
        for lag in range(1, 31)
    ]
})


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(aapl_close_acf["Lag"], aapl_close_acf["Autocorrelation"])
plt.axhline(0)
plt.title("Close Autocorrelation - AAPL")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.tight_layout()
plt.show()


Grafik menunjukkan bahwa harga penutupan AAPL memiliki autokorelasi yang sangat tinggi pada seluruh lag 1–30. Nilainya tetap mendekati 1 bahkan hingga lag 30, sehingga harga `Close` saat ini masih memiliki hubungan linear yang sangat kuat dengan harga beberapa hari perdagangan sebelumnya.

Lag 5 kira-kira mewakili satu minggu perdagangan, sedangkan lag 20–30 mewakili sekitar satu hingga satu setengah bulan perdagangan. Tingginya nilai autokorelasi pada seluruh lag menunjukkan bahwa level harga AAPL memiliki persistensi dan tren jangka panjang yang sangat kuat.

Namun, pola ini juga menjadi indikasi bahwa data `Close` bersifat non-stationary. Autokorelasi yang hampir sempurna tidak berarti perubahan harga berikutnya mudah diprediksi, karena hubungan tersebut sebagian besar dapat berasal dari tren harga. Oleh karena itu, performa model nantinya perlu dievaluasi dengan hati-hati agar model tidak hanya meniru harga terakhir.

In [ ]:
aapl_return_acf = pd.DataFrame({
    "Lag": range(1, 31),
    "Autocorrelation": [
        aapl_return.autocorr(lag)
        for lag in range(1, 31)
    ]
})


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(aapl_return_acf["Lag"], aapl_return_acf["Autocorrelation"])
plt.axhline(0)
plt.title("Daily Return Autocorrelation - AAPL")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.tight_layout()
plt.show()


Grafik menunjukkan bahwa autokorelasi return harian AAPL pada lag 1–30 berada sangat dekat dengan nol, yaitu sekitar -0,03 sampai 0,04. Artinya, return AAPL pada suatu hari memiliki hubungan linear yang sangat lemah dengan return pada beberapa hari perdagangan sebelumnya.

Nilai positif tertinggi terlihat sekitar lag 12, sedangkan beberapa nilai negatif terlihat pada lag 2, 3, 10, 21, dan 22. Namun, nilainya tetap kecil dan tidak membentuk pola yang konsisten.

Hasil ini menunjukkan bahwa meskipun harga `Close` AAPL memiliki autokorelasi yang sangat tinggi, perubahan harga hariannya jauh lebih sulit diprediksi hanya berdasarkan return masa lalu. Grafik ini belum menampilkan confidence interval, sehingga signifikansi statistik setiap lag belum dapat dipastikan.

### 12. Rolling Statistics

In [ ]:
amd_rolling = amd[["Date", "Close"]].copy()
amd_rolling["Rolling Mean"] = amd_rolling["Close"].rolling(252).mean()
amd_rolling["Rolling Std"] = amd_rolling["Close"].rolling(252).std()


In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(amd_rolling["Date"], amd_rolling["Close"], label="Close")
plt.plot(amd_rolling["Date"], amd_rolling["Rolling Mean"], label="Rolling Mean")
plt.plot(amd_rolling["Date"], amd_rolling["Rolling Std"], label="Rolling Std")
plt.title("252-Day Rolling Statistics - AMD")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()


Grafik menampilkan harga penutupan AMD beserta rolling mean dan rolling standard deviation menggunakan window 252 hari perdagangan, yang merepresentasikan sekitar satu tahun perdagangan.

Garis `Close` menunjukkan harga penutupan aktual, sedangkan garis `Rolling Mean` menggambarkan rata-rata harga selama 252 hari terakhir. Rolling mean terlihat lebih halus dan tertinggal dari harga aktual karena setiap nilainya merangkum data satu tahun sebelumnya.

Garis `Rolling Std` menunjukkan tingkat penyebaran harga dalam window yang sama. Nilainya meningkat pada periode ketika harga mengalami perubahan besar, terutama sekitar tahun 2000, 2006, dan menjelang 2020. Sebaliknya, rolling standard deviation lebih rendah ketika harga bergerak dalam rentang yang relatif sempit.

Perubahan rolling mean dan rolling standard deviation sepanjang waktu menunjukkan bahwa rata-rata dan penyebaran harga AMD tidak konstan. Dengan demikian, data `Close` bersifat non-stationary dan mengalami beberapa perubahan regime pasar.

Rolling standard deviation pada grafik ini dihitung dari level harga `Close`, sehingga menggambarkan penyebaran harga selama satu tahun terakhir, bukan volatilitas return harian.

In [ ]:
aapl_rolling = aapl[["Date", "Close"]].copy()
aapl_rolling["Rolling Mean"] = aapl_rolling["Close"].rolling(252).mean()
aapl_rolling["Rolling Std"] = aapl_rolling["Close"].rolling(252).std()


In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(aapl_rolling["Date"], aapl_rolling["Close"], label="Close")
plt.plot(aapl_rolling["Date"], aapl_rolling["Rolling Mean"], label="Rolling Mean")
plt.plot(aapl_rolling["Date"], aapl_rolling["Rolling Std"], label="Rolling Std")
plt.title("252-Day Rolling Statistics - AAPL")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()


Grafik menampilkan harga penutupan AAPL beserta rolling mean dan rolling standard deviation menggunakan window 252 hari perdagangan, yang merepresentasikan sekitar satu tahun perdagangan.

Garis `Close` menunjukkan harga penutupan aktual, sedangkan garis `Rolling Mean` menunjukkan rata-rata harga selama 252 hari terakhir. Rolling mean bergerak lebih halus dan tertinggal dari harga aktual karena setiap nilainya merangkum data satu tahun sebelumnya. Grafik memperlihatkan tren kenaikan jangka panjang yang semakin kuat, terutama setelah pertengahan tahun 2000-an.

Garis `Rolling Std` menunjukkan penyebaran nilai harga `Close` selama 252 hari terakhir. Nilainya meningkat ketika skala harga dan perubahan harga membesar, terutama setelah 2010 dan menjelang 2020. Hal ini menunjukkan bahwa variasi level harga AAPL tidak sama pada seluruh periode.

Perubahan rolling mean dan rolling standard deviation sepanjang waktu menunjukkan bahwa rata-rata dan penyebaran harga AAPL tidak konstan. Dengan demikian, data `Close` bersifat non-stationary dan mengalami perubahan skala serta regime pasar yang kuat.

Rolling standard deviation pada grafik ini dihitung dari level harga `Close`, bukan dari return harian. Oleh karena itu, nilainya menggambarkan penyebaran harga selama satu tahun terakhir dan tidak dapat langsung diartikan sebagai volatilitas return.

## SUMMARY EDA

Hasil EDA tidak hanya digunakan untuk mendeskripsikan data, tetapi juga menjadi dasar dalam menentukan preprocessing, pembagian data, model pembanding, serta arah pengembangan model yang akan dilakukan pada tahap selanjutnya.

- Sesuai ketentuan soal, hanya kolom `Date` dan `Close` yang digunakan. `Date` berfungsi untuk menjaga urutan waktu, sedangkan `Close` digunakan sebagai input sequence dan target prediksi.
- Tidak ditemukan missing value, duplikasi tanggal, nilai `Close` nol, nilai negatif, maupun nilai tidak terbatas. Dengan demikian, tidak diperlukan imputasi atau penghapusan baris pada kolom yang digunakan untuk modeling.
- Nilai ekstrem pada harga dan return tidak langsung dihapus karena dapat merepresentasikan perubahan pasar yang benar-benar terjadi. Nilai tersebut tetap dipertahankan agar model juga belajar dari periode kenaikan, penurunan, dan shock pasar.
- Anomali pada kolom `Open` dan `Volume` AMD tidak memengaruhi modeling karena kedua kolom tersebut tidak digunakan. Seluruh baris tetap dipertahankan agar urutan historis `Close` tidak terputus.
- AMD dan AAPL memiliki rentang harga yang berbeda. Oleh karena itu, setiap saham akan menggunakan scaler yang berbeda dan tidak akan digabungkan dalam satu proses scaling.
- Scaling tidak dilakukan secara permanen pada tahap EDA. EDA tetap menggunakan harga asli agar tren, volatilitas, nilai minimum, maksimum, return, dan drawdown dapat diinterpretasikan dalam skala harga sebenarnya.
- Scaling baru dilakukan setelah pembagian data. Scaler hanya di-fit menggunakan data train, kemudian digunakan untuk mentransformasikan validation dan test. Hal ini dilakukan untuk mencegah data leakage.
- Hasil EDA menunjukkan bahwa rata-rata harga, rolling mean, dan rolling standard deviation berubah sepanjang waktu. Hal tersebut menunjukkan bahwa data bersifat non-stationary dan mengalami perubahan regime.
- Karena data memiliki urutan waktu dan distribusinya berubah, pembagian data akan dilakukan secara kronologis tanpa random split.
- Data juga tidak akan diacak selama training karena `shuffle` dapat merusak hubungan temporal antara harga sebelumnya dan harga berikutnya.
- Test set akan menggunakan satu tahun kalender terakhir. Data sebelum test akan dibagi secara kronologis menjadi train dan validation.
- Validation akan digunakan untuk membandingkan kandidat model dan memilih konfigurasi terbaik. Test set hanya digunakan setelah model terbaik selesai ditentukan.
- Hasil EDA menunjukkan bahwa autokorelasi lag-1 `Close` sangat tinggi, yaitu sekitar 0,9979 pada AMD dan 0,9997 pada AAPL. Artinya, harga hari berikutnya biasanya berada dekat dengan harga hari sebelumnya.
- Tingginya autokorelasi `Close` mendukung penggunaan model berbasis sequence seperti LSTM karena nilai-nilai harga sebelumnya memiliki hubungan temporal yang kuat dengan harga berikutnya.
- Namun, autokorelasi lag-1 return sangat rendah, yaitu sekitar 0,009 pada AMD dan 0,019 pada AAPL. Artinya, meskipun level harga mudah terlihat persisten, arah dan besar perubahan harga berikutnya jauh lebih sulit diprediksi.
- Perbedaan tersebut menunjukkan bahwa model dapat terlihat mengikuti grafik aktual hanya karena harga hari berurutan memang berdekatan. Oleh karena itu, keberhasilan model tidak hanya dinilai dari bentuk grafik prediksi.
- Berdasarkan tingginya autokorelasi `Close`, naive baseline akan digunakan sebagai pembanding. Naive baseline memprediksi harga berikutnya sama dengan harga terakhir yang tersedia:

$$
\hat{P}\_{t+1}=P\_t
$$

- Naive baseline digunakan untuk menilai apakah LSTM benar-benar mempelajari pola tambahan atau hanya menghasilkan prediksi yang mendekati harga terakhir.
- Baseline LSTM akan mengikuti konfigurasi yang diwajibkan oleh soal, yaitu menggunakan lima hari perdagangan terakhir untuk memprediksi satu hari perdagangan berikutnya.
- Baseline menggunakan harga `Close` asli yang dinormalisasi. Transformasi log atau representasi lain belum langsung digunakan agar baseline tetap sederhana dan sesuai dengan ketentuan soal.
- EDA menunjukkan bahwa AMD memiliki volatilitas harian yang lebih tinggi dibandingkan AAPL. AMD juga memiliki pola yang lebih siklis, perubahan arah yang tajam, dan beberapa penurunan serta pemulihan yang panjang.
- Berdasarkan karakter tersebut, pada tahap modified model AMD akan dicoba window yang lebih panjang daripada lima hari. Window sekitar 20 dan 60 hari dapat diuji untuk mengetahui apakah konteks satu hingga tiga bulan perdagangan lebih sesuai dalam menangkap pola AMD.
- Karena AMD memiliki perubahan harga yang lebih ekstrem, pada tahap modified model juga dapat diuji metode training yang lebih tahan terhadap error besar, regularisasi yang berbeda, serta learning rate yang lebih kecil.
- EDA menunjukkan bahwa AAPL memiliki tren kenaikan jangka panjang yang lebih kuat, distribusi `Close` yang sangat right-skewed, serta perubahan skala harga yang besar dari periode awal ke periode akhir.
- Kondisi tersebut dapat menyulitkan model apabila model hanya mempelajari harga absolut. Model mungkin harus memprediksi pada rentang harga yang jauh lebih tinggi daripada rentang yang dominan pada periode sebelumnya.
- Oleh karena itu, pada tahap modified model AAPL akan dibandingkan dua pendekatan, yaitu tetap menggunakan harga absolut dan menggunakan representasi log atau perubahan relatif.
- Transformasi log dipertimbangkan karena dapat mengompresi rentang harga yang sangat lebar dan mengurangi dominasi nilai harga yang besar.
- Selain transformasi log terhadap level harga, akan dipertimbangkan pendekatan relative log-price. Pada pendekatan ini, model tidak langsung memprediksi harga absolut, tetapi mempelajari perubahan relatif terhadap harga terakhir.
- Relative log-price sesuai dengan hasil EDA karena harga terakhir memiliki hubungan yang sangat kuat dengan harga berikutnya. Harga terakhir dapat digunakan sebagai anchor, sedangkan model mempelajari koreksi naik atau turun terhadap harga tersebut.
- Secara umum, relative log-price dirumuskan sebagai:

$$
x\_t=
\log\left(
\frac{P\_t}{P\_{\mathrm{last}}}
\right)
$$

dengan:

- $P\_t$ merupakan harga pada posisi tertentu di dalam window.
- $P\_{\mathrm{last}}$ merupakan harga terakhir yang tersedia di dalam window.
- $x\_t$ merupakan posisi harga pada waktu ke-$t$ relatif terhadap harga terakhir.

Sedangkan target perubahan harga berikutnya dirumuskan sebagai:

$$
y=
\log\left(
\frac{P\_{\mathrm{next}}}{P\_{\mathrm{last}}}
\right)
$$

dengan:

- $P\_{\mathrm{next}}$ merupakan harga aktual pada hari perdagangan berikutnya.
- $P\_{\mathrm{last}}$ merupakan harga terakhir pada sequence input.
- $y$ merupakan log-return yang akan diprediksi oleh model.
- Prediksi kemudian dapat dikembalikan menjadi harga menggunakan:

$$
\hat{P}\_{\mathrm{next}}
=======================
P\_{\mathrm{last}}
\times
\exp\left(\hat{y}\right)
$$

- Jika model memprediksi perubahan yang mendekati nol, maka:

$$
\hat{y}\approx 0
$$

sehingga:

$$
\hat{P}\_{\mathrm{next}}
=======================
P\_{\mathrm{last}}
\times
\exp(0)
$$

Karena:

$$
\exp(0)=1
$$

maka:

$$
\hat{P}\_{\mathrm{next}}
=======================
P\_{\mathrm{last}}
$$

Dengan demikian, jika model memprediksi perubahan yang mendekati nol, hasil prediksi akan berada dekat dengan harga terakhir. Pendekatan tersebut secara konsep menyerupai naive baseline yang diberikan koreksi oleh LSTM.

- Window yang lebih panjang juga akan dipertimbangkan untuk AAPL karena tren dan perubahan harga dapat berlangsung lebih lama daripada lima hari. Namun, panjang window terbaik belum ditentukan dan harus dibuktikan menggunakan validation set.
- Rolling mean dan rolling standard deviation tidak langsung digunakan sebagai fitur model. Keduanya hanya digunakan pada EDA untuk memahami tren dan volatilitas karena soal membatasi input model pada urutan `Close`.
- Window 252 hari tidak langsung digunakan sebagai input LSTM. Meskipun periode tersebut digunakan untuk rolling statistics, window yang terlalu panjang dapat mengurangi jumlah sequence dan meningkatkan kompleksitas model.
- Seluruh kandidat model nantinya akan dibandingkan setelah prediksi dikembalikan ke skala harga asli.
- RMSE akan digunakan untuk memberikan penalti lebih besar terhadap error yang ekstrem.
- MAE akan digunakan untuk menunjukkan rata-rata besar kesalahan dalam satuan harga.
- MAPE akan digunakan untuk menunjukkan besar kesalahan secara relatif terhadap harga aktual.
- RMSE dan MAE AMD tidak akan dibandingkan langsung dengan AAPL karena skala harga keduanya berbeda. MAPE dapat digunakan sebagai perbandingan relatif, tetapi tetap harus dibaca bersama RMSE dan MAE.

Dengan demikian, rancangan modeling didasarkan pada karakteristik yang ditemukan melalui EDA. Naive baseline dipilih karena hubungan antara harga hari ini dan harga berikutnya sangat kuat. Baseline LSTM menggunakan window lima hari sesuai ketentuan soal. Pada AMD, eksperimen selanjutnya akan lebih berfokus pada window yang lebih panjang dan ketahanan terhadap volatilitas. Pada AAPL, eksperimen akan membandingkan harga absolut dengan representasi log atau relative log-price untuk menghadapi tren, right-skewness, dan perubahan skala harga. Konfigurasi model secara rinci akan dijelaskan setelah hasil baseline diperoleh.

### Beberapa sumber saya mendapatkan ide:

Naive baseline digunakan karena metode naive menetapkan prediksi berikutnya sama dengan observasi terakhir dan dapat menjadi pembanding yang kuat pada data ekonomi maupun finansial. Sementara itu, log-return digunakan untuk merepresentasikan perubahan harga secara relatif sehingga analisis tidak hanya bergantung pada level harga absolut. Dalam penelitian ini, kedua konsep tersebut diadaptasi menjadi pendekatan relative log-price, yaitu harga terakhir digunakan sebagai anchor dan LSTM mempelajari koreksi perubahan harga berikutnya **(Hyndman & Athanasopoulos, 2021; Tsay, 2010)**.

Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and practice* (3rd ed.). OTexts.

Tsay, R. S. (2010). *Analysis of financial time series* (3rd ed.). John Wiley & Sons. <https://doi.org/10.1002/9780470644560>

## B. Split Data

### Chronological Split

Chronological split merupakan metode pembagian data berdasarkan urutan waktu tanpa melakukan pengacakan atau `shuffle`. Data dengan tanggal paling lama digunakan sebagai data train, data setelahnya digunakan sebagai validation, dan periode paling baru digunakan sebagai test. Pendekatan ini sesuai untuk forecasting karena model harus mempelajari informasi masa lalu untuk memprediksi nilai pada masa depan.

Penggunaan chronological split didasarkan pada karakteristik data AMD dan AAPL yang telah ditemukan melalui EDA. Kedua dataset merupakan time series dengan kolom `Date` sebagai urutan waktu dan `Close` sebagai target numerik kontinu. Data sudah tersusun secara ascending, tidak memiliki duplikasi tanggal, dan satu observasi merepresentasikan satu hari perdagangan. Oleh karena itu, urutan observasi merupakan bagian penting dari struktur data dan tidak boleh diacak.

EDA juga menunjukkan bahwa harga `Close` AMD dan AAPL bersifat non-stationary. Grafik historis, rolling mean, dan rolling standard deviation memperlihatkan bahwa rata-rata, penyebaran, serta skala harga berubah sepanjang waktu. AMD memiliki pola yang lebih siklis dengan beberapa kenaikan dan penurunan besar, sedangkan AAPL memiliki tren kenaikan jangka panjang dan perubahan skala harga yang sangat kuat.

Nilai autokorelasi lag-1 `Close` juga sangat tinggi, yaitu sekitar 0,9979 untuk AMD dan 0,9997 untuk AAPL. Hasil tersebut menunjukkan adanya ketergantungan temporal yang kuat antara harga saat ini dan harga pada periode sebelumnya. Walaupun autokorelasi return harian hampir nol, urutan harga `Close` tetap harus dipertahankan agar LSTM dapat mempelajari pola sequence secara benar.

Random split tidak digunakan karena dapat menempatkan observasi masa depan ke dalam data train dan observasi masa lalu ke dalam data test. Kondisi tersebut dapat menyebabkan data leakage dan menghasilkan evaluasi yang terlalu optimistis. Dengan chronological split, evaluasi lebih menyerupai kondisi nyata, yaitu model dilatih menggunakan data historis dan diuji pada periode yang benar-benar terjadi setelah periode training.

Sesuai ketentuan soal, satu tahun terakhir digunakan sebagai test set. Karena tanggal terakhir dataset adalah 1 April 2020, test set dimulai pada 1 April 2019. Data sebelum test kemudian dibagi secara kronologis menjadi sekitar 90% train dan 10% validation. Data train digunakan untuk melatih model, validation digunakan untuk memilih arsitektur dan hyperparameter terbaik, sedangkan test hanya digunakan untuk evaluasi akhir.

Pada AMD, hasil pembagian menghasilkan 8.859 observasi train, 985 observasi validation, dan 254 observasi test(Ini saya lakukan pada next step, yaituu di bagian Distribution Shift) Pada AAPL, pembagian menghasilkan 8.689 observasi train, 966 observasi validation, dan 254 observasi test. Perbedaan jumlah train dan validation terjadi karena jumlah keseluruhan observasi AMD dan AAPL tidak sama.

Periode test mencakup kenaikan harga selama 2019, puncak harga pada awal 2020, serta penurunan tajam pada awal pandemi COVID-19. Hal ini membuat test set lebih menantang, tetapi tetap realistis karena menggambarkan kondisi masa depan yang berbeda dari data train. Perbedaan distribusi antara train, validation, dan test bukan merupakan kesalahan, melainkan bagian dari karakteristik time series yang mengalami distribution shift dan perubahan regime pasar.

Setelah proses split, `MinMaxScaler` hanya akan di-fit menggunakan data train. Data validation dan test kemudian ditransformasi menggunakan scaler yang sama agar tidak terjadi kebocoran informasi dari masa depan. Selanjutnya, sequence validation dan test tetap membawa beberapa observasi terakhir dari periode sebelumnya sebagai konteks temporal, tanpa menggunakan informasi setelah target yang diprediksi.

In [ ]:
amd_test_start = amd["Date"].max() - pd.DateOffset(years=1)

amd_before_test = amd[amd["Date"] < amd_test_start].copy()
amd_test = amd[amd["Date"] >= amd_test_start].copy()


In [ ]:
amd_cut = int(len(amd_before_test) * 0.90)

amd_train = amd_before_test.iloc[:amd_cut].copy()
amd_val = amd_before_test.iloc[amd_cut:].copy()


In [ ]:
print("AMD Train      :", len(amd_train), amd_train["Date"].min(), "-", amd_train["Date"].max())
print("AMD Validation :", len(amd_val), amd_val["Date"].min(), "-", amd_val["Date"].max())
print("AMD Test       :", len(amd_test), amd_test["Date"].min(), "-", amd_test["Date"].max())


Ini adalah waktu pembagian data AMD

In [ ]:
aapl_test_start = aapl["Date"].max() - pd.DateOffset(years=1)

aapl_before_test = aapl[aapl["Date"] < aapl_test_start].copy()
aapl_test = aapl[aapl["Date"] >= aapl_test_start].copy()


In [ ]:
aapl_cut = int(len(aapl_before_test) * 0.90)

aapl_train = aapl_before_test.iloc[:aapl_cut].copy()
aapl_val = aapl_before_test.iloc[aapl_cut:].copy()


In [ ]:
print("AAPL Train      :", len(aapl_train), aapl_train["Date"].min(), "-", aapl_train["Date"].max())
print("AAPL Validation :", len(aapl_val), aapl_val["Date"].min(), "-", aapl_val["Date"].max())
print("AAPL Test       :", len(aapl_test), aapl_test["Date"].min(), "-", aapl_test["Date"].max())


Ini adalah waktu pembagian data AAPL

In [ ]:
amd_split_summary = pd.DataFrame([
    {
        "Split": "Train",
        "Observations": len(amd_train),
        "Start Date": amd_train["Date"].min(),
        "End Date": amd_train["Date"].max(),
        "Minimum Close": amd_train["Close"].min(),
        "Median Close": amd_train["Close"].median(),
        "Mean Close": amd_train["Close"].mean(),
        "Maximum Close": amd_train["Close"].max()
    },
    {
        "Split": "Validation",
        "Observations": len(amd_val),
        "Start Date": amd_val["Date"].min(),
        "End Date": amd_val["Date"].max(),
        "Minimum Close": amd_val["Close"].min(),
        "Median Close": amd_val["Close"].median(),
        "Mean Close": amd_val["Close"].mean(),
        "Maximum Close": amd_val["Close"].max()
    },
    {
        "Split": "Test",
        "Observations": len(amd_test),
        "Start Date": amd_test["Date"].min(),
        "End Date": amd_test["Date"].max(),
        "Minimum Close": amd_test["Close"].min(),
        "Median Close": amd_test["Close"].median(),
        "Mean Close": amd_test["Close"].mean(),
        "Maximum Close": amd_test["Close"].max()
    }
])

amd_split_summary


Hasil chronological split AMD menghasilkan 8.859 data train, 985 data validation, dan 254 data test.

Data train mencakup periode 17 Maret 1980–30 April 2015 dengan harga `Close` 1,80–47,50. Data validation mencakup 1 Mei 2015–29 Maret 2019 dengan rentang 1,62–32,72. Data test mencakup 1 April 2019–1 April 2020 dengan rentang 26,24–58,90.

Nilai rata-rata dan median pada test jauh lebih tinggi daripada train dan validation, sehingga terdapat distribution shift pada periode terbaru. Kondisi ini bukan kesalahan split, melainkan mencerminkan perubahan skala harga AMD. Karena maksimum test melebihi maksimum train, hasil scaling pada test dapat lebih dari 1 dan tetap valid selama scaler hanya di-fit pada data train.

In [ ]:
aapl_split_summary = pd.DataFrame([
    {
        "Split": "Train",
        "Observations": len(aapl_train),
        "Start Date": aapl_train["Date"].min(),
        "End Date": aapl_train["Date"].max(),
        "Minimum Close": aapl_train["Close"].min(),
        "Median Close": aapl_train["Close"].median(),
        "Mean Close": aapl_train["Close"].mean(),
        "Maximum Close": aapl_train["Close"].max()
    },
    {
        "Split": "Validation",
        "Observations": len(aapl_val),
        "Start Date": aapl_val["Date"].min(),
        "End Date": aapl_val["Date"].max(),
        "Minimum Close": aapl_val["Close"].min(),
        "Median Close": aapl_val["Close"].median(),
        "Mean Close": aapl_val["Close"].mean(),
        "Maximum Close": aapl_val["Close"].max()
    },
    {
        "Split": "Test",
        "Observations": len(aapl_test),
        "Start Date": aapl_test["Date"].min(),
        "End Date": aapl_test["Date"].max(),
        "Minimum Close": aapl_test["Close"].min(),
        "Median Close": aapl_test["Close"].median(),
        "Mean Close": aapl_test["Close"].mean(),
        "Maximum Close": aapl_test["Close"].max()
    }
])

aapl_split_summary


Hasil chronological split AAPL menghasilkan 8.689 data train, 966 data validation, dan 254 data test.

Data train mencakup periode 12 Desember 1980–28 Mei 2015 dengan harga `Close` 0,196–133,00. Data validation mencakup 29 Mei 2015–29 Maret 2019 dengan rentang 90,34–232,07. Data test mencakup 1 April 2019–1 April 2020 dengan rentang 173,30–327,20.

Rata-rata dan median meningkat sangat besar pada setiap split, sehingga AAPL mengalami distribution shift yang kuat. Seluruh nilai test juga berada di atas maksimum awal data train. Kondisi ini bukan kesalahan split, melainkan mencerminkan tren kenaikan jangka panjang AAPL. Oleh karena itu, hasil scaling validation dan test dapat melebihi 1 dan tetap valid selama scaler hanya di-fit pada data train.

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(amd_train["Date"], amd_train["Close"], label="Train")
plt.plot(amd_val["Date"], amd_val["Close"], label="Validation")
plt.plot(amd_test["Date"], amd_test["Close"], label="Test")
plt.title("Chronological Data Split - AMD")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(aapl_train["Date"], aapl_train["Close"], label="Train")
plt.plot(aapl_val["Date"], aapl_val["Close"], label="Validation")
plt.plot(aapl_test["Date"], aapl_test["Close"], label="Test")
plt.title("Chronological Data Split - AAPL")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()


Visualisasi chronological split menunjukkan bahwa data AMD dan AAPL telah dibagi berdasarkan urutan waktu tanpa melakukan `shuffle`. Data dengan periode paling lama digunakan sebagai train, periode setelahnya digunakan sebagai validation, dan periode paling baru digunakan sebagai test.

Pada AMD, pola harga terlihat lebih fluktuatif dan siklis. Sementara itu, AAPL menunjukkan tren kenaikan jangka panjang yang lebih kuat, terutama pada periode validation dan test.

Perbedaan pola dan skala harga antar-split menunjukkan adanya distribution shift dan perubahan regime pasar. Kondisi ini bukan kesalahan pembagian data, melainkan mencerminkan perubahan historis yang terjadi sepanjang waktu.

Chronological split tetap digunakan agar model mempelajari informasi masa lalu dan dievaluasi pada periode masa depan secara realistis. Karena validation dan test dapat memiliki nilai di luar rentang train, hasil transformasi `MinMaxScaler` dapat berada di luar rentang normal dan tetap valid selama scaler hanya di-fit menggunakan data train.

### Distribution Shift

In [ ]:
amd_train_min = amd_train["Close"].min()
amd_train_max = amd_train["Close"].max()

amd_shift = pd.Series({
    "Train Minimum": amd_train_min,
    "Train Maximum": amd_train_max,
    "Validation Minimum": amd_val["Close"].min(),
    "Validation Maximum": amd_val["Close"].max(),
    "Test Minimum": amd_test["Close"].min(), 
    "Test Maximum": amd_test["Close"].max(),
    "Validation Above Train Max": (amd_val["Close"] > amd_train_max).sum(),
    "Test Above Train Max": (amd_test["Close"] > amd_train_max).sum(),
    "Validation Below Train Min": (amd_val["Close"] < amd_train_min).sum(),
    "Test Below Train Min": (amd_test["Close"] < amd_train_min).sum()
}, name="AMD")

amd_shift.to_frame()


In [ ]:
aapl_train_min = aapl_train["Close"].min()
aapl_train_max = aapl_train["Close"].max()

aapl_shift = pd.Series({
    "Train Minimum": aapl_train_min,
    "Train Maximum": aapl_train_max,
    "Validation Minimum": aapl_val["Close"].min(),
    "Validation Maximum": aapl_val["Close"].max(),
    "Test Minimum": aapl_test["Close"].min(),
    "Test Maximum": aapl_test["Close"].max(),
    "Validation Above Train Max": (aapl_val["Close"] > aapl_train_max).sum(),
    "Test Above Train Max": (aapl_test["Close"] > aapl_train_max).sum(),
    "Validation Below Train Min": (aapl_val["Close"] < aapl_train_min).sum(),
    "Test Below Train Min": (aapl_test["Close"] < aapl_train_min).sum()
}, name="AAPL")

aapl_shift.to_frame()


In [ ]:
distribution_shift = pd.DataFrame([
    amd_shift,
    aapl_shift
])

distribution_shift


Hasil ini menunjukkan adanya distribution shift antara data train, validation, dan test.

Pada AMD, tidak ada nilai validation yang melebihi maksimum train, tetapi terdapat beberapa nilai validation yang lebih rendah daripada minimum train. Pada test, terdapat sejumlah observasi yang melebihi maksimum train, sedangkan tidak ada nilai yang berada di bawah minimum train.

Pada AAPL, banyak nilai validation berada di atas maksimum train. Seluruh observasi test juga berada di atas maksimum train, yang menunjukkan perubahan skala harga yang sangat kuat pada periode terbaru. Tidak terdapat nilai validation maupun test yang berada di bawah minimum train.

Kolom `Validation Above Train Max` dan `Test Above Train Max` menunjukkan jumlah observasi yang melampaui nilai maksimum train, bukan persentase. Kondisi ini menegaskan bahwa distribusi masa depan berbeda dari distribusi data training.

Akibatnya, setelah menggunakan `MinMaxScaler` yang di-fit hanya pada train, beberapa nilai validation atau test dapat lebih besar dari 1. Pada AMD, beberapa nilai validation juga dapat lebih kecil dari 0 karena berada di bawah minimum train. Hasil tersebut tetap valid dan tidak boleh diperbaiki dengan melakukan fit scaler pada seluruh data karena dapat menyebabkan data leakage.

#### Penjelasan lebih detail sudah saya masukkan ke summary EDA

### Test Period

In [ ]:
amd_test_drawdown = amd_test["Close"] / amd_test["Close"].cummax() - 1
amd_test_trough = amd_test_drawdown.idxmin()
amd_test_peak = amd_test.loc[:amd_test_trough, "Close"].idxmax()


In [ ]:
amd_test_summary = pd.Series({
    "Ticker": "AMD",
    "Observations": len(amd_test),
    "Start Date": amd_test["Date"].min(),
    "End Date": amd_test["Date"].max(),
    "First Close": amd_test["Close"].iloc[0],
    "Last Close": amd_test["Close"].iloc[-1],
    "Total Change (%)": (
        amd_test["Close"].iloc[-1] / amd_test["Close"].iloc[0] - 1
    ) * 100,
    "Minimum Close": amd_test["Close"].min(),
    "Median Close": amd_test["Close"].median(),
    "Mean Close": amd_test["Close"].mean(),
    "Maximum Close": amd_test["Close"].max(),
    "Maximum Drawdown (%)": amd_test_drawdown.min() * 100,
    "Peak Date": amd_test.loc[amd_test_peak, "Date"],
    "Trough Date": amd_test.loc[amd_test_trough, "Date"]
}, name="AMD")

amd_test_summary.to_frame("Value")


Test set AMD terdiri dari 254 hari perdagangan pada periode 1 April 2019 sampai 1 April 2020. Harga `Close` meningkat dari 26,36 pada awal periode menjadi 43,66 pada akhir periode, sehingga menghasilkan kenaikan total sekitar 65,63%.

Harga selama test memiliki nilai minimum 26,24, median 32,61, rata-rata 36,18, dan maksimum 58,90. Rata-rata yang lebih tinggi daripada median menunjukkan bahwa beberapa harga tinggi pada akhir 2019 hingga awal 2020 meningkatkan nilai rata-rata periode tersebut.

Meskipun secara keseluruhan harga mengalami kenaikan, test set juga mengandung penurunan yang besar. Setelah mencapai puncak pada 19 Februari 2020, harga mengalami maximum drawdown sekitar -34,28% hingga 16 Maret 2020. Hal ini menunjukkan bahwa test set mencakup fase pertumbuhan sekaligus shock pasar pada awal 2020.

Nilai minimum periode tidak harus sama dengan titik trough maximum drawdown. Nilai minimum merupakan harga terendah di seluruh test set, sedangkan trough drawdown merupakan titik terendah setelah peak tertentu. Karakteristik tersebut membuat test set AMD cukup menantang tetapi realistis untuk mengevaluasi kemampuan model dalam menghadapi kenaikan harga, perubahan regime, dan penurunan tajam.

In [ ]:
aapl_test_drawdown = aapl_test["Close"] / aapl_test["Close"].cummax() - 1
aapl_test_trough = aapl_test_drawdown.idxmin()
aapl_test_peak = aapl_test.loc[:aapl_test_trough, "Close"].idxmax()


In [ ]:
aapl_test_summary = pd.Series({
    "Ticker": "AAPL",
    "Observations": len(aapl_test),
    "Start Date": aapl_test["Date"].min(),
    "End Date": aapl_test["Date"].max(),
    "First Close": aapl_test["Close"].iloc[0],
    "Last Close": aapl_test["Close"].iloc[-1],
    "Total Change (%)": (
        aapl_test["Close"].iloc[-1] / aapl_test["Close"].iloc[0] - 1
    ) * 100,
    "Minimum Close": aapl_test["Close"].min(),
    "Median Close": aapl_test["Close"].median(),
    "Mean Close": aapl_test["Close"].mean(),
    "Maximum Close": aapl_test["Close"].max(),
    "Maximum Drawdown (%)": aapl_test_drawdown.min() * 100,
    "Peak Date": aapl_test.loc[aapl_test_peak, "Date"],
    "Trough Date": aapl_test.loc[aapl_test_trough, "Date"]
}, name="AAPL")

aapl_test_summary.to_frame("Value")


Test set AAPL terdiri dari 254 hari perdagangan pada periode 1 April 2019 sampai 1 April 2020. Harga `Close` meningkat dari 191,24 pada awal periode menjadi 240,91 pada akhir periode, sehingga menghasilkan kenaikan total sekitar 25,97%.

Harga selama test memiliki nilai minimum 173,30, median 223,34, rata-rata 238,62, dan maksimum 327,20. Rata-rata yang lebih tinggi daripada median menunjukkan bahwa harga-harga tinggi pada akhir 2019 dan awal 2020 meningkatkan nilai rata-rata periode tersebut.

Meskipun secara keseluruhan mengalami kenaikan, test set juga mencakup penurunan besar. Setelah mencapai puncak pada 12 Februari 2020, harga mengalami maximum drawdown sekitar -31,43% hingga mencapai titik trough pada 23 Maret 2020.

Hasil ini menunjukkan bahwa test set AAPL mencakup fase kenaikan, pencapaian harga tertinggi, dan shock pasar pada awal 2020. Oleh karena itu, periode tersebut menjadi test set yang cukup menantang dan realistis untuk mengevaluasi kemampuan model menghadapi tren, distribution shift, dan penurunan harga yang tajam.

In [ ]:
test_summary_table = pd.DataFrame([
    amd_test_summary,
    aapl_test_summary
])

test_summary_table


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(amd_test["Date"], amd_test["Close"])
plt.title("Closing Price During Test Period - AMD")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(aapl_test["Date"], aapl_test["Close"])
plt.title("Closing Price During Test Period - AAPL")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.tight_layout()
plt.show()


## Summary Hasil Split dan Tes Period

- Data train digunakan untuk melakukan `fit` pada scaler dan melatih model, sedangkan validation digunakan untuk memilih konfigurasi model terbaik. Test hanya digunakan untuk evaluasi akhir.
- Distribution shift menunjukkan bahwa model harus memprediksi harga pada kondisi yang berbeda dari data training. Tantangan ini lebih kuat pada AAPL karena seluruh harga test berada di atas maksimum train.
- `MinMaxScaler` hanya boleh di-fit pada train. Nilai validation atau test yang berada di luar rentang 0–1 setelah scaling tetap valid dan menunjukkan bahwa harga masa depan berada di luar rentang data training.
- Saat membentuk sequence, validation harus menggunakan beberapa nilai terakhir train sebagai konteks. Test juga harus menggunakan beberapa nilai terakhir sebelum periode test. Untuk baseline dengan window 5, setiap prediksi menggunakan lima harga sebelumnya.
- Test set mencakup tren kenaikan sekaligus penurunan tajam pada awal 2020. Oleh karena itu, model tidak hanya diuji dalam mengikuti tren, tetapi juga dalam menghadapi perubahan arah dan shock harga.
- Kesalahan model kemungkinan meningkat pada periode penurunan tajam. Hal tersebut tidak langsung menunjukkan bahwa proses modeling salah, tetapi dapat menunjukkan keterbatasan model dalam menghadapi perubahan regime yang belum dipelajari dari train.
- Pemilihan baseline dan modified model harus berdasarkan performa validation. Test tidak boleh digunakan untuk menentukan window, transformasi log, jumlah unit LSTM, maupun hyperparameter lainnya.
- Prediksi test harus dikembalikan ke skala harga asli sebelum menghitung RMSE, MAE, dan MAPE agar hasil evaluasi dapat diinterpretasikan dengan benar.

## C. Data Preparation

In [ ]:
# CONFIGURATION
PRICE_COL = "Close"
WINDOW_SIZE = 5
HORIZON = 1


sesuai perintah soal

### 1. Window interpretation audit

In [ ]:
def audit_window(data, window=5, horizon=1):
    data = data.sort_values("Date").copy()
    date_set = set(data["Date"])

    weekly_dates = (
        data.groupby(data["Date"].dt.to_period("W-FRI"))["Date"]
        .apply(list)
    )

    weekly_samples = sum(
        {date.weekday() for date in dates} == {0, 1, 2, 3, 4}
        and max(dates) + pd.Timedelta(days=3) in date_set
        for dates in weekly_dates
    )

    return {
        "Observations": len(data),
        "Rolling 5D Samples": len(data) - window - horizon + 1,
        "Mon-Fri to Next Monday Samples": weekly_samples
    }


window_audit = pd.DataFrame(
    {
        "AMD": audit_window(amd, WINDOW_SIZE, HORIZON),
        "AAPL": audit_window(aapl, WINDOW_SIZE, HORIZON)
    }
).T

window_audit


### 2. Memastikan urutan setiap split

In [ ]:
amd_train = amd_train.sort_values("Date").reset_index(drop=True)
amd_val = amd_val.sort_values("Date").reset_index(drop=True)
amd_test = amd_test.sort_values("Date").reset_index(drop=True)

aapl_train = aapl_train.sort_values("Date").reset_index(drop=True)
aapl_val = aapl_val.sort_values("Date").reset_index(drop=True)
aapl_test = aapl_test.sort_values("Date").reset_index(drop=True)


### 3. Baseline Scalling

In [ ]:
# BASELINE SCALING
from sklearn.preprocessing import MinMaxScaler

amd_scaler = MinMaxScaler()
aapl_scaler = MinMaxScaler()

amd_train_scaled = amd_scaler.fit_transform(amd_train[["Close"]]).astype("float32")
amd_val_scaled = amd_scaler.transform(amd_val[["Close"]]).astype("float32")
amd_test_scaled = amd_scaler.transform(amd_test[["Close"]]).astype("float32")

aapl_train_scaled = aapl_scaler.fit_transform(aapl_train[["Close"]]).astype("float32")
aapl_val_scaled = aapl_scaler.transform(aapl_val[["Close"]]).astype("float32")
aapl_test_scaled = aapl_scaler.transform(aapl_test[["Close"]]).astype("float32")

print("Baseline scaling selesai.")


### 4. Fungsi sequence creation

In [ ]:
def create_sequences(values, window=5, horizon=1):
    X, y = [], []

    for i in range(window, len(values) - horizon + 1):
        X.append(values[i - window:i])
        y.append(values[i + horizon - 1])

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32)
    )


def create_split_sequences(
    train_scaled,
    val_scaled,
    test_scaled,
    window=5,
    horizon=1
):
    X_train, y_train = create_sequences(
        train_scaled, window, horizon
    )

    val_with_context = np.vstack([
        train_scaled[-window:],
        val_scaled
    ])

    test_with_context = np.vstack([
        val_scaled[-window:],
        test_scaled
    ])

    X_val, y_val = create_sequences(
        val_with_context, window, horizon
    )

    X_test, y_test = create_sequences(
        test_with_context, window, horizon
    )

    return X_train, y_train, X_val, y_val, X_test, y_test


Pada ketentuan soal, model diminta menggunakan `window size = 5` dan `horizon = 1`. Namun, terdapat ambiguitas karena soal juga menggunakan frasa “Senin sampai Jumat” dan “memprediksi hari Senin”. Kalimat tersebut dapat menghasilkan dua interpretasi yang berbeda dalam pembentukan sequence.

Interpretasi pertama adalah **rolling lima hari perdagangan untuk memprediksi satu hari perdagangan berikutnya**. Pada pendekatan ini, setiap lima observasi `Close` yang berurutan digunakan sebagai input, kemudian observasi setelahnya menjadi target.

```
Hari perdagangan 1–5 → prediksi hari perdagangan ke-6
Hari perdagangan 2–6 → prediksi hari perdagangan ke-7
Hari perdagangan 3–7 → prediksi hari perdagangan ke-8
```

Hari-hari tersebut tidak harus selalu dimulai pada Senin dan berakhir pada Jumat. Sebagai contoh, apabila terdapat hari libur pasar pada hari Rabu, sequence tetap menggunakan lima observasi perdagangan yang tersedia secara berurutan.

Interpretasi kedua adalah **blok mingguan Senin–Jumat untuk memprediksi Senin berikutnya**. Pada pendekatan ini, hanya minggu yang memiliki data perdagangan lengkap dari Senin sampai Jumat yang dapat digunakan.

```
Senin–Jumat minggu pertama → prediksi Senin minggu berikutnya
Senin–Jumat minggu kedua → prediksi Senin minggu berikutnya
```

Masalah pada interpretasi kedua adalah tidak semua minggu memiliki lima hari perdagangan lengkap. Hari libur pasar dapat menyebabkan suatu minggu hanya memiliki tiga atau empat observasi. Akibatnya, banyak minggu harus dibuang dan jumlah sequence yang dapat digunakan untuk training menjadi jauh lebih sedikit.

Hasil audit kalender menunjukkan bahwa dataset tidak memiliki observasi pada akhir pekan, tetapi terdapat jeda tanggal karena hari libur atau penutupan pasar. Dengan demikian, lima hari kalender tidak selalu sama dengan lima hari perdagangan. Hal ini membuat pendekatan blok Senin–Jumat kurang fleksibel dan dapat menghilangkan banyak informasi historis.

Pada AMD, hanya sekitar 1.744 dari 2.090 kelompok minggu yang memiliki lima hari perdagangan lengkap. Pada AAPL, hanya sekitar 1.710 dari 2.052 kelompok minggu yang lengkap. Jumlah sampel menjadi lebih sedikit lagi apabila setiap minggu lengkap juga diwajibkan memiliki data Senin berikutnya sebagai target.

Berdasarkan pertimbangan tersebut, penelitian ini menggunakan interpretasi pertama, yaitu:

> Lima nilai `Close` dari lima hari perdagangan terakhir digunakan untuk memprediksi nilai `Close` pada satu hari perdagangan berikutnya.

Dengan keputusan tersebut, parameter baseline ditetapkan sebagai:

```
WINDOW_SIZE = 5
HORIZON = 1
```

Pemilihan rolling window lima hari perdagangan dilakukan karena beberapa alasan. Pertama, pendekatan ini tetap memenuhi ketentuan `window size = 5` dan `horizon = 1`. Kedua, seluruh observasi perdagangan dapat digunakan secara lebih optimal tanpa membuang minggu yang tidak lengkap. Ketiga, pendekatan rolling merupakan bentuk umum transformasi supervised learning untuk LSTM karena window digeser satu observasi pada setiap pembentukan sampel.

Keputusan ini juga sesuai dengan hasil EDA. Nilai autokorelasi lag-1 `Close` sangat tinggi, yaitu sekitar 0,9979 pada AMD dan 0,9997 pada AAPL. Hal tersebut menunjukkan bahwa harga penutupan saat ini memiliki hubungan temporal yang kuat dengan harga-harga sebelumnya. Oleh karena itu, penggunaan lima observasi `Close` berurutan dapat memberikan konteks lokal bagi LSTM untuk memprediksi harga berikutnya.

Contoh pembentukan sequence dari data harga berikut:

```
[10, 11, 12, 11, 13, 14, 15]
```

akan menghasilkan:

```
X₁ = [10, 11, 12, 11, 13] → y₁ = 14
X₂ = [11, 12, 11, 13, 14] → y₂ = 15
```

Setiap `X` memiliki bentuk lima timestep dan satu fitur, yaitu `Close`. Oleh karena itu, bentuk input LSTM adalah:

```
(jumlah sequence, 5, 1)
```

Pada data validation dan test, pembentukan sequence tetap membawa konteks dari periode sebelumnya. Target pertama validation menggunakan lima observasi terakhir dari train, sedangkan target pertama test menggunakan lima observasi terakhir sebelum periode test. Penggunaan konteks tersebut bukan data leakage karena seluruh nilai input terjadi sebelum target yang diprediksi.

Interpretasi rolling lima hari perdagangan ini digunakan secara konsisten pada baseline AMD dan AAPL. Untuk modified model, window yang lebih panjang seperti 20 atau 60 hari perdagangan dapat diuji, tetapi model baseline tetap menggunakan window 5 dan horizon 1 sesuai ketentuan soal.

Dengan demikian, frasa “Senin sampai Jumat” dipahami sebagai ilustrasi lima hari perdagangan, bukan sebagai kewajiban bahwa setiap sequence harus selalu dimulai pada Senin dan berakhir pada Jumat. Keputusan ini dijelaskan secara eksplisit agar pembentukan sequence tidak dilakukan berdasarkan asumsi yang tersembunyi.

#### 5. Sequence creation AMD

In [ ]:
(
    X_train_amd,
    y_train_amd,
    X_val_amd,
    y_val_amd,
    X_test_amd,
    y_test_amd
) = create_split_sequences(
    amd_train_scaled,
    amd_val_scaled,
    amd_test_scaled,
    WINDOW_SIZE,
    HORIZON
)

amd_train_dates = amd_train["Date"].iloc[
    WINDOW_SIZE + HORIZON - 1:
].reset_index(drop=True)

amd_val_dates = amd_val["Date"].iloc[
    HORIZON - 1:
].reset_index(drop=True)

amd_test_dates = amd_test["Date"].iloc[
    HORIZON - 1:
].reset_index(drop=True)


#### 6. Sequence creation AAPL

In [ ]:
(
    X_train_aapl,
    y_train_aapl,
    X_val_aapl,
    y_val_aapl,
    X_test_aapl,
    y_test_aapl
) = create_split_sequences(
    aapl_train_scaled,
    aapl_val_scaled,
    aapl_test_scaled,
    WINDOW_SIZE,
    HORIZON
)

aapl_train_dates = aapl_train["Date"].iloc[
    WINDOW_SIZE + HORIZON - 1:
].reset_index(drop=True)

aapl_val_dates = aapl_val["Date"].iloc[
    HORIZON - 1:
].reset_index(drop=True)

aapl_test_dates = aapl_test["Date"].iloc[
    HORIZON - 1:
].reset_index(drop=True)


### 7. Validasi hasil sequence

In [ ]:
assert X_train_amd.shape[1:] == (WINDOW_SIZE, 1)
assert X_train_aapl.shape[1:] == (WINDOW_SIZE, 1)

assert len(X_val_amd) == len(amd_val_dates)
assert len(X_test_amd) == len(amd_test_dates)

assert len(X_val_aapl) == len(aapl_val_dates)
assert len(X_test_aapl) == len(aapl_test_dates)

assert len(X_test_amd) == len(amd_test)
assert len(X_test_aapl) == len(aapl_test)


## D. Modelling Baseline

### Konfigurasi dan reproducibility

In [ ]:
# CONFIGURATION
SEED = 42
WINDOW_SIZE = 5
HORIZON = 1
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# REPRODUCIBILITY
tf.keras.utils.set_random_seed(SEED)

# BASELINE REQUIREMENT CHECK
assert WINDOW_SIZE == 5
assert HORIZON == 1


### Validasi input baseline

In [ ]:
# AMD INPUT CHECK
assert X_train_amd.shape[1:] == (WINDOW_SIZE, 1)
assert X_val_amd.shape[1:] == (WINDOW_SIZE, 1)

assert len(X_train_amd) == len(y_train_amd)
assert len(X_val_amd) == len(y_val_amd)

# AAPL INPUT CHECK
assert X_train_aapl.shape[1:] == (WINDOW_SIZE, 1)
assert X_val_aapl.shape[1:] == (WINDOW_SIZE, 1)

assert len(X_train_aapl) == len(y_train_aapl)
assert len(X_val_aapl) == len(y_val_aapl)


### Fungsi inverse scaling dan evaluasi

In [ ]:
def inverse_scale(values, scaler):
    values = np.asarray(values).reshape(-1, 1)
    return scaler.inverse_transform(values).ravel()


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE (%)": mean_absolute_percentage_error(
            y_true, y_pred
        ) * 100
    }


### Naive baseline pada validation (CUMA TAMBAHAN)

Naive baseline memprediksi bahwa harga berikutnya sama dengan harga terakhir pada sequence.

In [ ]:
# AMD NAIVE BASELINE
amd_val_actual = inverse_scale(y_val_amd, amd_scaler)

amd_naive_val_pred = inverse_scale(
    X_val_amd[:, -1, 0],
    amd_scaler
)

amd_naive_metrics = calculate_metrics(
    amd_val_actual,
    amd_naive_val_pred
)

# AAPL NAIVE BASELINE
aapl_val_actual = inverse_scale(y_val_aapl, aapl_scaler)

aapl_naive_val_pred = inverse_scale(
    X_val_aapl[:, -1, 0],
    aapl_scaler
)

aapl_naive_metrics = calculate_metrics(
    aapl_val_actual,
    aapl_naive_val_pred
)


Naive baseline merupakan metode prediksi paling sederhana yang menetapkan bahwa harga pada hari perdagangan berikutnya sama dengan harga terakhir yang tersedia pada sequence.

Dengan window lima hari, prinsipnya adalah:

```
[Close hari 1, hari 2, hari 3, hari 4, hari 5]
→ prediksi hari berikutnya = Close hari 5
```

Pada kode, bagian:

```
X_val_amd[:, -1, 0]
```

mengambil nilai `Close` terakhir dari setiap sequence validation AMD. Proses yang sama diterapkan pada AAPL. Nilai tersebut kemudian dikembalikan ke skala harga asli menggunakan `inverse_scale()` sebelum dihitung RMSE, MAE, dan MAPE.

Saya menggunakan naive baseline karena hasil EDA menunjukkan bahwa autokorelasi lag-1 `Close` sangat tinggi, yaitu sekitar 0,9979 pada AMD dan 0,9997 pada AAPL. Artinya, harga hari berikutnya biasanya sangat dekat dengan harga hari sebelumnya. Oleh karena itu, metode sederhana yang menggunakan harga terakhir dapat menjadi pembanding yang cukup kuat.

Naive baseline tidak menjalani proses training, tidak memiliki bobot, dan tidak mempelajari pola seperti LSTM. Fungsinya adalah sebagai standar minimum untuk menilai apakah penggunaan model deep learning benar-benar memberikan manfaat.

Jika LSTM menghasilkan error yang lebih rendah daripada naive baseline, berarti model berhasil mempelajari pola tambahan dari sequence. Sebaliknya, jika error LSTM lebih tinggi, berarti model yang lebih kompleks tersebut belum mampu mengalahkan strategi sederhana yang hanya mengikuti harga terakhir.

Naive baseline dihitung pada validation set agar dapat dibandingkan secara adil dengan baseline LSTM tanpa menggunakan test set. Dengan demikian, test set tetap tidak terlibat dalam proses pemilihan atau pengembangan model.

### 1. Fungsi baseline LSTM

In [ ]:
def build_baseline_lstm(ticker):
    model = Sequential(
        [
            Input(shape=(WINDOW_SIZE, 1)),
            LSTM(
                units=50,
                activation="relu"
            ),
            Dense(1)
        ],
        name=f"baseline_lstm_{ticker.lower()}"
    )

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="mse",
        metrics=["mae"]
    )

    return model


Saya menggunakan fungsi `build_baseline_lstm()` untuk membangun arsitektur baseline yang sama bagi AMD dan AAPL.

`Input(shape=(5, 1))`, `LSTM(50, activation="relu")`, dan `Dense(1)` digunakan sesuai ketentuan soal. Artinya, model menerima lima nilai `Close` sebelumnya dan menghasilkan satu prediksi harga `Close` berikutnya.

LSTM digunakan karena data berbentuk time series dan hasil EDA menunjukkan adanya ketergantungan temporal yang kuat pada nilai `Close`.

Optimizer Adam dengan learning rate 0,001 digunakan sebagai tambahan implementasi untuk memperbarui bobot secara adaptif. Loss MSE dipilih karena target berupa nilai numerik kontinu dan memberikan penalti lebih besar pada kesalahan ekstrem. MAE digunakan untuk memantau rata-rata kesalahan absolut selama training.

Parameter `ticker` hanya digunakan untuk membedakan nama model AMD dan AAPL serta tidak memengaruhi proses pembelajaran.

### Fungsi callback

In [ ]:
def create_baseline_callbacks(ticker):
    ticker = ticker.lower()

    return [
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            mode="min",
            restore_best_weights=True
        ),
        ModelCheckpoint(
            filepath=MODEL_DIR / f"baseline_{ticker}_best.keras",
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1
        ),
        CSVLogger(
            filename=LOG_DIR / f"baseline_{ticker}_training.csv"
        )
    ]


Saya menggunakan callback `EarlyStopping` karena hasil EDA menunjukkan bahwa data AMD dan AAPL memiliki pola non-stationary, perubahan regime, distribution shift, serta volatilitas yang berubah sepanjang waktu. Kondisi tersebut dapat menyebabkan model mempelajari data train dengan sangat baik, tetapi performanya pada validation justru memburuk setelah beberapa epoch.

Saya mengatur `EarlyStopping` untuk memantau `val_loss` dan menghentikan training apabila tidak terjadi perbaikan selama 15 epoch berturut-turut. Saya juga menggunakan `restore_best_weights=True` agar model akhir kembali menggunakan bobot dari epoch dengan validation loss terbaik, bukan bobot dari epoch terakhir.

Dengan demikian, saya menggunakan callback untuk mengurangi risiko overfitting, mencegah training berlangsung terlalu lama, dan memperoleh model yang memiliki kemampuan generalisasi lebih baik pada data validation.

### 2. Training baseline AMD

In [ ]:
tf.keras.utils.set_random_seed(SEED)

baseline_amd = build_baseline_lstm("AMD")

early_stopping_amd = EarlyStopping(
    monitor="val_loss",
    patience=15,
    mode="min",
    restore_best_weights=True
)

history_amd = baseline_amd.fit(
    X_train_amd,
    y_train_amd,
    validation_data=(X_val_amd, y_val_amd),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=[early_stopping_amd],
    verbose=1
)


Hasil training menunjukkan bahwa baseline LSTM AMD belajar dengan sangat cepat pada beberapa epoch awal. Nilai `loss` turun tajam dari sekitar 0,0046 pada epoch pertama menjadi sekitar 0,0003 pada epoch keempat. Pada saat yang sama, `val_loss` juga turun dari sekitar 0,0043 menjadi sekitar 0,00067. Hal ini menunjukkan bahwa model berhasil menangkap pola dasar hubungan lima harga penutupan sebelumnya dengan harga berikutnya.

Nilai validation loss terbaik dari output yang terlihat terjadi sekitar epoch ke-4. Setelah itu, training loss tetap rendah, tetapi validation loss justru meningkat hingga berada di sekitar 0,0010–0,0012. Kondisi ini menunjukkan adanya indikasi overfitting.

Overfitting terlihat karena model terus menyesuaikan bobot terhadap pola data train, tetapi peningkatan tersebut tidak membuat performa pada validation menjadi lebih baik. Jarak antara train loss dan validation loss juga menjadi lebih besar. Contohnya, train loss berada di sekitar 0,0003, sedangkan validation loss berada di atas 0,001. Artinya, kesalahan pada validation lebih dari tiga kali kesalahan pada train.

Namun, overfitting yang terjadi belum dapat disebut sangat parah karena validation loss tidak terus meningkat tanpa batas. Pada beberapa epoch berikutnya, validation loss kembali sedikit menurun, tetapi tetap belum mengalahkan nilai terbaik pada epoch awal. Pola ini lebih tepat disebut overfitting ringan hingga sedang.

Hasil ini memberikan beberapa insight untuk modeling:

- Model tidak membutuhkan seluruh 100 epoch karena pola utama sudah dipelajari pada beberapa epoch awal.
- Arsitektur LSTM 50 unit sudah memiliki kapasitas yang cukup untuk mempelajari data train.
- Penambahan epoch setelah titik terbaik tidak meningkatkan kemampuan generalisasi.
- Perbedaan pola antara train dan validation kemungkinan membuat model lebih sulit mempertahankan performanya pada data yang lebih baru.
- Window lima hari dapat menangkap pola lokal jangka pendek, tetapi mungkin belum cukup untuk menghadapi perubahan regime yang terjadi pada validation.
- Early stopping sangat penting agar model tidak terus dilatih setelah validation loss berhenti membaik.

Karena `restore_best_weights=True`, model akhir akan menggunakan bobot dari epoch dengan validation loss terbaik, bukan bobot dari epoch terakhir. Dengan demikian, efek overfitting setelah epoch terbaik dapat dikurangi.

Meskipun demikian, hasil loss ini masih berada pada skala hasil normalisasi. Kualitas model belum dapat dinilai hanya dari angka tersebut. Evaluasi selanjutnya harus dilakukan melalui inverse scaling, kemudian menghitung RMSE, MAE, dan MAPE pada skala harga asli serta membandingkannya dengan naive baseline.

### 3. Training baseline AAPL

In [ ]:
# TRAINING BASELINE AAPL
tf.keras.utils.set_random_seed(SEED)

baseline_aapl = build_baseline_lstm("AAPL")

early_stopping_aapl = EarlyStopping(
    monitor="val_loss",
    patience=15,
    mode="min",
    restore_best_weights=True
)

history_aapl = baseline_aapl.fit(
    X_train_aapl,
    y_train_aapl,
    validation_data=(X_val_aapl, y_val_aapl),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=[early_stopping_aapl],
    verbose=1
)


Proses training baseline LSTM AAPL menunjukkan pola yang cukup tidak stabil. Pada beberapa epoch awal, nilai training loss dan validation loss mengalami fluktuasi yang besar. Sebagai contoh, validation loss meningkat dari 0,0017 pada epoch pertama menjadi 0,0103 pada epoch kedua, kemudian kembali menurun pada epoch berikutnya.

Pada sekitar epoch ke-10 sampai ke-12, validation loss mencapai nilai yang relatif rendah, yaitu sekitar 0,0012–0,0013. Namun, setelah itu training loss terus menurun, sedangkan validation loss kembali meningkat. Pada epoch ke-33, training loss sudah sangat kecil, yaitu sekitar 0,000050, tetapi validation loss meningkat menjadi 0,0063.

Pola tersebut menunjukkan adanya indikasi **overfitting**, karena model semakin baik mempelajari data train, tetapi performanya pada validation set memburuk. Fluktuasi validation loss juga menunjukkan bahwa proses optimasi baseline belum berjalan secara stabil.

Kondisi ini dapat dipengaruhi oleh penggunaan aktivasi ReLU pada LSTM, learning rate yang relatif besar, serta tidak adanya mekanisme penurunan learning rate pada baseline. Meskipun demikian, konfigurasi tersebut tetap digunakan karena mengikuti ketentuan baseline pada soal.

Karena output training terpotong, best epoch tidak dapat ditentukan hanya berdasarkan cuplikan tersebut. Best epoch harus diambil dari epoch dengan nilai `val_loss` terendah pada seluruh training history. Apabila digunakan `EarlyStopping` dengan `restore_best_weights=True`, model akan menggunakan kembali bobot dari epoch terbaik.

Nilai loss dan MAE pada proses training masih berada dalam skala hasil normalisasi. Oleh karena itu, performa akhir model tetap dinilai setelah inverse scaling menggunakan RMSE, MAE, dan MAPE pada skala harga asli.

### 4. Prediksi validation dalam skala harga asli

In [ ]:
# AMD VALIDATION PREDICTION
amd_lstm_val_scaled = baseline_amd.predict(
    X_val_amd,
    verbose=0
)

amd_lstm_val_pred = inverse_scale(
    amd_lstm_val_scaled,
    amd_scaler
)

amd_lstm_metrics = calculate_metrics(
    amd_val_actual,
    amd_lstm_val_pred
)

# AAPL VALIDATION PREDICTION
aapl_lstm_val_scaled = baseline_aapl.predict(
    X_val_aapl,
    verbose=0
)

aapl_lstm_val_pred = inverse_scale(
    aapl_lstm_val_scaled,
    aapl_scaler
)

aapl_lstm_metrics = calculate_metrics(
    aapl_val_actual,
    aapl_lstm_val_pred
)


### 5. Perbandingan naive baseline dan LSTM

In [ ]:
baseline_validation_results = pd.DataFrame(
    [
        {
            "Ticker": "AMD",
            "Model": "Naive Baseline",
            **amd_naive_metrics
        },
        {
            "Ticker": "AMD",
            "Model": "Baseline LSTM",
            **amd_lstm_metrics
        },
        {
            "Ticker": "AAPL",
            "Model": "Naive Baseline",
            **aapl_naive_metrics
        },
        {
            "Ticker": "AAPL",
            "Model": "Baseline LSTM",
            **aapl_lstm_metrics
        }
    ]
)

baseline_validation_results


Hasil validation menunjukkan bahwa naive baseline menghasilkan performa yang lebih baik daripada baseline LSTM pada AMD maupun AAPL. Pada seluruh metrik, naive baseline memiliki nilai RMSE, MAE, dan MAPE yang lebih rendah.

Pada AMD, naive baseline menghasilkan RMSE sebesar 0,5430, MAE sebesar 0,3025, dan MAPE sebesar 2,81%. Baseline LSTM menghasilkan RMSE sebesar 1,1852, MAE sebesar 0,7981, dan MAPE sebesar 8,10%. Artinya, baseline LSTM belum mampu mengalahkan strategi sederhana yang menggunakan harga terakhir sebagai prediksi harga berikutnya.

Pada AAPL, naive baseline menghasilkan RMSE sebesar 2,3762, MAE sebesar 1,5850, dan MAPE sebesar 1,09%. Baseline LSTM menghasilkan RMSE sebesar 4,1660, MAE sebesar 3,3431, dan MAPE sebesar 2,43%. Dengan demikian, naive baseline juga lebih baik pada seluruh metrik AAPL.

Hasil ini sesuai dengan temuan EDA bahwa nilai `Close` memiliki autokorelasi lag-1 yang sangat tinggi. Harga pada hari perdagangan berikutnya umumnya berada dekat dengan harga terakhir, sehingga metode naive dapat memberikan prediksi yang kuat meskipun tidak menjalani proses training.

Sebaliknya, baseline LSTM harus mempelajari hubungan dari lima harga sebelumnya. Proses tersebut dapat menghasilkan prediksi yang lebih halus atau mengalami keterlambatan dalam mengikuti perubahan harga. Pada AMD, pola yang siklis dan volatil membuat model lebih sulit merespons perubahan arah yang cepat. Pada AAPL, tren yang kuat serta perbedaan skala harga antara train dan validation menyebabkan model lebih sulit melakukan generalisasi pada periode yang lebih baru.

Hasil training sebelumnya juga menunjukkan adanya gap antara train loss dan validation loss. Kondisi tersebut mengindikasikan bahwa model dapat mempelajari data train dengan baik, tetapi belum mampu mempertahankan performanya pada validation. Indikasi ini lebih kuat pada AAPL karena validation loss lebih berfluktuasi dan jauh lebih tinggi daripada train loss.

Nilai RMSE dan MAE AAPL lebih besar daripada AMD karena skala harga AAPL juga jauh lebih tinggi. Oleh karena itu, RMSE dan MAE tidak digunakan untuk membandingkan performa antarsaham secara langsung. MAPE lebih sesuai untuk melihat kesalahan relatif. Berdasarkan MAPE, naive baseline tetap lebih baik pada kedua saham.

Hasil ini bukan berarti penggunaan LSTM salah. Baseline LSTM tetap diperlukan untuk memenuhi ketentuan soal dan menjadi titik awal eksperimen. Namun, hasil validation membuktikan bahwa arsitektur baseline belum memberikan peningkatan dibandingkan metode naive.

Berdasarkan hasil tersebut, modified model perlu diarahkan untuk memperbaiki kemampuan generalisasi. Pada AMD, eksperimen dapat mencakup window yang lebih panjang, learning rate yang lebih kecil, perubahan jumlah unit, dan regularisasi. Pada AAPL, eksperimen dapat mencakup window yang lebih panjang, transformasi `log1p(Close)`, learning rate yang lebih kecil, dan regularisasi untuk menghadapi tren serta perubahan skala harga.

Nanti hasil seluruh modified model akan saya pilih berdasarkan hasil validation.

### 6. Menghitung peningkatan terhadap naive baseline

In [ ]:
baseline_improvement = pd.DataFrame(
    [
        {
            "Ticker": "AMD",
            "RMSE Improvement (%)": (
                amd_naive_metrics["RMSE"]
                - amd_lstm_metrics["RMSE"]
            ) / amd_naive_metrics["RMSE"] * 100,
            "MAE Improvement (%)": (
                amd_naive_metrics["MAE"]
                - amd_lstm_metrics["MAE"]
            ) / amd_naive_metrics["MAE"] * 100,
            "MAPE Improvement (%)": (
                amd_naive_metrics["MAPE (%)"]
                - amd_lstm_metrics["MAPE (%)"]
            ) / amd_naive_metrics["MAPE (%)"] * 100
        },
        {
            "Ticker": "AAPL",
            "RMSE Improvement (%)": (
                aapl_naive_metrics["RMSE"]
                - aapl_lstm_metrics["RMSE"]
            ) / aapl_naive_metrics["RMSE"] * 100,
            "MAE Improvement (%)": (
                aapl_naive_metrics["MAE"]
                - aapl_lstm_metrics["MAE"]
            ) / aapl_naive_metrics["MAE"] * 100,
            "MAPE Improvement (%)": (
                aapl_naive_metrics["MAPE (%)"]
                - aapl_lstm_metrics["MAPE (%)"]
            ) / aapl_naive_metrics["MAPE (%)"] * 100
        }
    ]
)

baseline_improvement


### Ringkasan proses training

In [ ]:
baseline_training_summary = pd.DataFrame(
    [
        {
            "Ticker": "AMD",
            "Epochs Trained": len(history_amd.history["loss"]),
            "Best Epoch": (
                np.argmin(history_amd.history["val_loss"]) + 1
            ),
            "Best Validation Loss": min(
                history_amd.history["val_loss"]
            )
        },
        {
            "Ticker": "AAPL",
            "Epochs Trained": len(history_aapl.history["loss"]),
            "Best Epoch": (
                np.argmin(history_aapl.history["val_loss"]) + 1
            ),
            "Best Validation Loss": min(
                history_aapl.history["val_loss"]
            )
        }
    ]
)

baseline_training_summary


Nilai improvement dihitung dengan membandingkan error baseline LSTM terhadap naive baseline. Nilai positif menunjukkan bahwa LSTM menghasilkan error yang lebih rendah, sedangkan nilai negatif menunjukkan bahwa LSTM justru memiliki error yang lebih tinggi daripada naive baseline.

Pada AMD, seluruh nilai improvement bernilai negatif. RMSE baseline LSTM sekitar 118,26% lebih tinggi daripada naive baseline, MAE sekitar 163,85% lebih tinggi, dan MAPE sekitar 188,47% lebih tinggi. Hasil ini menunjukkan bahwa baseline LSTM AMD belum mampu mengalahkan prediksi sederhana yang menggunakan harga terakhir sebagai prediksi berikutnya.

Pada AAPL, baseline LSTM juga masih lebih buruk daripada naive baseline. RMSE LSTM sekitar 75,33% lebih tinggi, MAE sekitar 110,93% lebih tinggi, dan MAPE sekitar 123,29% lebih tinggi daripada naive baseline.

Hasil negatif pada seluruh metrik menegaskan bahwa baseline LSTM belum memberikan peningkatan performa pada validation set. Model kemungkinan menghasilkan prediksi yang lebih halus atau terlambat mengikuti perubahan harga, sedangkan naive baseline langsung menggunakan harga terakhir yang sangat kuat karena autokorelasi `Close` sangat tinggi.

Performa LSTM yang lebih buruk bukan berarti proses modeling salah. Baseline tetap diperlukan untuk memenuhi ketentuan soal dan menjadi titik pembanding. Hasil ini justru menjadi dasar bahwa modified model perlu meningkatkan kemampuan generalisasi melalui eksperimen window, learning rate, jumlah unit, regularisasi, dan transformasi log yang dipilih berdasarkan validation set.

### Grafik learning curve AMD

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history_amd.history["loss"],
    label="Train Loss"
)

plt.plot(
    history_amd.history["val_loss"],
    label="Validation Loss"
)

plt.title("Baseline LSTM Learning Curve - AMD")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()


### Grafik learning curve AAPL

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history_aapl.history["loss"],
    label="Train Loss"
)

plt.plot(
    history_aapl.history["val_loss"],
    label="Validation Loss"
)

plt.title("Baseline LSTM Learning Curve - AAPL")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()


### Grafik prediksi validation AMD

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    amd_val_dates,
    amd_val_actual,
    label="Actual"
)

plt.plot(
    amd_val_dates,
    amd_lstm_val_pred,
    label="Baseline LSTM"
)

plt.plot(
    amd_val_dates,
    amd_naive_val_pred,
    label="Naive Baseline",
    alpha=0.7
)

plt.title("Validation Prediction - AMD")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()


### Grafik prediksi validation AAPL

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    aapl_val_dates,
    aapl_val_actual,
    label="Actual"
)

plt.plot(
    aapl_val_dates,
    aapl_lstm_val_pred,
    label="Baseline LSTM"
)

plt.plot(
    aapl_val_dates,
    aapl_naive_val_pred,
    label="Naive Baseline",
    alpha=0.7
)

plt.title("Validation Prediction - AAPL")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()


Baseline modeling dilakukan secara terpisah pada AMD dan AAPL menggunakan konfigurasi yang sama sesuai ketentuan soal. Setiap model menerima lima nilai `Close` dari lima hari perdagangan sebelumnya untuk memprediksi satu nilai `Close` pada hari perdagangan berikutnya.

Arsitektur baseline yang digunakan adalah:

```
Input (5, 1)
    ↓
LSTM 50 units, activation ReLU
    ↓
Dense(1)
    ↓
Prediksi Close hari berikutnya
```

Model menggunakan optimizer Adam dengan learning rate 0,001, loss MSE, batch size 32, dan maksimum 100 epoch. Proses training dilakukan tanpa `shuffle` agar urutan temporal tetap dipertahankan.

Nilai `Close` asli digunakan pada baseline tanpa transformasi log. AMD dan AAPL menggunakan `MinMaxScaler` yang berbeda. Setiap scaler hanya di-fit menggunakan data train setelah chronological split dilakukan, sedangkan data validation ditransformasikan menggunakan scaler yang sudah dipelajari dari train.

Validation digunakan untuk memantau kemampuan generalisasi model dan menentukan bobot terbaik melalui `EarlyStopping`. Parameter `restore_best_weights=True` memastikan model menggunakan bobot dari epoch dengan validation loss terendah. Test set belum digunakan dalam training, pemilihan konfigurasi, maupun pengembangan modified model.

Sebagai pembanding, digunakan naive baseline yang memprediksi harga hari berikutnya sama dengan harga terakhir pada sequence:

[
\hat{P}\_{t+1}=P\_t
]

Naive baseline digunakan karena hasil EDA menunjukkan bahwa nilai `Close` memiliki autokorelasi lag-1 yang sangat tinggi. Autokorelasi lag-1 `Close` AMD adalah sekitar 0,9979, sedangkan AAPL sekitar 0,9997. Artinya, harga hari perdagangan berikutnya umumnya berada dekat dengan harga hari sebelumnya.

Sebaliknya, autokorelasi lag-1 return jauh lebih rendah, yaitu sekitar 0,009 pada AMD dan 0,019 pada AAPL. Hal tersebut menunjukkan bahwa meskipun level harga sangat persisten, arah dan besar perubahan harga berikutnya jauh lebih sulit diprediksi.

Pada AMD, proses training menunjukkan bahwa train loss menurun dengan cepat. Validation loss mencapai nilai rendah pada beberapa epoch awal, kemudian cenderung meningkat ketika train loss tetap menurun. Kondisi ini menunjukkan adanya indikasi overfitting ringan hingga sedang. `EarlyStopping` digunakan untuk menghentikan training ketika validation loss tidak lagi membaik dan mengembalikan bobot dari epoch terbaik.

Pada AAPL, proses training terlihat lebih tidak stabil. Train loss dapat turun sangat rendah, tetapi validation loss mengalami fluktuasi dan beberapa kali meningkat cukup besar. Gap antara train loss dan validation loss lebih jelas dibandingkan AMD. Kondisi ini menunjukkan bahwa baseline AAPL mengalami kesulitan generalisasi yang lebih kuat.

Hasil evaluasi validation baseline adalah sebagai berikut:

- Naive baseline AMD menghasilkan RMSE 0,5430, MAE 0,3025, dan MAPE 2,81%.
- Baseline LSTM AMD menghasilkan RMSE 1,1852, MAE 0,7981, dan MAPE 8,10%.
- Naive baseline AAPL menghasilkan RMSE 2,3762, MAE 1,5850, dan MAPE 1,09%.
- Baseline LSTM AAPL menghasilkan RMSE 4,1660, MAE 3,3431, dan MAPE 2,43%.

Naive baseline memberikan error yang lebih rendah daripada baseline LSTM pada seluruh metrik dan kedua saham.

Pada AMD, RMSE, MAE, dan MAPE baseline LSTM masing-masing sekitar 118,26%, 163,85%, dan 188,47% lebih tinggi daripada naive baseline. Pada AAPL, RMSE, MAE, dan MAPE baseline LSTM masing-masing sekitar 75,33%, 110,93%, dan 123,29% lebih tinggi daripada naive baseline.

Hasil tersebut menunjukkan bahwa baseline LSTM belum mampu memanfaatkan sequence lima hari secara lebih baik daripada hanya menggunakan harga terakhir. Model LSTM kemungkinan menghasilkan prediksi yang lebih halus atau terlambat mengikuti perubahan harga, sedangkan naive baseline sangat kuat karena tingginya persistensi nilai `Close`.

Performa baseline yang lebih buruk tidak menunjukkan bahwa proses modeling salah. Baseline tetap diperlukan untuk memenuhi ketentuan soal, memberikan titik pembanding, dan mengidentifikasi aspek yang perlu diperbaiki melalui modified model.

Modified model akan dikembangkan secara terpisah untuk AMD dan AAPL karena hasil EDA menunjukkan bahwa kedua saham memiliki karakteristik yang berbeda.

**Dasar pengembangan modified model AMD**

AMD memiliki volatilitas harian yang lebih tinggi daripada AAPL, yaitu sekitar 3,78%. AMD juga memiliki pola yang lebih siklis, perubahan arah yang tajam, serta beberapa perubahan regime pasar.

Baseline AMD hanya menggunakan lima hari perdagangan sebagai input. Window tersebut mungkin terlalu pendek untuk menangkap pola AMD yang berlangsung dalam periode lebih panjang. Oleh karena itu, pengembangan modified model AMD difokuskan pada penambahan konteks temporal, pengurangan kompleksitas model, penyesuaian regularisasi, dan peningkatan stabilitas training.

Tiga kandidat modified model AMD yang akan dibandingkan adalah sebagai berikut.

**1. Modified AMD W20**

Model ini menggunakan 20 nilai `Close` sebelumnya untuk memprediksi satu nilai `Close` berikutnya.

Arsitekturnya adalah:

```
Input (20, 1)
     ↓
LSTM 32 units, activation tanh
     ↓
Dropout 0.2
     ↓
Dense(1)
     ↓
Prediksi Close berikutnya
```

Konfigurasi model terdiri atas:

- window sebanyak 20 hari perdagangan;
- satu layer LSTM dengan 32 unit;
- aktivasi `tanh`;
- dropout sebesar 0,2;
- `Dense(1)` sebagai output;
- optimizer Adam;
- learning rate sebesar 0,0005;
- loss MSE;
- batch size 32;
- maksimum 100 epoch;
- training tanpa `shuffle`;
- `EarlyStopping` dengan patience 15;
- `restore_best_weights=True`.

Window 20 digunakan untuk memberikan konteks sekitar satu bulan perdagangan. Pengurangan jumlah unit dari 50 menjadi 32 dilakukan untuk mengurangi kompleksitas model, sedangkan dropout ringan digunakan untuk membantu mengurangi overfitting.

**2. Modified AMD W60**

Model ini menggunakan 60 nilai `Close` sebelumnya untuk memprediksi satu nilai `Close` berikutnya.

Arsitekturnya adalah:

```
Input (60, 1)
     ↓
LSTM 32 units, activation tanh
     ↓
Dropout 0.2
     ↓
Dense(1)
     ↓
Prediksi Close berikutnya
```

Konfigurasi model terdiri atas:

- window sebanyak 60 hari perdagangan;
- satu layer LSTM dengan 32 unit;
- aktivasi `tanh`;
- dropout sebesar 0,2;
- `Dense(1)` sebagai output;
- optimizer Adam;
- learning rate sebesar 0,0005;
- loss MSE;
- batch size 32;
- maksimum 100 epoch;
- training tanpa `shuffle`;
- `EarlyStopping` dengan patience 15;
- `restore_best_weights=True`.

Window 60 memberikan konteks sekitar tiga bulan perdagangan. Kandidat ini digunakan untuk menguji apakah karakter AMD yang lebih siklis dan volatil lebih mudah dipelajari menggunakan konteks temporal yang lebih panjang.

**3. AMD W60 U32 No Dropout**

Kandidat ini tetap menggunakan window 60 dan 32 unit LSTM, tetapi tidak menggunakan dropout.

Arsitekturnya adalah:

```
Input (60, 1)
     ↓
LSTM 32 units, activation tanh
     ↓
Dense(1)
     ↓
Prediksi Close berikutnya
```

Konfigurasi model terdiri atas:

- window sebanyak 60 hari perdagangan;
- satu layer LSTM dengan 32 unit;
- aktivasi `tanh`;
- tanpa dropout;
- `Dense(1)` sebagai output;
- optimizer Adam;
- learning rate sebesar 0,00025;
- Huber loss;
- gradient clipping dengan `clipnorm=1.0`;
- batch size 32;
- maksimum 100 epoch;
- training tanpa `shuffle`;
- `EarlyStopping` dengan patience 15;
- `ReduceLROnPlateau` dengan patience 5;
- `restore_best_weights=True`.

Kandidat ini digunakan untuk menguji kombinasi arsitektur yang lebih sederhana dan training yang lebih stabil. Penghapusan dropout dilakukan agar seluruh informasi temporal dari output LSTM dapat diteruskan menuju layer output.

Namun, kandidat ini tidak hanya berbeda pada penggunaan dropout. Learning rate, loss function, gradient clipping, dan callback juga disesuaikan. Oleh karena itu, apabila model ini memberikan hasil lebih baik, peningkatannya tidak boleh dinyatakan hanya berasal dari penghapusan dropout.

Huber loss digunakan karena lebih tahan terhadap error besar dibandingkan MSE murni. Gradient clipping digunakan untuk membatasi besar gradien selama backpropagation dan mengurangi risiko exploding gradient. `ReduceLROnPlateau` digunakan untuk menurunkan learning rate ketika validation loss tidak lagi membaik.

Ketiga kandidat AMD tetap menggunakan nilai `Close` asli. Data dinormalisasi menggunakan `MinMaxScaler` yang hanya di-fit pada data train. Prediksi kemudian dikembalikan ke skala harga asli sebelum RMSE, MAE, dan MAPE dihitung.

**Dasar pengembangan modified model AAPL**

AAPL memiliki karakteristik yang berbeda dari AMD. Hasil EDA menunjukkan bahwa AAPL memiliki tren kenaikan jangka panjang yang sangat kuat, distribusi `Close` sangat right-skewed, rolling mean dan rolling standard deviation yang berubah sepanjang waktu, serta distribution shift yang cukup besar.

Maksimum harga AAPL pada train hanya sekitar 133, sedangkan validation mencapai sekitar 232. Perubahan rentang tersebut menunjukkan bahwa model perlu melakukan prediksi pada level harga yang lebih tinggi daripada sebagian besar data yang dipelajari selama training.

Selain itu, baseline AAPL menunjukkan training yang lebih tidak stabil dan gap train-validation yang lebih besar daripada AMD. Oleh karena itu, pengembangan modified model AAPL tidak hanya mempertimbangkan panjang window dan kapasitas LSTM, tetapi juga membandingkan dua cara merepresentasikan harga.

Pendekatan pertama tetap menggunakan harga absolut sebagai kontrol eksperimen. Pendekatan kedua menggunakan relative log-price agar model mempelajari perubahan harga relatif, bukan level harga absolut.

Tiga kandidat modified model AAPL yang akan dibandingkan adalah sebagai berikut.

**1. Modified AAPL Raw W20**

Model ini menggunakan nilai `Close` absolut selama 20 hari perdagangan untuk memprediksi harga absolut hari berikutnya.

Alur prediksinya adalah:

```
Harga Close selama 20 hari sebelumnya
                    ↓
Prediksi harga Close hari berikutnya
```

Arsitekturnya adalah:

```
Input (20, 1)
     ↓
LSTM 32 units, activation tanh
     ↓
Dropout 0.2
     ↓
Dense(1)
     ↓
Prediksi Close berikutnya
```

Konfigurasi model terdiri atas:

- window sebanyak 20 hari perdagangan;
- satu layer LSTM dengan 32 unit;
- aktivasi `tanh`;
- dropout sebesar 0,2;
- `Dense(1)` sebagai output;
- optimizer Adam;
- learning rate sebesar 0,0005;
- loss MSE;
- batch size 32;
- maksimum 100 epoch;
- training tanpa `shuffle`;
- `EarlyStopping` dengan patience 15;
- `restore_best_weights=True`.

Kandidat ini digunakan sebagai kontrol untuk mengetahui apakah penambahan konteks dari lima menjadi 20 hari sudah cukup meningkatkan baseline tanpa mengubah representasi harga.

Dengan menyertakan kandidat raw-price, pengaruh perubahan panjang window dapat dibandingkan dengan pengaruh perubahan representasi data.

**2. AAPL Relative Log W20 U32 Huber**

Kandidat ini tidak memprediksi harga absolut secara langsung. Model menggunakan pola perubahan relatif selama 20 hari untuk memprediksi log-return hari berikutnya.

Alur prediksinya adalah:

```
Pola perubahan relatif selama 20 hari
                    ↓
Prediksi log-return hari berikutnya
                    ↓
Harga terakhir × exp(prediksi log-return)
                    ↓
Prediksi harga hari berikutnya
```

Untuk setiap sequence, harga pada setiap posisi diubah relatif terhadap harga terakhir di dalam window. Input dirumuskan sebagai:

[
x\_t=
\log\left(
\frac{P\_t}{P\_{\text{last}}}
\right)
]

dengan:

- (P\_t) merupakan harga pada posisi tertentu di dalam window;
- (P\_{\text{last}}) merupakan harga terakhir pada sequence;
- (x\_t) merupakan posisi harga pada waktu (t) relatif terhadap harga terakhir.

Sebagai contoh, pada window 20, input model berbentuk:

[
\left[
\log\left(\frac{P\_{t-19}}{P\_t}\right),
\log\left(\frac{P\_{t-18}}{P\_t}\right),
\ldots,
\log\left(\frac{P\_t}{P\_t}\right)
\right]
]

Nilai terakhir pada setiap sequence adalah:

# [ \log\left( \frac{P\_{\text{last}}}{P\_{\text{last}}} \right)

# \log(1)

0
]

Nilai negatif menunjukkan bahwa harga historis tersebut lebih rendah daripada harga terakhir, sedangkan nilai positif menunjukkan bahwa harga historis tersebut lebih tinggi daripada harga terakhir.

Target model adalah log-return antara harga berikutnya dan harga terakhir:

[
y=
\log\left(
\frac{P\_{\text{next}}}{P\_{\text{last}}}
\right)
]

Apabila target positif, harga berikutnya lebih tinggi daripada harga terakhir. Apabila target negatif, harga berikutnya lebih rendah daripada harga terakhir. Target yang mendekati nol menunjukkan bahwa harga berikutnya berada dekat dengan harga terakhir.

Setelah model menghasilkan prediksi log-return (\hat{y}), prediksi harga direkonstruksi menggunakan:

# [ \hat{P}\_{\text{next}}

P\_{\text{last}}
\times
\exp(\hat{y})
]

Konfigurasi model terdiri atas:

- window sebanyak 20 hari perdagangan;
- satu layer LSTM dengan 32 unit;
- aktivasi internal `tanh`;
- recurrent activation `sigmoid`;
- tanpa dropout;
- `Dense(1)` sebagai output log-return;
- optimizer Adam;
- learning rate sebesar 0,0003;
- Huber loss;
- gradient clipping dengan `clipnorm=1.0`;
- batch size 32;
- maksimum 150 epoch;
- training tanpa `shuffle`;
- `EarlyStopping` dengan patience 20;
- `ReduceLROnPlateau` dengan patience 6;
- `restore_best_weights=True`.

Window 20 memberikan konteks sekitar satu bulan perdagangan. Kandidat ini digunakan untuk menguji apakah representasi perubahan relatif lebih sesuai untuk AAPL dibandingkan harga absolut.

**3. AAPL Relative Log W40 U32 Huber**

Kandidat ini menggunakan representasi relative log-price dan arsitektur yang sama, tetapi menggunakan window 40 hari.

Alur prediksinya adalah:

```
Pola perubahan relatif selama 40 hari
                    ↓
Prediksi log-return hari berikutnya
                    ↓
Harga terakhir × exp(prediksi log-return)
                    ↓
Prediksi harga hari berikutnya
```

Konfigurasi model terdiri atas:

- window sebanyak 40 hari perdagangan;
- satu layer LSTM dengan 32 unit;
- aktivasi internal `tanh`;
- recurrent activation `sigmoid`;
- tanpa dropout;
- `Dense(1)` sebagai output log-return;
- optimizer Adam;
- learning rate sebesar 0,0003;
- Huber loss;
- gradient clipping dengan `clipnorm=1.0`;
- batch size 32;
- maksimum 150 epoch;
- training tanpa `shuffle`;
- `EarlyStopping` dengan patience 20;
- `ReduceLROnPlateau` dengan patience 6;
- `restore_best_weights=True`.

Window 40 memberikan konteks sekitar dua bulan perdagangan. Kandidat ini digunakan untuk mengetahui apakah konteks perubahan relatif yang lebih panjang mampu menangkap pola AAPL dengan lebih baik daripada window 20.

Relative Log-Price LSTM tetap hanya menggunakan kolom `Close`. Perbedaannya terletak pada bentuk representasi input dan target.

Pada raw-price model, hubungan yang dipelajari adalah:

[
P\_{t-w+1},\ldots,P\_t
\rightarrow
P\_{t+1}
]

Pada relative log-price model, hubungan yang dipelajari adalah:

[
\log\left(\frac{P\_{t-w+1}}{P\_t}\right),
\ldots,
\log\left(\frac{P\_t}{P\_t}\right)
\rightarrow
\log\left(\frac{P\_{t+1}}{P\_t}\right)
]

Pendekatan ini menggunakan harga terakhir sebagai anchor. Secara konsep, model bekerja seperti naive baseline yang diberikan koreksi kecil oleh LSTM.

Naive baseline menggunakan:

[
\hat{P}\_{t+1}=P\_t
]

Apabila Relative Log-Price LSTM memprediksi log-return nol, maka:

# [ \hat{P}\_{t+1}

# P\_t\times\exp(0)

P\_t
]

Dengan demikian, ketika model tidak menemukan pola perubahan yang kuat, prediksi secara alami akan tetap dekat dengan naive baseline. Apabila model menemukan pola tertentu, LSTM dapat memberikan koreksi naik atau turun terhadap harga terakhir.

Istilah residual dalam pendekatan ini merujuk pada perubahan atau koreksi relatif terhadap harga terakhir. Istilah tersebut tidak menunjukkan penggunaan residual connection atau skip connection seperti pada ResNet.

Pendekatan relative log-price dipilih sebagai kandidat berdasarkan temuan EDA berikut:

- `Close` AAPL memiliki autokorelasi lag-1 sangat tinggi;
- return memiliki autokorelasi lag-1 yang hampir nol;
- AAPL memiliki tren naik jangka panjang;
- distribusi harga sangat right-skewed;
- rolling mean dan rolling standard deviation berubah sepanjang waktu;
- data bersifat non-stationary;
- terdapat perubahan regime pasar;
- terdapat perubahan rentang harga yang kuat antara train dan validation;
- naive baseline merupakan pembanding yang sangat kuat.

Transformasi relative log-price berbeda dari sekadar menggunakan `log1p(Close)`. Transformasi `log1p(Close)` masih merepresentasikan level harga dalam bentuk logaritmik. Relative log-price secara langsung merepresentasikan posisi harga terhadap harga terakhir dan memprediksi perubahan relatif berikutnya.

Pada model relative log-price, input dan target distandardisasi menggunakan dua `StandardScaler` yang berbeda:

- `x_scaler` digunakan untuk input relative log-price;
- `y_scaler` digunakan untuk target log-return.

Kedua scaler hanya di-fit menggunakan sequence train. Validation hanya ditransformasikan menggunakan scaler yang telah dipelajari dari train.

Pemisahan scaler diperlukan karena distribusi nilai relative log-price pada input tidak selalu sama dengan distribusi log-return pada target.

## E. Modelling Modified

### Import, konfigurasi, dan pemeriksaan variabel

In [ ]:
# LIBRARIES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error
)
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.losses import Huber
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam


# CONFIGURATION
SEED = 42
BASELINE_WINDOW = 5
HORIZON = 1

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)


# REQUIRED VARIABLES
required_variables = [
    "amd_train",
    "amd_val",
    "amd_test",
    "aapl_train",
    "aapl_val",
    "aapl_test",
    "baseline_amd",
    "baseline_aapl",
    "amd_scaler",
    "aapl_scaler"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise NameError(
        "Variabel berikut belum tersedia: "
        + ", ".join(missing_variables)
    )

print("Seluruh data split, scaler, dan baseline model telah tersedia.")


### Fungsi evaluasi

In [ ]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    mape = (
        mean_absolute_percentage_error(
            y_true,
            y_pred
        )
        * 100
    )

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape
    }


### 1. PEMODELAN RAW CLOSE

Fungsi pembentukan sequence raw price

Fungsi ini digunakan oleh:

Modified AMD W60,
Modified AMD W20,
AMD W60 U32 No Dropout,
Modified AAPL Raw W20,

In [ ]:
def create_price_sequences(values, window):
    values = np.asarray(
        values,
        dtype=np.float32
    ).reshape(-1, 1)

    X = []
    y = []

    for i in range(window, len(values)):
        X.append(
            values[i - window:i]
        )

        y.append(
            values[i]
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32)
    )


### Preprocessing train-validation raw price

In [ ]:
def prepare_raw_train_validation(
    train_df,
    val_df,
    window
):
    train_values = (
        train_df[["Close"]]
        .to_numpy(dtype=np.float32)
    )

    val_values = (
        val_df[["Close"]]
        .to_numpy(dtype=np.float32)
    )

    scaler = MinMaxScaler()

    train_scaled = scaler.fit_transform(
        train_values
    )

    val_scaled = scaler.transform(
        val_values
    )

    X_train, y_train = create_price_sequences(
        train_scaled,
        window
    )

    val_with_context = np.vstack([
        train_scaled[-window:],
        val_scaled
    ])

    X_val, y_val = create_price_sequences(
        val_with_context,
        window
    )

    assert len(X_val) == len(val_df)
    assert len(y_val) == len(val_df)

    val_actual = scaler.inverse_transform(
        y_val.reshape(-1, 1)
    ).ravel()

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "val_actual": val_actual,
        "val_dates": (
            val_df["Date"]
            .reset_index(drop=True)
        ),
        "scaler": scaler
    }


Fungsi ini melakukan preprocessing data `Close` untuk model modified berbasis harga asli.

Data train dan validation terlebih dahulu diubah menjadi array `float32`. Selanjutnya, `MinMaxScaler` hanya di-fit pada data train untuk mencegah data leakage, sedangkan data validation hanya ditransformasikan menggunakan scaler dari train.

Data yang sudah dinormalisasi kemudian dibentuk menjadi sequence berdasarkan nilai `window`. Pada validation set, sejumlah data terakhir dari train ditambahkan sebagai konteks agar seluruh observasi validation dapat diprediksi tanpa kehilangan target awal. Penambahan konteks ini tidak menyebabkan leakage karena seluruh input berasal dari periode sebelum target.

Target validation kemudian dikembalikan ke skala harga asli menggunakan `inverse_transform`. Fungsi ini mengembalikan sequence train dan validation, target, tanggal validation, nilai aktual validation, serta scaler yang digunakan.

### Fungsi membangun raw-price LSTM

In [ ]:
def build_raw_lstm(config):
    model_layers = [
        Input(
            shape=(
                config["window"],
                1
            )
        ),
        LSTM(
            units=config["units"],
            activation="tanh"
        )
    ]

    if config["dropout"] > 0:
        model_layers.append(
            Dropout(
                config["dropout"]
            )
        )

    model_layers.append(
        Dense(1)
    )

    model_name = (
        config["name"]
        .lower()
        .replace(" ", "_")
    )

    model = Sequential(
        model_layers,
        name=model_name
    )

    optimizer_arguments = {
        "learning_rate": config[
            "learning_rate"
        ]
    }

    if config.get("clipnorm") is not None:
        optimizer_arguments["clipnorm"] = (
            config["clipnorm"]
        )

    optimizer = Adam(
        **optimizer_arguments
    )

    if config["loss"] == "huber":
        loss_function = Huber(delta=1.0)
    else:
        loss_function = "mse"

    model.compile(
        optimizer=optimizer,
        loss=loss_function,
        metrics=["mae"]
    )

    return model


### Fungsi training raw-price candidate

In [ ]:
def train_raw_candidate(
    ticker,
    train_df,
    val_df,
    config
):
    tf.keras.utils.set_random_seed(SEED)
    np.random.seed(SEED)

    data = prepare_raw_train_validation(
        train_df=train_df,
        val_df=val_df,
        window=config["window"]
    )

    model = build_raw_lstm(
        config
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=config["patience"],
            mode="min",
            restore_best_weights=True,
            verbose=0
        )
    ]

    if config.get("reduce_lr", False):
        callbacks.append(
            ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=config[
                    "lr_patience"
                ],
                min_lr=1e-6,
                mode="min",
                verbose=0
            )
        )

    history = model.fit(
        data["X_train"],
        data["y_train"],
        validation_data=(
            data["X_val"],
            data["y_val"]
        ),
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        shuffle=False,
        callbacks=callbacks,
        verbose=0
    )

    prediction_scaled = model.predict(
        data["X_val"],
        verbose=0
    )

    validation_prediction = (
        data["scaler"]
        .inverse_transform(
            prediction_scaled.reshape(-1, 1)
        )
        .ravel()
    )

    metrics = calculate_metrics(
        data["val_actual"],
        validation_prediction
    )

    best_epoch = (
        np.argmin(
            history.history["val_loss"]
        )
        + 1
    )

    return {
        "ticker": ticker,
        "representation": "raw",
        "name": config["name"],
        "config": config,
        "model": model,
        "history": history,
        "data": data,
        "actual": data["val_actual"],
        "prediction": validation_prediction,
        "dates": data["val_dates"],
        "metrics": {
            "Ticker": ticker,
            "Model": config["name"],
            **metrics,
            "Best Epoch": best_epoch
        }
    }


### 2. PEMODELAN RELATIVE LOG AAPL

### Fungsi pembentukan relative log sequence

In [ ]:
def create_relative_log_sequences(
    prices,
    window
):
    prices = np.asarray(
        prices,
        dtype=np.float64
    ).reshape(-1)

    if np.any(prices <= 0):
        raise ValueError(
            "Seluruh harga Close harus lebih besar dari nol."
        )

    X = []
    y = []
    last_prices = []

    for i in range(window, len(prices)):
        history = prices[
            i - window:i
        ]

        next_price = prices[i]
        last_price = history[-1]

        relative_history = np.log(
            history / last_price
        )

        next_log_return = np.log(
            next_price / last_price
        )

        X.append(
            relative_history.reshape(
                window,
                1
            )
        )

        y.append(
            next_log_return
        )

        last_prices.append(
            last_price
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(
            y,
            dtype=np.float32
        ).reshape(-1, 1),
        np.asarray(
            last_prices,
            dtype=np.float64
        )
    )


### Preprocessing relative log train-validation

In [ ]:
def prepare_relative_train_validation(
    train_df,
    val_df,
    window
):
    train_prices = (
        train_df["Close"]
        .to_numpy(dtype=np.float64)
    )

    val_prices = (
        val_df["Close"]
        .to_numpy(dtype=np.float64)
    )

    X_train_raw, y_train_raw, _ = (
        create_relative_log_sequences(
            train_prices,
            window
        )
    )

    val_with_context = np.concatenate([
        train_prices[-window:],
        val_prices
    ])

    (
        X_val_raw,
        y_val_raw,
        val_last_prices
    ) = create_relative_log_sequences(
        val_with_context,
        window
    )

    assert len(X_val_raw) == len(val_df)
    assert len(y_val_raw) == len(val_df)
    assert len(val_last_prices) == len(val_df)

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train = (
        x_scaler
        .fit_transform(
            X_train_raw.reshape(-1, 1)
        )
        .reshape(X_train_raw.shape)
        .astype(np.float32)
    )

    X_val = (
        x_scaler
        .transform(
            X_val_raw.reshape(-1, 1)
        )
        .reshape(X_val_raw.shape)
        .astype(np.float32)
    )

    y_train = (
        y_scaler
        .fit_transform(y_train_raw)
        .astype(np.float32)
    )

    y_val = (
        y_scaler
        .transform(y_val_raw)
        .astype(np.float32)
    )

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "y_val_raw": y_val_raw,
        "val_actual": val_prices,
        "val_last_prices": val_last_prices,
        "val_dates": (
            val_df["Date"]
            .reset_index(drop=True)
        ),
        "x_scaler": x_scaler,
        "y_scaler": y_scaler
    }


Fungsi ini melakukan preprocessing untuk model **relative log-price**.

Data harga train dan validation diubah menjadi array, kemudian dibentuk menjadi sequence relative log menggunakan harga terakhir pada setiap window sebagai acuan. Target model bukan harga langsung, melainkan perubahan log harga menuju hari berikutnya.

Pada validation set, sejumlah harga terakhir dari train ditambahkan sebagai konteks agar seluruh data validation dapat diprediksi tanpa kehilangan target awal. Penambahan konteks ini tidak menyebabkan data leakage karena seluruh input berada sebelum target.

Input dan target kemudian distandardisasi menggunakan dua `StandardScaler` yang berbeda. `x_scaler` digunakan untuk input sequence, sedangkan `y_scaler` digunakan untuk target. Keduanya hanya di-fit pada data train, sementara validation hanya ditransformasikan.

Fungsi mengembalikan data train dan validation yang telah diproses, target relative log asli, harga aktual validation, harga terakhir sebagai anchor, tanggal validation, serta kedua scaler yang diperlukan untuk mengembalikan prediksi ke skala harga asli.

### Fungsi membangun relative-log LSTM

In [ ]:
def build_relative_lstm(config):
    model_name = (
        config["name"]
        .lower()
        .replace(" ", "_")
    )

    model = Sequential(
        [
            Input(
                shape=(
                    config["window"],
                    1
                )
            ),
            LSTM(
                units=config["units"],
                activation="tanh",
                recurrent_activation="sigmoid"
            ),
            Dense(1)
        ],
        name=model_name
    )

    optimizer = Adam(
        learning_rate=config[
            "learning_rate"
        ],
        clipnorm=config["clipnorm"]
    )

    model.compile(
        optimizer=optimizer,
        loss=Huber(delta=1.0),
        metrics=["mae"]
    )

    return model


### Train Relative Log Candidates

In [ ]:
def train_relative_candidate(
    ticker,
    train_df,
    val_df,
    config
):
    # Memastikan hasil dapat direproduksi
    tf.keras.utils.set_random_seed(SEED)
    np.random.seed(SEED)

    # Menyiapkan sequence relative log-price
    data = prepare_relative_train_validation(
        train_df=train_df,
        val_df=val_df,
        window=config["window"]
    )

    # Membangun model berdasarkan konfigurasi
    model = build_relative_lstm(
        config=config
    )

    # Menghentikan training saat validation loss tidak membaik
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=config["patience"],
        mode="min",
        restore_best_weights=True,
        verbose=0
    )

    # Menurunkan learning rate saat validation loss stagnan
    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=config["lr_patience"],
        min_lr=1e-6,
        mode="min",
        verbose=0
    )

    # Melatih model menggunakan train dan validation
    history = model.fit(
        data["X_train"],
        data["y_train"],
        validation_data=(
            data["X_val"],
            data["y_val"]
        ),
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        shuffle=False,
        callbacks=[
            early_stopping,
            reduce_lr
        ],
        verbose=0
    )

    # Memprediksi target validation dalam skala standardisasi
    prediction_scaled = model.predict(
        data["X_val"],
        verbose=0
    )

    # Mengembalikan prediksi ke skala log-return asli
    predicted_log_return = (
        data["y_scaler"]
        .inverse_transform(
            prediction_scaled
        )
        .ravel()
    )

    # Mengembalikan target validation ke log-return asli
    actual_log_return = (
        data["y_scaler"]
        .inverse_transform(
            data["y_val"]
        )
        .ravel()
    )

    # Merekonstruksi harga prediksi menggunakan harga terakhir
    validation_prediction = (
        data["val_last_prices"]
        * np.exp(
            predicted_log_return
        )
    )

    # Harga aktual validation
    validation_actual = (
        data["val_actual"]
    )

    # Memastikan jumlah prediksi sesuai jumlah target
    assert (
        len(validation_prediction)
        == len(validation_actual)
    ), "Jumlah prediksi dan target validation tidak sama."

    assert (
        len(validation_prediction)
        == len(val_df)
    ), "Jumlah prediksi tidak sama dengan jumlah data validation."

    # Menghitung metrik pada skala harga asli
    metrics = calculate_metrics(
        validation_actual,
        validation_prediction
    )

    # Menentukan epoch dengan validation loss terendah
    best_epoch = (
        np.argmin(
            history.history["val_loss"]
        )
        + 1
    )

    # Menyimpan seluruh hasil eksperimen
    return {
        "ticker": ticker,
        "representation": "relative_log",
        "name": config["name"],
        "config": config,
        "model": model,
        "history": history,
        "data": data,
        "actual": validation_actual,
        "prediction": validation_prediction,
        "dates": data["val_dates"],
        "predicted_log_return": predicted_log_return,
        "actual_log_return": actual_log_return,
        "metrics": {
            "Ticker": ticker,
            "Model": config["name"],
            **metrics,
            "Best Epoch": best_epoch
        }
    }


### 3. KONFIGURASI SELURUH MODIFIED MODEL

### Konfigurasi kandidat AMD

In [ ]:
AMD_MODIFIED_CONFIGS = [
    {
        "name": "Modified AMD W60",
        "representation": "raw",
        "window": 60,
        "units": 32,
        "dropout": 0.2,
        "learning_rate": 0.0005,
        "loss": "mse",
        "clipnorm": None,
        "epochs": 100,
        "batch_size": 32,
        "patience": 15,
        "reduce_lr": False,
        "lr_patience": 5
    },
    {
        "name": "Modified AMD W20",
        "representation": "raw",
        "window": 20,
        "units": 32,
        "dropout": 0.2,
        "learning_rate": 0.0005,
        "loss": "mse",
        "clipnorm": None,
        "epochs": 100,
        "batch_size": 32,
        "patience": 15,
        "reduce_lr": False,
        "lr_patience": 5
    },
    {
        "name": "AMD W60 U32 No Dropout",
        "representation": "raw",
        "window": 60,
        "units": 32,
        "dropout": 0.0,
        "learning_rate": 0.00025,
        "loss": "huber",
        "clipnorm": 1.0,
        "epochs": 100,
        "batch_size": 32,
        "patience": 15,
        "reduce_lr": True,
        "lr_patience": 5
    }
]


Tiga konfigurasi digunakan untuk membandingkan pengaruh **window size, dropout, loss function, learning rate, dan pengaturan optimasi** terhadap performa AMD.

- `representation="raw"` dipilih karena model menggunakan harga `Close` asli yang telah dinormalisasi dengan `MinMaxScaler`.
- `window=20` digunakan untuk menangkap pola jangka pendek sekitar satu bulan perdagangan, sedangkan `window=60` digunakan untuk menangkap pola yang lebih panjang sekitar tiga bulan perdagangan.
- `units=32` dipilih agar model lebih ringan dibandingkan baseline 50 unit, tetapi tetap memiliki kapasitas yang cukup untuk mempelajari pola temporal.
- `dropout=0.2` pada dua kandidat awal digunakan untuk mengurangi risiko overfitting. Kandidat ketiga tidak menggunakan dropout untuk menguji apakah seluruh informasi sequence dapat dipertahankan.
- `learning_rate=0.0005` digunakan pada kandidat awal, sedangkan `0.00025` pada kandidat ketiga digunakan agar proses pembaruan bobot lebih stabil.
- `loss="mse"` digunakan sebagai loss standar regresi pada dua kandidat awal. Kandidat ketiga menggunakan `Huber loss` agar model lebih tahan terhadap perubahan harga ekstrem.
- `clipnorm=1.0` pada kandidat ketiga digunakan untuk membatasi gradient yang terlalu besar selama training.
- `epochs=100` memberikan kesempatan yang cukup bagi model untuk belajar, sedangkan `batch_size=32` dipilih sebagai ukuran batch yang umum dan stabil.
- `patience=15` digunakan pada `EarlyStopping` agar training berhenti apabila validation loss tidak membaik selama 15 epoch.
- `reduce_lr=True` pada kandidat ketiga memungkinkan learning rate diturunkan ketika validation loss stagnan, dengan `lr_patience=5`.

Secara keseluruhan, `Modified AMD W20` dan `Modified AMD W60` digunakan untuk membandingkan panjang window, sedangkan `AMD W60 U32 No Dropout` digunakan sebagai eksperimen lanjutan dengan optimasi yang lebih stabil melalui Huber loss, gradient clipping, dan penurunan learning rate.

### Konfigurasi kandidat AAPL

In [ ]:
AAPL_MODIFIED_CONFIGS = [
    {
        "name": "Modified AAPL Raw W20",
        "representation": "raw",
        "window": 20,
        "units": 32,
        "dropout": 0.2,
        "learning_rate": 0.0005,
        "loss": "mse",
        "clipnorm": None,
        "epochs": 100,
        "batch_size": 32,
        "patience": 15,
        "reduce_lr": False,
        "lr_patience": 5
    },
    {
        "name": "AAPL Relative Log W40 U32 Huber",
        "representation": "relative_log",
        "window": 40,
        "units": 32,
        "learning_rate": 0.0003,
        "clipnorm": 1.0,
        "epochs": 150,
        "batch_size": 32,
        "patience": 20,
        "lr_patience": 6
    },
    {
        "name": "AAPL Relative Log W20 U32 Huber",
        "representation": "relative_log",
        "window": 20,
        "units": 32,
        "learning_rate": 0.0003,
        "clipnorm": 1.0,
        "epochs": 150,
        "batch_size": 32,
        "patience": 20,
        "lr_patience": 6
    }
]


Tiga konfigurasi AAPL digunakan untuk membandingkan representasi harga mentah dengan relative log-price, serta melihat pengaruh perbedaan window.

- `Modified AAPL Raw W20` menggunakan harga `Close` asli dengan window 20 hari. `units=32` dipilih agar model lebih ringan, sedangkan `dropout=0.2` digunakan untuk mengurangi risiko overfitting. Model memakai learning rate `0.0005` dan MSE sebagai konfigurasi regresi awal.
- `AAPL Relative Log W40 U32 Huber` menggunakan perubahan harga relatif berbasis log dengan window 40 hari. Representasi ini dipilih untuk mengurangi pengaruh tren dan perubahan skala harga AAPL. Window 40 digunakan untuk memberikan konteks temporal yang lebih panjang.
- `AAPL Relative Log W20 U32 Huber` menggunakan representasi yang sama, tetapi dengan window 20 hari. Konfigurasi ini digunakan untuk mengetahui apakah konteks yang lebih pendek lebih sesuai dibandingkan window 40.

Pada kedua model relative log, `units=32` digunakan untuk menjaga kompleksitas model. Learning rate `0.0003` dipilih agar pembaruan bobot lebih stabil, sedangkan `clipnorm=1.0` digunakan untuk membatasi gradient yang terlalu besar. Huber loss digunakan agar model lebih tahan terhadap perubahan harga ekstrem.

Jumlah epoch maksimum dibuat lebih besar, yaitu 150, karena target relative log cenderung lebih sulit dipelajari. `EarlyStopping` dengan `patience=20` mencegah training berlangsung terlalu lama ketika validation loss tidak membaik. Selain itu, learning rate akan diturunkan setelah 6 epoch tanpa peningkatan melalui `ReduceLROnPlateau`.

Secara keseluruhan, konfigurasi pertama menguji pendekatan harga mentah, sedangkan dua konfigurasi berikutnya menguji relative log-price dengan window 20 dan 40 hari. Model terbaik kemudian dipilih berdasarkan nilai RMSE validation terendah.

### 4. TRAINING SELURUH MODIFIED MODEL

### Menjalankan seluruh kandidat AMD

In [ ]:
amd_modified_runs = {}

for config in AMD_MODIFIED_CONFIGS:
    print(
        f"Training: {config['name']}"
    )

    run = train_raw_candidate(
        ticker="AMD",
        train_df=amd_train,
        val_df=amd_val,
        config=config
    )

    amd_modified_runs[
        config["name"]
    ] = run

print("Training seluruh kandidat AMD selesai.")


### Menjalankan seluruh kandidat AAPL

In [ ]:
aapl_modified_runs = {}

for config in AAPL_MODIFIED_CONFIGS:
    print(
        f"Training: {config['name']}"
    )

    if (
        config["representation"]
        == "raw"
    ):
        run = train_raw_candidate(
            ticker="AAPL",
            train_df=aapl_train,
            val_df=aapl_val,
            config=config
        )

    else:
        run = train_relative_candidate(
            ticker="AAPL",
            train_df=aapl_train,
            val_df=aapl_val,
            config=config
        )

    aapl_modified_runs[
        config["name"]
    ] = run

print("Training seluruh kandidat AAPL selesai.")


### 5. PERBANDINGAN VALIDATION

### Membandingkan seluruh modified model

In [ ]:
modified_validation_results = pd.DataFrame(
    [
        run["metrics"]
        for run in amd_modified_runs.values()
    ]
    +
    [
        run["metrics"]
        for run in aapl_modified_runs.values()
    ]
)

modified_validation_results = (
    modified_validation_results
    .sort_values(
        [
            "Ticker",
            "RMSE",
            "MAE",
            "MAPE (%)"
        ]
    )
    .reset_index(drop=True)
)

modified_validation_results


### Memilih modified model terbaik

In [ ]:
amd_modified_validation_results = (
    modified_validation_results[
        modified_validation_results[
            "Ticker"
        ] == "AMD"
    ]
    .sort_values(
        [
            "RMSE",
            "MAE",
            "MAPE (%)"
        ]
    )
    .reset_index(drop=True)
)

aapl_modified_validation_results = (
    modified_validation_results[
        modified_validation_results[
            "Ticker"
        ] == "AAPL"
    ]
    .sort_values(
        [
            "RMSE",
            "MAE",
            "MAPE (%)"
        ]
    )
    .reset_index(drop=True)
)


best_modified_amd_name = (
    amd_modified_validation_results
    .iloc[0]["Model"]
)

best_modified_aapl_name = (
    aapl_modified_validation_results
    .iloc[0]["Model"]
)


best_modified_amd_run = (
    amd_modified_runs[
        best_modified_amd_name
    ]
)

best_modified_aapl_run = (
    aapl_modified_runs[
        best_modified_aapl_name
    ]
)


best_modified_amd_model = (
    best_modified_amd_run["model"]
)

best_modified_aapl_model = (
    best_modified_aapl_run["model"]
)


print(
    "Best Modified AMD :",
    best_modified_amd_name
)

print(
    "Best Modified AAPL:",
    best_modified_aapl_name
)


Model terbaik dipilih berdasarkan **nilai RMSE pada validation set yang paling rendah**. RMSE digunakan sebagai kriteria utama karena memberikan penalti lebih besar pada kesalahan prediksi yang besar, sehingga sesuai untuk mengevaluasi forecasting harga saham yang dapat mengalami perubahan ekstrem.

Validation set digunakan untuk membandingkan seluruh kandidat agar test set tetap digunakan hanya untuk evaluasi akhir. MAE dan MAPE tetap diperhatikan sebagai metrik pendukung untuk melihat rata-rata kesalahan absolut dan kesalahan relatif dalam persentase.

Berdasarkan hasil validation, **AMD W60 U32 No Dropout** dipilih sebagai modified model terbaik untuk AMD, sedangkan **AAPL Relative Log W40 U32 Huber** dipilih sebagai modified model terbaik untuk AAPL karena masing-masing menghasilkan RMSE validation paling rendah dibandingkan kandidat modified lainnya.

### Ringkasan model terbaik

In [ ]:
best_modified_summary = pd.DataFrame([
    best_modified_amd_run["metrics"],
    best_modified_aapl_run["metrics"]
])

best_modified_summary


### 6. OUTPUT TRAINING DAN VALIDATION MODEL TERBAIK

### Fungsi menampilkan seluruh riwayat epoch

In [ ]:
# FULL TRAINING HISTORY
from IPython.display import display


def show_full_training_history(run):
    history_df = pd.DataFrame(
        run["history"].history
    )

    history_df.insert(
        0,
        "Epoch",
        np.arange(1, len(history_df) + 1)
    )

    best_epoch = (
        history_df["val_loss"].idxmin() + 1
    )

    print(f"Model       : {run['name']}")
    print(f"Ticker      : {run['ticker']}")
    print(f"Total Epoch : {len(history_df)}")
    print(f"Best Epoch  : {best_epoch}")
    print(
        f"Best Val Loss: "
        f"{history_df['val_loss'].min():.8f}"
    )

    print("\nSeluruh riwayat training dan validation:")

    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.float_format",
        "{:.8f}".format
    ):
        display(history_df)

    return history_df


### Menampilkan seluruh epoch AMD terbaik

In [ ]:
amd_best_history_df = show_full_training_history(
    best_modified_amd_run
)


Training model **AMD W60 U32 No Dropout** berjalan stabil. Pada epoch awal, training loss dan validation loss sama-sama menurun dengan cepat, menunjukkan bahwa model mampu mempelajari pola sequence 60 hari dan tetap melakukan generalisasi pada validation.

`ReduceLROnPlateau` menurunkan learning rate secara bertahap dari 0,00025 hingga 0,000001. Setiap penurunan learning rate membantu validation loss kembali membaik ketika proses optimasi mulai stagnan.

Pada epoch ke-100 diperoleh:

- Training loss: 0,00012433
- Validation loss: 0,00014801
- Training MAE: 0,00958295
- Validation MAE: 0,01039819

Validation loss dan validation MAE hanya sedikit lebih tinggi daripada nilai train. Selain itu, training dan validation loss tetap menurun hingga epoch terakhir. Oleh karena itu, tidak terlihat indikasi overfitting yang kuat maupun underfitting.

Best epoch berada pada epoch ke-100 karena validation loss terendah diperoleh pada epoch terakhir. Namun, karena learning rate sudah mencapai nilai minimum dan penurunannya sangat kecil, model dapat dianggap telah mendekati konvergensi.

Secara keseluruhan, model menunjukkan proses training yang stabil, gap train-validation yang kecil, dan kemampuan generalisasi yang cukup baik. Meskipun demikian, nilai loss dan MAE pada tabel masih berada dalam skala normalisasi. Performa akhir tetap dinilai menggunakan RMSE, MAE, dan MAPE setelah prediksi dikembalikan ke skala harga asli.

### Menampilkan seluruh epoch AAPL terbaik

In [ ]:
aapl_best_history_df = show_full_training_history(
    best_modified_aapl_run
)


Training model AAPL menunjukkan bahwa nilai training loss dan validation loss hanya menurun secara perlahan. Training loss berkurang dari 0,3521 menjadi 0,3492, sedangkan validation loss mencapai nilai terendah sekitar 0,1225 pada epoch ke-26.

Setelah epoch ke-26, training loss masih terus menurun, tetapi validation loss mulai meningkat secara perlahan hingga 0,1227. Pola tersebut menunjukkan indikasi **overfitting ringan**, karena model semakin menyesuaikan diri dengan data train tanpa menghasilkan peningkatan pada validation.

`ReduceLROnPlateau` menurunkan learning rate secara bertahap dari 0,0003 menjadi 0,00001875. Namun, penurunan learning rate setelah epoch terbaik tidak berhasil memperbaiki validation loss secara berarti. Oleh karena itu, bobot pada epoch ke-26 menjadi bobot yang paling sesuai dan akan dikembalikan oleh `EarlyStopping` melalui `restore_best_weights=True`.

Nilai validation loss yang lebih rendah daripada training loss tidak otomatis menunjukkan kesalahan atau data leakage. Hal tersebut dapat terjadi karena distribusi log-return validation lebih sempit atau lebih mudah diprediksi dibandingkan train. Nilai loss dan MAE juga masih berada pada skala target log-return yang telah distandardisasi, bukan dalam satuan harga asli.

Secara keseluruhan, model menunjukkan proses training yang stabil, tetapi peningkatannya relatif kecil. Hal ini sesuai dengan hasil EDA bahwa autokorelasi return AAPL sangat rendah, sehingga pola perubahan harga berikutnya sulit dipelajari. Model kemungkinan menghasilkan prediksi log-return yang dekat dengan nol, sehingga prediksi harga akhirnya berada dekat dengan harga terakhir atau naive baseline.

Performa akhir tetap harus dinilai menggunakan RMSE, MAE, dan MAPE setelah prediksi direkonstruksi dan dikembalikan ke skala harga asli.

### Learning curve seluruh proses training

In [ ]:
# LEARNING CURVE
def plot_complete_learning_curve(run):
    history_df = pd.DataFrame(
        run["history"].history
    )

    epochs = np.arange(
        1,
        len(history_df) + 1
    )

    best_epoch = (
        history_df["val_loss"].idxmin() + 1
    )

    plt.figure(figsize=(11, 5))

    plt.plot(
        epochs,
        history_df["loss"],
        label="Training Loss"
    )

    plt.plot(
        epochs,
        history_df["val_loss"],
        label="Validation Loss"
    )

    plt.axvline(
        best_epoch,
        linestyle="--",
        label=f"Best Epoch: {best_epoch}"
    )

    plt.scatter(
        best_epoch,
        history_df.loc[
            best_epoch - 1,
            "val_loss"
        ],
        label="Minimum Validation Loss"
    )

    plt.title(
        f"Training and Validation Loss - "
        f"{run['name']}"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


### AMD

In [ ]:
plot_complete_learning_curve(
    best_modified_amd_run
)


### AAPL

In [ ]:
plot_complete_learning_curve(
    best_modified_aapl_run
)


## F. Data Tesstiingg

### Menyiapkan test untuk baseline LSTM

In [ ]:
def prepare_baseline_test(
    val_df,
    test_df,
    scaler,
    window=5
):
    val_values = (
        val_df[["Close"]]
        .to_numpy(dtype=np.float32)
    )

    test_values = (
        test_df[["Close"]]
        .to_numpy(dtype=np.float32)
    )

    val_scaled = scaler.transform(
        val_values
    )

    test_scaled = scaler.transform(
        test_values
    )

    test_with_context = np.vstack([
        val_scaled[-window:],
        test_scaled
    ])

    X_test, y_test = create_price_sequences(
        test_with_context,
        window
    )

    actual = scaler.inverse_transform(
        y_test.reshape(-1, 1)
    ).ravel()

    assert len(X_test) == len(test_df)
    assert len(y_test) == len(test_df)

    return X_test, y_test, actual


### Menyiapkan test untuk modified raw-price model

In [ ]:
def evaluate_raw_modified_test(
    run,
    val_df,
    test_df
):
    window = run["config"]["window"]
    scaler = run["data"]["scaler"]

    val_values = (
        val_df[["Close"]]
        .to_numpy(dtype=np.float32)
    )

    test_values = (
        test_df[["Close"]]
        .to_numpy(dtype=np.float32)
    )

    val_scaled = scaler.transform(
        val_values
    )

    test_scaled = scaler.transform(
        test_values
    )

    test_with_context = np.vstack([
        val_scaled[-window:],
        test_scaled
    ])

    X_test, y_test = create_price_sequences(
        test_with_context,
        window
    )

    prediction_scaled = (
        run["model"]
        .predict(
            X_test,
            verbose=0
        )
    )

    actual = scaler.inverse_transform(
        y_test.reshape(-1, 1)
    ).ravel()

    prediction = scaler.inverse_transform(
        prediction_scaled.reshape(-1, 1)
    ).ravel()

    assert len(actual) == len(test_df)
    assert len(prediction) == len(test_df)

    return {
        "actual": actual,
        "prediction": prediction,
        "dates": (
            test_df["Date"]
            .reset_index(drop=True)
        )
    }


### Menyiapkan test untuk relative-log model

In [ ]:
def evaluate_relative_modified_test(
    run,
    val_df,
    test_df
):
    window = run["config"]["window"]

    val_prices = (
        val_df["Close"]
        .to_numpy(dtype=np.float64)
    )

    test_prices = (
        test_df["Close"]
        .to_numpy(dtype=np.float64)
    )

    test_with_context = np.concatenate([
        val_prices[-window:],
        test_prices
    ])

    (
        X_test_raw,
        y_test_raw,
        test_last_prices
    ) = create_relative_log_sequences(
        test_with_context,
        window
    )

    X_test = (
        run["data"]["x_scaler"]
        .transform(
            X_test_raw.reshape(-1, 1)
        )
        .reshape(X_test_raw.shape)
        .astype(np.float32)
    )

    prediction_scaled = (
        run["model"]
        .predict(
            X_test,
            verbose=0
        )
    )

    predicted_log_return = (
        run["data"]["y_scaler"]
        .inverse_transform(
            prediction_scaled
        )
        .ravel()
    )

    prediction = (
        test_last_prices
        * np.exp(
            predicted_log_return
        )
    )

    actual = test_prices

    assert len(actual) == len(test_df)
    assert len(prediction) == len(test_df)

    return {
        "actual": actual,
        "prediction": prediction,
        "dates": (
            test_df["Date"]
            .reset_index(drop=True)
        )
    }


### Fungsi evaluasi modified model terbaik

In [ ]:
def evaluate_best_modified_test(
    run,
    val_df,
    test_df
):
    if (
        run["representation"]
        == "raw"
    ):
        return evaluate_raw_modified_test(
            run=run,
            val_df=val_df,
            test_df=test_df
        )

    return evaluate_relative_modified_test(
        run=run,
        val_df=val_df,
        test_df=test_df
    )


### Prediksi naive baseline

In [ ]:
def create_naive_test_prediction(
    val_df,
    test_df
):
    last_validation_price = (
        val_df["Close"]
        .iloc[-1]
    )

    previous_test_prices = (
        test_df["Close"]
        .iloc[:-1]
        .to_numpy(dtype=np.float64)
    )

    naive_prediction = np.concatenate([
        [last_validation_price],
        previous_test_prices
    ])

    assert (
        len(naive_prediction)
        == len(test_df)
    )

    return naive_prediction


### Testing naive dan baseline AMD

In [ ]:
amd_test_dates = (
    amd_test["Date"]
    .reset_index(drop=True)
)

amd_test_actual = (
    amd_test["Close"]
    .to_numpy(dtype=np.float64)
)

amd_naive_test_prediction = (
    create_naive_test_prediction(
        val_df=amd_val,
        test_df=amd_test
    )
)

(
    X_test_amd_baseline,
    y_test_amd_baseline,
    amd_baseline_actual
) = prepare_baseline_test(
    val_df=amd_val,
    test_df=amd_test,
    scaler=amd_scaler,
    window=BASELINE_WINDOW
)

amd_baseline_prediction_scaled = (
    baseline_amd.predict(
        X_test_amd_baseline,
        verbose=0
    )
)

amd_baseline_test_prediction = (
    amd_scaler
    .inverse_transform(
        amd_baseline_prediction_scaled
        .reshape(-1, 1)
    )
    .ravel()
)

assert np.allclose(
    amd_test_actual,
    amd_baseline_actual,
    atol=1e-5
)


### Testing naive dan baseline AAPL

In [ ]:
aapl_test_dates = (
    aapl_test["Date"]
    .reset_index(drop=True)
)

aapl_test_actual = (
    aapl_test["Close"]
    .to_numpy(dtype=np.float64)
)

aapl_naive_test_prediction = (
    create_naive_test_prediction(
        val_df=aapl_val,
        test_df=aapl_test
    )
)

(
    X_test_aapl_baseline,
    y_test_aapl_baseline,
    aapl_baseline_actual
) = prepare_baseline_test(
    val_df=aapl_val,
    test_df=aapl_test,
    scaler=aapl_scaler,
    window=BASELINE_WINDOW
)

aapl_baseline_prediction_scaled = (
    baseline_aapl.predict(
        X_test_aapl_baseline,
        verbose=0
    )
)

aapl_baseline_test_prediction = (
    aapl_scaler
    .inverse_transform(
        aapl_baseline_prediction_scaled
        .reshape(-1, 1)
    )
    .ravel()
)

assert np.allclose(
    aapl_test_actual,
    aapl_baseline_actual,
    atol=1e-5
)


### Testing modified AMD terbaik

In [ ]:
best_modified_amd_test = (
    evaluate_best_modified_test(
        run=best_modified_amd_run,
        val_df=amd_val,
        test_df=amd_test
    )
)

assert np.allclose(
    amd_test_actual,
    best_modified_amd_test["actual"],
    atol=1e-5
)


### Testing modified AAPL terbaik

In [ ]:
best_modified_aapl_test = (
    evaluate_best_modified_test(
        run=best_modified_aapl_run,
        val_df=aapl_val,
        test_df=aapl_test
    )
)

assert np.allclose(
    aapl_test_actual,
    best_modified_aapl_test["actual"],
    atol=1e-5
)


## G. HASIL FINAL TEST

### Tabel perbandingan test

In [ ]:
final_test_results = pd.DataFrame([
    {
        "Ticker": "AMD",
        "Model": "Naive Baseline",
        **calculate_metrics(
            amd_test_actual,
            amd_naive_test_prediction
        )
    },
    {
        "Ticker": "AMD",
        "Model": "Baseline LSTM",
        **calculate_metrics(
            amd_test_actual,
            amd_baseline_test_prediction
        )
    },
    {
        "Ticker": "AMD",
        "Model": best_modified_amd_name,
        **calculate_metrics(
            amd_test_actual,
            best_modified_amd_test[
                "prediction"
            ]
        )
    },
    {
        "Ticker": "AAPL",
        "Model": "Naive Baseline",
        **calculate_metrics(
            aapl_test_actual,
            aapl_naive_test_prediction
        )
    },
    {
        "Ticker": "AAPL",
        "Model": "Baseline LSTM",
        **calculate_metrics(
            aapl_test_actual,
            aapl_baseline_test_prediction
        )
    },
    {
        "Ticker": "AAPL",
        "Model": best_modified_aapl_name,
        **calculate_metrics(
            aapl_test_actual,
            best_modified_aapl_test[
                "prediction"
            ]
        )
    }
])

final_test_results = (
    final_test_results
    .sort_values(
        [
            "Ticker",
            "RMSE",
            "MAE",
            "MAPE (%)"
        ]
    )
    .reset_index(drop=True)
)

final_test_results


Hasil final test menunjukkan bahwa **AAPL Relative Log W40 U32 Huber** memberikan performa terbaik pada AAPL dengan RMSE 6,1952, MAE 3,8172, dan MAPE 1,57%. Model ini sedikit mengungguli naive baseline pada seluruh metrik. Hasil tersebut menunjukkan bahwa representasi relative log-price membantu model menghadapi perubahan skala harga AAPL dengan menggunakan harga terakhir sebagai anchor dan memprediksi koreksi berupa log-return.

Pada AMD, naive baseline tetap memberikan hasil terbaik dengan RMSE 1,3745, MAE 0,9112, dan MAPE 2,44%. Meskipun belum mampu mengalahkan naive baseline, **AMD W60 U32 No Dropout** menghasilkan performa yang jauh lebih baik daripada baseline LSTM. Dibandingkan baseline, modified model AMD menurunkan RMSE sekitar 42,01%, MAE sekitar 54,37%, dan MAPE sekitar 57,23%.

Secara keseluruhan, modified model berhasil memperbaiki baseline LSTM pada kedua saham. Modified AAPL bahkan mampu sedikit mengungguli naive baseline, sedangkan modified AMD masih berada di bawah naive baseline. Perbedaan tersebut menunjukkan bahwa konfigurasi dan representasi data perlu disesuaikan dengan karakteristik masing-masing saham.

### Menghitung peningkatan modified terhadap baseline

In [ ]:
def calculate_metric_improvement(
    reference_metrics,
    candidate_metrics
):
    return {
        "RMSE Improvement (%)": (
            (
                reference_metrics["RMSE"]
                - candidate_metrics["RMSE"]
            )
            / reference_metrics["RMSE"]
            * 100
        ),
        "MAE Improvement (%)": (
            (
                reference_metrics["MAE"]
                - candidate_metrics["MAE"]
            )
            / reference_metrics["MAE"]
            * 100
        ),
        "MAPE Improvement (%)": (
            (
                reference_metrics[
                    "MAPE (%)"
                ]
                - candidate_metrics[
                    "MAPE (%)"
                ]
            )
            / reference_metrics[
                "MAPE (%)"
            ]
            * 100
        )
    }


In [ ]:
amd_baseline_metrics = calculate_metrics(
    amd_test_actual,
    amd_baseline_test_prediction
)

amd_modified_metrics = calculate_metrics(
    amd_test_actual,
    best_modified_amd_test[
        "prediction"
    ]
)

aapl_baseline_metrics = calculate_metrics(
    aapl_test_actual,
    aapl_baseline_test_prediction
)

aapl_modified_metrics = calculate_metrics(
    aapl_test_actual,
    best_modified_aapl_test[
        "prediction"
    ]
)

modified_test_improvement = pd.DataFrame([
    {
        "Ticker": "AMD",
        "Comparison": (
            f"{best_modified_amd_name} "
            "vs Baseline LSTM"
        ),
        **calculate_metric_improvement(
            amd_baseline_metrics,
            amd_modified_metrics
        )
    },
    {
        "Ticker": "AAPL",
        "Comparison": (
            f"{best_modified_aapl_name} "
            "vs Baseline LSTM"
        ),
        **calculate_metric_improvement(
            aapl_baseline_metrics,
            aapl_modified_metrics
        )
    }
])

modified_test_improvement


Hasil perbandingan menunjukkan bahwa modified model berhasil meningkatkan performa baseline LSTM pada kedua saham.

Pada AMD, **AMD W60 U32 No Dropout** menurunkan RMSE sebesar 42,01%, MAE sebesar 54,37%, dan MAPE sebesar 57,23% dibandingkan baseline LSTM. Peningkatan terbesar terjadi pada MAPE, yang menunjukkan bahwa modified model menghasilkan kesalahan relatif yang jauh lebih kecil.

Pada AAPL, **AAPL Relative Log W40 U32 Huber** menurunkan RMSE sebesar 28,32%, MAE sebesar 48,05%, dan MAPE sebesar 48,33% dibandingkan baseline LSTM. Hasil ini menunjukkan bahwa penggunaan relative log-price membantu model menghadapi tren dan perubahan skala harga AAPL.

Seluruh nilai improvement bernilai positif, sehingga kedua modified model terbukti lebih baik daripada baseline LSTM pada final test. Modified AMD memberikan peningkatan yang lebih besar, tetapi modified AAPL memiliki keunggulan tambahan karena juga sedikit mengalahkan naive baseline.

## H. Visualisasi Final Test

### Grafik final test AMD

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    amd_test_dates,
    amd_test_actual,
    label="Actual"
)

plt.plot(
    amd_test_dates,
    amd_naive_test_prediction,
    label="Naive Baseline",
    alpha=0.7
)

plt.plot(
    amd_test_dates,
    amd_baseline_test_prediction,
    label="Baseline LSTM"
)

plt.plot(
    amd_test_dates,
    best_modified_amd_test[
        "prediction"
    ],
    label=best_modified_amd_name
)

plt.title(
    "Final Test Prediction - AMD"
)

plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()


### Grafik final test AAPL

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    aapl_test_dates,
    aapl_test_actual,
    label="Actual"
)

plt.plot(
    aapl_test_dates,
    aapl_naive_test_prediction,
    label="Naive Baseline",
    alpha=0.7
)

plt.plot(
    aapl_test_dates,
    aapl_baseline_test_prediction,
    label="Baseline LSTM"
)

plt.plot(
    aapl_test_dates,
    best_modified_aapl_test[
        "prediction"
    ],
    label=best_modified_aapl_name
)

plt.title(
    "Final Test Prediction - AAPL"
)

plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()


### Tabel prediksi AMD(Sebaggai Contoh Aja)

In [ ]:
amd_final_predictions = pd.DataFrame({
    "Date": amd_test_dates,
    "Actual": amd_test_actual,
    "Naive Baseline": (
        amd_naive_test_prediction
    ),
    "Baseline LSTM": (
        amd_baseline_test_prediction
    ),
    best_modified_amd_name: (
        best_modified_amd_test[
            "prediction"
        ]
    )
})

amd_final_predictions.head()


### Tabel prediksi AAPL(Sebagai Contoh Aja)

In [ ]:
aapl_final_predictions = pd.DataFrame({
    "Date": aapl_test_dates,
    "Actual": aapl_test_actual,
    "Naive Baseline": (
        aapl_naive_test_prediction
    ),
    "Baseline LSTM": (
        aapl_baseline_test_prediction
    ),
    best_modified_aapl_name: (
        best_modified_aapl_test[
            "prediction"
        ]
    )
})

aapl_final_predictions.head()


## SUMMARY AKHIR

Evaluasi dilakukan pada **test set satu tahun terakhir**, yaitu 1 April 2019 sampai 1 April 2020. Model yang dibandingkan adalah arsitektur **baseline LSTM** dan **modified LSTM** untuk masing-masing saham. Sebagai pembanding tambahan, digunakan naive baseline yang memprediksi harga berikutnya sama dengan harga terakhir pada input.

Nilai evaluasi dihitung setelah hasil prediksi dikembalikan ke skala harga asli. Metrik yang digunakan adalah:

- **RMSE** mengukur besar kesalahan prediksi dengan memberikan penalti lebih besar pada kesalahan yang ekstrem. Semakin kecil RMSE, semakin baik performa model.
- **MAE** menunjukkan rata-rata selisih absolut antara harga aktual dan harga prediksi. Nilainya dapat diinterpretasikan langsung dalam satuan harga saham.
- **MAPE** menunjukkan rata-rata persentase kesalahan prediksi terhadap harga aktual. Semakin kecil MAPE, semakin akurat model secara relatif.

| Ticker | Model | RMSE | MAE | MAPE |
| --- | --- | --- | --- | --- |
| AAPL | AAPL Relative Log W40 U32 Huber | 6.1952 | 3.8172 | 1.5670% |
| AAPL | Baseline LSTM | 8.6425 | 7.3481 | 3.0325% |
| AMD | AMD W60 U32 No Dropout | 1.8757 | 1.3268 | 3.4720% |
| AMD | Baseline LSTM | 3.2348 | 2.9075 | 8.1176% |

Pada saham **AMD**, modified model **AMD W60 U32 No Dropout** menghasilkan RMSE sebesar 1,8757, MAE sebesar 1,3268, dan MAPE sebesar 3,47%. Hasil tersebut lebih baik daripada baseline LSTM yang menghasilkan RMSE 3,2348, MAE 2,9075, dan MAPE 8,12%.

Dibandingkan baseline LSTM, modified model AMD berhasil menurunkan RMSE sebesar **42,01%**, MAE sebesar **54,37%**, dan MAPE sebesar **57,23%**. Peningkatan ini menunjukkan bahwa penggunaan window 60 hari, aktivasi `tanh`, Huber loss, learning rate yang lebih kecil, gradient clipping, dan penyesuaian learning rate membantu model mempelajari pola harga AMD dengan lebih stabil.

Meskipun modified model AMD jauh lebih baik daripada baseline LSTM, hasilnya masih berada di bawah naive baseline yang memiliki RMSE 1,3745, MAE 0,9112, dan MAPE 2,44%. RMSE modified AMD sekitar 36,47% lebih tinggi daripada naive baseline. Hal ini menunjukkan bahwa pola harga AMD pada test set masih sangat dipengaruhi oleh harga hari sebelumnya, sehingga prediksi sederhana menggunakan harga terakhir menjadi pembanding yang sulit dikalahkan.

Pada saham **AAPL**, modified model **AAPL Relative Log W40 U32 Huber** menghasilkan RMSE sebesar 6,1952, MAE sebesar 3,8172, dan MAPE sebesar 1,57%. Hasil tersebut lebih baik daripada baseline LSTM yang memperoleh RMSE 8,6425, MAE 7,3481, dan MAPE 3,03%.

Dibandingkan baseline LSTM, modified model AAPL berhasil menurunkan RMSE sebesar **28,32%**, MAE sebesar **48,05%**, dan MAPE sebesar **48,33%**. Penggunaan relative log-price membantu model mempelajari perubahan harga secara relatif terhadap harga terakhir, sehingga model tidak terlalu bergantung pada skala harga absolut AAPL yang terus meningkat.

Modified model AAPL juga sedikit lebih baik daripada naive baseline. Dibandingkan naive baseline, modified model AAPL menurunkan RMSE sekitar **1,38%**, MAE sekitar **0,37%**, dan MAPE sekitar **0,15%**. Walaupun peningkatannya kecil, hasil ini menunjukkan bahwa model mampu mempelajari sedikit informasi tambahan di luar harga terakhir.

Secara keseluruhan, kedua modified model berhasil memberikan performa yang lebih baik daripada arsitektur baseline LSTM pada seluruh metrik. Modified AMD memberikan peningkatan yang lebih besar terhadap baseline LSTM, sedangkan modified AAPL menjadi satu-satunya model yang mampu sedikit mengungguli naive baseline.

Hasil tersebut juga menunjukkan bahwa model dengan arsitektur lebih kompleks tidak selalu secara otomatis lebih baik daripada metode sederhana. Naive baseline tetap menjadi pembanding penting karena harga saham memiliki autokorelasi yang sangat tinggi pada level harga. Oleh karena itu, keberhasilan model tidak hanya dilihat dari kemampuannya mengalahkan baseline LSTM, tetapi juga dari kemampuannya menyamai atau mengungguli naive baseline.

Nilai RMSE dan MAE AMD tidak boleh dibandingkan secara langsung dengan AAPL karena kedua saham memiliki skala harga yang berbeda. Untuk perbandingan relatif antarsaham, MAPE lebih sesuai digunakan. Berdasarkan MAPE, modified AAPL memiliki kesalahan relatif yang lebih kecil, yaitu 1,57%, sedangkan modified AMD memiliki MAPE 3,47%.

Berdasarkan hasil akhir, arsitektur modified terbukti lebih efektif daripada baseline LSTM. **AAPL Relative Log W40 U32 Huber** memberikan hasil terbaik untuk AAPL, sedangkan **AMD W60 U32 No Dropout** menjadi arsitektur LSTM terbaik untuk AMD, walaupun pada AMD naive baseline masih memberikan performa keseluruhan terbaik.